# Polyp-DDPM · Đánh giá theo nguồn công bố · FINAL v8

**Nguồn đúng:** repo Polyp-DDPM là https://github.com/mobaidoctor/polyp-ddpm, commit e74e67979c1a305533b1b32522008c364ea3e0c2. Tree tại commit này không có chương trình downstream segmentation. Paper dẫn [9] cho model và hyperparameters downstream. Vì vậy mã segmentation được đối chiếu từ https://github.com/simulamet-host/conditional-polyp-diffusion, commit46a1a4e156b152fc10ea324f0aba5b1323d7159b. Không nhầm hai nhóm tác giả.

**Chạy:** Kaggle phiên mới, T4×2, Internet On; attach master có bank900, Kvasir-SEG, HyperKvasir annotated, ETIS. Điền MASTER_ROOT nếu nhiều master. Run All. Train trực tiếp100epoch, không pilot20epoch, không chọn seed thắng. Output v8 độc lập, resume không giới hạn giờ nội bộ.

**Hai quyết định công khai:**
1. `preprocessing_mode=reference_code` tái hiện cả preprocessing trước model và normalization trong model như code[9]. Đây là chuẩn hóa hai lần có trong nguồn tham chiếu, KHÔNG có bằng chứng xác nhận Polyp-DDPM đã dùng nguyên trạng. `corrected_single` là biến thể sửa một lần, không tự đổi theo điểm test.
2. `best_kvasir_validation` lưu best theo validation IoU mỗi epoch theo callback[9]. Với split900/100 hiện tại,100ảnh Kvasir được dùng làm validation. Điểm của tập này là **validation-selected**, không phải kiểm thử độc lập. ETIS và Hyper dùng cùng checkpoint được chọn, không chọn lại trên từng tập. Có thể dùng `last` để đánh giá checkpoint cuối100epoch, là biến thể công bố riêng.

**Giới hạn chính xác:** các setting có nguồn được đối chiếu; split được phép khác. Paper không công bố toàn bộ mã/config downstream Polyp-DDPM; không thể chứng nhận exact reproduction hoặc hứa trùng số trong paper. PyTorch/SMP hiện đại thay runtime Python3.8/PyTorch1.13/SMP0.3.2; version thực chạy và các file nguồn được ghi vào manifest.

**Nhánh chính:** real900, synthetic900 từ checkpoint tác giả, mixed1800. **Nhánh phụ:** checkpoint texture/nền của bạn nếu có bank900. LDM/SinGAN thiếu dữ liệu sẽ ghi MISSING, không tạo số và không chặn các nhánh đang có.


In [ ]:
from pathlib import Path

# ROOT chứa trực tiếp run_config.json và author/images của MASTER đã attach.
# None: tự tìm nếu chỉ có đúng một master phù hợp.
MASTER_ROOT = None
REAL_IMAGES = None
REAL_MASKS = None

# ROOT output của notebook ĐÁNH GIÁ này khi cần resume; không trỏ tới master.
RESUME_EVALUATION = None
OUTPUT = "/kaggle/working/polyp_ddpm_evaluation_v8"
INPUT_ROOT = "/kaggle/input"

ARCHITECTURES = ["unetpp", "fpn", "deeplabv3plus"]
SEEDS = [42]  # Seed cố định công bố; không chọn bằng pilot hoặc test.
TRAIN_EPOCHS = 100
BATCH_SIZE = 8
AMP = False
MAX_WORKERS = 2

EXTERNAL = {
    "HyperKvasir": {"images": None, "masks": None, "expected_n":1000, "mask_prefix":""},
    "ETIS": {"images": None, "masks": None, "expected_n":196, "mask_prefix":""},
}
# Nếu không tự tìm được, điền đúng hai thư mục leaf images/masks.
# ETIS có tên mask thêm prefix thì điền mask_prefix theo tên thực tế.

# Để None khi chưa có. Hai baseline này KHÔNG chặn real/synthetic/mixed.
BASELINES = {
    "LDM": {"pairs_csv": None, "quality_images": None},
    "SinGAN-Seg": {"pairs_csv": None, "quality_images": None},
}

# Nhánh phụ của bạn: để None khi chưa có bank900 đúng từ sampler refinement.
# CSV: sample_id,image,mask; sample_id là train ID; mask là file condition gốc.
REPAIRED_PAIRS_CSV = None
REPAIRED_LABEL = "Checkpoint chỉnh texture và nền"
# Không tự gán mốc30k/49k; ghi mốc thực tế trong metadata của bank.

PREPROCESSING_MODE = "reference_code"
CHECKPOINT_POLICY = "best_kvasir_validation"
# Dùng last cho checkpoint cuối; nếu đổi chế độ, dùng OUTPUT khác.


## 1. Môi trường đã chốt
Giữ PyTorch GPU của Kaggle. Pin augmentation1.3.0 đúng nguồn[9] và SMP0.5.0 cho runtime Kaggle hiện tại. Đây là thay đổi môi trường được công bố, không giả là binary nguyên bản. Ghi versions vào protocol. Train FP32, không AMP, không clipgradient.


In [ ]:
import sys,importlib.util,subprocess,json,os,shutil,threading,importlib.metadata
subprocess.check_call([sys.executable,"-m","pip","install","--quiet",
    "albumentations==1.3.0","opencv-python-headless<4.12","segmentation-models-pytorch==0.5.0",
    "torchmetrics","torch-fidelity","scipy","psutil","tqdm"])
import torch,numpy as np,pandas as pd,albumentations as albu
from IPython.display import display
assert torch.cuda.is_available(), "Bật GPU Kaggle"
display(pd.DataFrame([{"GPU":i,"name":torch.cuda.get_device_name(i)} for i in range(torch.cuda.device_count())]))


### Mã thực thi đi kèm
Cell dưới chứa đầy đủ mã nguồn, được thu gọn để dễ đọc. Mã này chạy thật trong các subprocess và có thể mở để kiểm tra. Các định nghĩa generator chỉ được dùng nếu cần sinh thêm ảnh cho metric; không khởi tạo generator trong tiến trình notebook.

In [ ]:
RUNTIME = Path(OUTPUT)/"runtime"
RUNTIME.mkdir(parents=True, exist_ok=True)
SOURCES = {'common.py': '"""Shared primitives for the report master. No service or repository calls."""\n\nimport os, sys, json, time, math, random, hashlib, gc, shutil, contextlib\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom refiner_core import (\n    canonical,\n    digest,\n    write_json,\n    save_pt,\n    load_pt,\n    png,\n    save_png,\n    factory,\n    seed_for,\n    seed_all,\n    state_fingerprint,\n)\n\n\ndef rng_state(device):\n    return {\n        "python": random.getstate(),\n        "numpy": np.random.get_state(),\n        "torch": torch.get_rng_state(),\n        "cuda": (\n            torch.cuda.get_rng_state(device).cpu() if device.type == "cuda" else None\n        ),\n    }\n\n\ndef restore_rng(value, device):\n    random.setstate(value["python"])\n    np.random.set_state(value["numpy"])\n    torch.set_rng_state(value["torch"])\n    if device.type == "cuda":\n        torch.cuda.set_rng_state(value["cuda"], device)\n\n\n@contextlib.contextmanager\ndef fixed_seed(seed, device):\n    value = rng_state(device)\n    seed_all(int(seed))\n    try:\n        yield\n    finally:\n        restore_rng(value, device)\n\n\ndef tensor_pair(record, size, seed=None, synthetic=None):\n    with Image.open(synthetic or record["image"]) as handle:\n        image = handle.convert("RGB").resize((size, size), Image.Resampling.BILINEAR)\n    with Image.open(record["mask"]) as handle:\n        mask = handle.convert("L").resize((size, size), Image.Resampling.NEAREST)\n    if seed is not None:\n        rng = random.Random(int(seed))\n        if rng.random() < 0.5:\n            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n            mask = mask.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n        angle = rng.uniform(-45, 45)\n        image = image.rotate(angle, Image.Resampling.BILINEAR)\n        mask = mask.rotate(angle, Image.Resampling.NEAREST)\n    x = torch.from_numpy(np.asarray(image).copy().transpose(2, 0, 1)).float() / 255\n    y = torch.from_numpy((np.asarray(mask) > 127).astype("float32").copy())[None]\n    return x, y\n\n\ndef stream_ids(ids, positions, seed):\n    cache = {}\n    chosen = []\n    for position in positions:\n        epoch, index = divmod(int(position), len(ids))\n        if epoch not in cache:\n            cache[epoch] = np.random.default_rng(seed + epoch).permutation(len(ids))\n        chosen.append(ids[int(cache[epoch][index])])\n    return chosen\n\n\ndef runtime(cfg, training=False):\n    import torch.distributed as dist\n\n    rank = int(os.environ.get("RANK", 0))\n    world = int(os.environ.get("WORLD_SIZE", 1))\n    local = int(os.environ.get("LOCAL_RANK", 0))\n    if cfg.get("cpu_fixture", False):\n        device = torch.device("cpu")\n    else:\n        if not torch.cuda.is_available() or local >= torch.cuda.device_count():\n            raise RuntimeError("Missing assigned CUDA device")\n        torch.cuda.set_device(local)\n        device = torch.device("cuda", local)\n    if world != cfg["world_size"]:\n        raise ValueError("World size differs from protocol")\n    if training and world > 1:\n        import datetime\n\n        dist.init_process_group(\n            "gloo" if device.type == "cpu" else "nccl",\n            timeout=datetime.timedelta(minutes=10),\n        )\n    torch.set_num_threads(cfg.get("cpu_threads_per_rank", 2))\n    seed_all(cfg["seed"] + rank)\n    return rank, world, device\n\n\ndef telemetry(cfg, phase, rank, device, **extra):\n    row = {\n        "phase": phase,\n        "rank": rank,\n        "device": str(device),\n        "pid": os.getpid(),\n        "timestamp": time.time(),\n        **extra,\n    }\n    if device.type == "cuda":\n        row.update(\n            name=torch.cuda.get_device_name(device),\n            allocated_GiB=torch.cuda.memory_allocated(device) / 2**30,\n            peak_GiB=torch.cuda.max_memory_allocated(device) / 2**30,\n        )\n    write_json(Path(cfg["out"]) / "execution" / f"{phase}_rank{rank}.json", row)\n    print(json.dumps(row), flush=True)\n\n\ndef resource_stop(cfg, device=None, synchronize=False):\n    import psutil\n    import torch.distributed as dist\n\n    total = 0.0\n    try:\n        parent = psutil.Process(os.getpid())\n        total = parent.memory_info().rss + sum(\n            p.memory_info().rss\n            for p in parent.children(recursive=True)\n            if p.is_running()\n        )\n    except psutil.Error:\n        pass\n    rss_stop = total / 2**30 > cfg.get(\n        "host_rss_limit_gib", 20.0\n    ) or psutil.virtual_memory().available / 2**30 < cfg.get("host_free_min_gib", 3.0)\n    container_stop = False\n    for used, limit in [\n        ("/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.max"),\n        (\n            "/sys/fs/cgroup/memory/memory.usage_in_bytes",\n            "/sys/fs/cgroup/memory/memory.limit_in_bytes",\n        ),\n    ]:\n        try:\n            a, b = int(Path(used).read_text()), int(Path(limit).read_text())\n            if b < 2**60:\n                container_stop = a / b > cfg.get("host_memory_fraction", 0.80)\n                break\n        except (OSError, ValueError):\n            pass\n    deadline_stop = time.time() > cfg["deadline"] - 180\n    stop = rss_stop or container_stop or deadline_stop\n    if stop:\n        write_json(\n            Path(cfg["out"]) / "execution" / f"stop_reason_{os.getpid()}.json",\n            {\n                "time": time.time(),\n                "deadline": deadline_stop,\n                "host_rss_or_free": bool(rss_stop),\n                "cgroup_memory": bool(container_stop),\n                "rss_gib": total / 2**30,\n            },\n        )\n    if synchronize and dist.is_initialized():\n        flag = torch.tensor(int(stop), device=device)\n        dist.all_reduce(flag)\n        stop = bool(flag.item())\n    return bool(stop)\n\n\ndef assert_config(cfg, source_dir):\n    if canonical(cfg["locked"]) != cfg["protocol"]:\n        raise ValueError("Invalid master protocol")\n    contract = cfg.get("execution_contract")\n    if contract and canonical(contract["payload"]) != contract["hash"]:\n        raise ValueError("Invalid execution compatibility contract")\n    code = contract["payload"]["runtime_code"] if contract else cfg["locked"]["code"]\n    if contract and contract["payload"]["compute_protocol"] != cfg["protocol"]:\n        raise ValueError("Compatibility contract belongs to another experiment")\n    for name, sha in code.items():\n        if digest(Path(source_dir) / name) != sha:\n            raise ValueError("Changed runtime source: " + name)\n    for key, value in cfg["locked"]["settings"].items():\n        if cfg[key] != value:\n            raise ValueError("Changed locked setting: " + key)\n    if cfg["author_sha256"] != cfg["locked"]["author_sha256"]:\n        raise ValueError("Runtime author identity differs from protocol")\n    if (\n        cfg["split"] != cfg["locked"]["split"]\n        or cfg["train_ids"] != cfg["split"]["train"]\n        or cfg["test_ids"] != cfg["split"]["test"]\n    ):\n        raise ValueError("Runtime split differs from protocol")\n    expected_records = {r["id"]: r for r in cfg["locked"]["data"]}\n    if set(expected_records) != {r["id"] for r in cfg["records"]}:\n        raise ValueError("Runtime data IDs differ from protocol")\n    for record in cfg["records"]:\n        for key in ["image_sha256", "mask_sha256"]:\n            if record[key] != expected_records[record["id"]][key]:\n                raise ValueError("Runtime data identity differs from protocol")\n    for r in cfg["records"]:\n        for key in ["image", "mask"]:\n            if digest(r[key]) != r[key + "_sha256"]:\n                raise ValueError("Changed data: " + r["id"])\n    if digest(cfg["author_checkpoint"]) != cfg["author_sha256"]:\n        raise ValueError("Author checkpoint changed")\n\n\n@torch.no_grad()\ndef full_reverse(diff, condition, seed, trace=False):\n    """One full 250-step trajectory with its own RNG scope; [0,1] RGB out."""\n    diff.eval()\n    device = condition.device\n    frames = []\n    with fixed_seed(seed, device):\n        x = torch.randn(condition.shape, device=device)\n        if trace:\n            frames.append((x[0].cpu(), "initial Gaussian noise"))\n        for timestep in reversed(range(diff.num_timesteps)):\n            t = torch.full((len(x),), timestep, device=device, dtype=torch.long)\n            x = diff.p_sample(x, t, condition_tensors=condition)\n            if trace and timestep in [199, 149, 99, 49, 0]:\n                frames.append((x[0].cpu(), f"after t={timestep}"))\n    return (x + 1).div(2).clamp(0, 1), frames\n\n\ndef segmentation_model(cfg, device, pretrained=False):\n    if cfg.get("cpu_fixture"):\n        return nn.Sequential(\n            nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.Conv2d(8, 1, 1)\n        ).to(device)\n    import segmentation_models_pytorch as smp\n\n    return smp.UnetPlusPlus(\n        encoder_name="resnet34",\n        encoder_weights=cfg["encoder_weights"] if pretrained else None,\n        in_channels=3,\n        classes=1,\n    ).to(device)\n\n\ndef normalize_segmenter(x):\n    mean = x.new_tensor([0.485, 0.456, 0.406])[None, :, None, None]\n    std = x.new_tensor([0.229, 0.224, 0.225])[None, :, None, None]\n    return (x - mean) / std\n\n\ndef dice_loss(logits, y):\n    p = logits.sigmoid()\n    return 1 - (2 * (p * y).sum() + 1e-7) / (p.sum() + y.sum() + 1e-7)\n\n\ndef binary_metrics(prob, mask):\n    from scipy.ndimage import binary_erosion, binary_dilation\n\n    pred = prob >= 0.5\n    gt = mask.astype(bool)\n    tp = int((pred & gt).sum())\n    fp = int((pred & ~gt).sum())\n    fn = int((~pred & gt).sum())\n    tn = int((~pred & ~gt).sum())\n    a = pred ^ binary_erosion(pred)\n    b = gt ^ binary_erosion(gt)\n    if not a.any() or not b.any():\n        bf = float(not a.any() and not b.any())\n    else:\n        precision = (a & binary_dilation(b, iterations=2)).sum() / a.sum()\n        recall = (b & binary_dilation(a, iterations=2)).sum() / b.sum()\n        bf = float(2 * precision * recall / (precision + recall + 1e-7))\n    return {\n        "tp": tp,\n        "fp": fp,\n        "fn": fn,\n        "tn": tn,\n        "iou": float((tp + 1e-7) / (tp + fp + fn + 1e-7)),\n        "dice": float((2 * tp + 1e-7) / (2 * tp + fp + fn + 1e-7)),\n        "boundary_f1": bf,\n        "precision": float((tp + 1e-7) / (tp + fp + 1e-7)),\n        "recall": float((tp + 1e-7) / (tp + fn + 1e-7)),\n    }\n', 'refiner_core.py': '"""Mask-conditioned partial DDPM refinement. New research adaptation; no quality claim."""\n\nimport os, json, hashlib, math, random, time\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom models import GaussianDiffusion, create_model, extract\nfrom color_base import (\n    rgb_to_lab,\n    lab_to_rgb,\n    lab_moments,\n    tissue_support,\n    gradient_match,\n)\n\n\ndef canonical(value):\n    return hashlib.sha256(\n        json.dumps(value, sort_keys=True, separators=(",", ":")).encode()\n    ).hexdigest()\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for b in iter(lambda: f.read(1 << 20), b""):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_text(json.dumps(value, indent=2))\n    os.replace(tmp, path)\n\n\ndef save_pt(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    torch.save(value, tmp)\n    os.replace(tmp, path)\n\n\ndef load_pt(path):\n    return torch.load(path, map_location="cpu", weights_only=False)\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef seed_for(sid, replicate=0):\n    return int(\n        hashlib.sha256((sid + ":" + str(replicate)).encode()).hexdigest()[:8], 16\n    )\n\n\ndef png(path, size, mask=False):\n    with Image.open(path) as im:\n        a = np.array(\n            im.convert("L" if mask else "RGB").resize(\n                (size, size),\n                Image.Resampling.NEAREST if mask else Image.Resampling.BILINEAR,\n            ),\n            copy=True,\n        )\n    return (\n        torch.from_numpy((a > 127).astype(np.float32))[None]\n        if mask\n        else torch.from_numpy(a).permute(2, 0, 1).float() / 255\n    )\n\n\ndef save_png(x, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    arr = x.detach().cpu().clamp(0, 1).mul(255).round().byte().permute(1, 2, 0).numpy()\n    tmp = path.with_name(path.stem + ".tmp.png")\n    Image.fromarray(arr).save(tmp)\n    os.replace(tmp, path)\n\n\nclass TinyDenoiser(nn.Module):\n    """Explicit CPU test fixture only; never used for production outputs."""\n\n    def __init__(self):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(6, 8, 3, padding=1), nn.SiLU(), nn.Conv2d(8, 3, 3, padding=1)\n        )\n\n    def forward(self, x, t):\n        return self.conv(x)\n\n\ndef factory(cfg, device):\n    net = (\n        TinyDenoiser()\n        if cfg.get("cpu_fixture", False)\n        else create_model(256, 64, 2, in_channels=6, out_channels=3)\n    )\n    return GaussianDiffusion(\n        net, image_size=cfg["size"], timesteps=250, loss_type="l1"\n    ).to(device)\n\n\nclass Perceptual:\n    def __init__(self, cfg, device):\n        self.fixture = cfg.get("cpu_fixture", False)\n        if not self.fixture:\n            import lpips\n\n            self.model = (\n                lpips.LPIPS(net="alex", verbose=False)\n                .to(device)\n                .eval()\n                .requires_grad_(False)\n            )\n\n    def loss(self, a, b):\n        return (a - b).abs().mean() if self.fixture else self.model(a, b).mean()\n\n    @torch.no_grad()\n    def features(self, x, mask):\n        # Low-level pretrained AlexNet regional feature moments, descriptive only.\n        fmaps = (\n            [x, F.avg_pool2d(x, 2)]\n            if self.fixture\n            else list(self.model.net(self.model.scaling_layer(x)))[:2]\n        )\n        keep = tissue_support((x + 1) / 2)\n        vectors = []\n        for reg in [mask, 1 - mask]:\n            parts = []\n            for f in fmaps:\n                w = F.interpolate(reg * keep, size=f.shape[-2:], mode="area")\n                n = w.sum((2, 3)).clamp_min(1)\n                mu = (f * w).sum((2, 3)) / n\n                sd = (\n                    ((f - mu[:, :, None, None]) ** 2 * w).sum((2, 3)) / n + 1e-6\n                ).sqrt()\n                parts.extend([mu, sd])\n            vectors.append(torch.cat(parts, 1))\n        return torch.stack(vectors, 1).cpu().numpy()\n\n\ndef low_pass(x, side=32):\n    # Smooth approximate low-pass; avoid block boundaries from nearest upsampling.\n    small = F.interpolate(\n        x, size=(side, side), mode="bilinear", align_corners=False, antialias=True\n    )\n    return F.interpolate(small, size=x.shape[-2:], mode="bilinear", align_corners=False)\n\n\ndef chroma_lock(image, reference, strength=0.75):\n    """Adjust a,b only before RGB gamut conversion; L change after gamut is measured."""\n    a = rgb_to_lab(image)\n    b = rgb_to_lab(reference)\n    ma, mb = lab_moments(image), lab_moments(reference)\n    scale = (mb[:, 4:6] / ma[:, 4:6].clamp_min(0.015)).clamp(0.5, 2)\n    target = (a[:, 1:] - ma[:, 1:3, None, None]) * scale[:, :, None, None] + mb[\n        :, 1:3, None, None\n    ]\n    strength_map = strength * ((a[:, :1] - 0.02) / 0.1).clamp(0, 1)\n    corrected = torch.cat(\n        [a[:, :1], a[:, 1:] * (1 - strength_map) + target * strength_map], 1\n    )\n    out = lab_to_rgb(corrected)\n    preserve = (reference.amax(1, keepdim=True) <= 0.025) | (\n        reference.amin(1, keepdim=True) >= 0.995\n    )\n    out = torch.where(preserve, reference, out)\n    keep_output = (image.amax(1, keepdim=True) <= 0.025) | (\n        image.amin(1, keepdim=True) >= 0.995\n    )\n    return torch.where(keep_output, image, out)\n\n\n@torch.no_grad()\ndef partial_reverse(\n    diff, reference, condition, steps, seed, low_side=32, anchor=0.5, lock=True\n):\n    """Steps=60 uses original indices59..0. Noise-consistent ILVR-style coarse anchor."""\n    if not 1 <= steps <= diff.num_timesteps:\n        raise ValueError("Invalid reverse steps")\n    device = reference.device\n    gen = torch.Generator(device=device).manual_seed(int(seed))\n    clean = reference * 2 - 1\n    noise = torch.randn(clean.shape, device=device, generator=gen)\n    ts = torch.full((len(clean),), steps - 1, device=device, dtype=torch.long)\n    x = diff.q_sample(clean, ts, noise=noise)\n    for t in reversed(range(steps)):\n        ts = torch.full((len(x),), t, device=device, dtype=torch.long)\n        eps = diff.denoise_fn(torch.cat([x, condition], 1), ts)\n        x0 = diff.predict_start_from_noise(x, ts, eps).clamp(-1, 1)\n        mean, _, logvar = diff.q_posterior(x0, x, ts)\n        z = torch.randn(x.shape, device=device, generator=gen)\n        proposal = mean + (t > 0) * (0.5 * logvar).exp() * z\n        # Anchor at t-1, not by inserting a clean image into a noisy state.\n        if t:\n            a = diff.alphas_cumprod[t - 1]\n            ref_state = a.sqrt() * clean + (1 - a).sqrt() * noise\n        else:\n            ref_state = clean\n        x = proposal + anchor * (\n            low_pass(ref_state, low_side) - low_pass(proposal, low_side)\n        )\n    out = ((x + 1) / 2).clamp(0, 1)\n    return chroma_lock(out, reference) if lock else out\n\n\ndef paired_loss(diff, eps, noisy, clean, noise, t, perceptual, cfg):\n    pred = diff.predict_start_from_noise(noisy, t, eps).clamp(-1, 1)\n    l_eps = (eps - noise).abs().mean()\n    l_grad = gradient_match(\n        rgb_to_lab((pred + 1) / 2)[:, :1], rgb_to_lab((clean + 1) / 2)[:, :1]\n    )\n    l_per = perceptual.loss(pred, clean)\n    total = l_eps + cfg["gradient_weight"] * l_grad + cfg["lpips_weight"] * l_per\n    return total, torch.stack([l_eps, l_grad, l_per])\n\n\ndef sample_ids(ids, positions, seed):\n    n = len(ids)\n    memo = {}\n    out = []\n    for pos in positions:\n        epoch, index = divmod(int(pos), n)\n        if epoch not in memo:\n            memo[epoch] = np.random.default_rng(seed + epoch).permutation(n)\n        out.append(ids[int(memo[epoch][index])])\n    return out\n\n\ndef training_batch(cfg, records, step, rank, world, micro):\n    b = cfg["per_gpu_batch"]\n    acc = cfg["accumulation"]\n    total = b * world * acc\n    positions = [step * total + micro * b * world + rank * b + j for j in range(b)]\n    chosen = sample_ids(cfg["fit_ids"], positions, cfg["seed"] + 9)\n    images = []\n    masks = []\n    for sid, pos in zip(chosen, positions):\n        r = records[sid]\n        x = png(r["image"], cfg["size"])\n        m = png(r["mask"], cfg["size"], True)\n        rng = np.random.default_rng(cfg["seed"] + pos)\n        if rng.random() < 0.5:\n            x = x.flip(-1)\n            m = m.flip(-1)\n        k = int(rng.integers(4))\n        x = torch.rot90(x, k, [-2, -1])\n        m = torch.rot90(m, k, [-2, -1])\n        images.append(x)\n        masks.append(m.repeat(3, 1, 1))\n    return torch.stack(images) * 2 - 1, torch.stack(masks) * 2 - 1, chosen\n\n\ndef descriptive_metrics(x, source, mask):\n    lab, base = rgb_to_lab(x), rgb_to_lab(source)\n    support = tissue_support(source)\n    color = (lab_moments(x)[:, :3] - lab_moments(source)[:, :3]).abs().mean()\n    drift = (\n        (\n            low_pass(x, 32 if x.shape[-1] >= 32 else 4)\n            - low_pass(source, 32 if x.shape[-1] >= 32 else 4)\n        )\n        .abs()\n        .mean()\n    )\n    luma = (lab[:, :1] - base[:, :1]).abs().mean()\n    grey = lab[:, :1]\n    lap = F.conv2d(\n        grey, grey.new_tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]])[None, None], padding=1\n    ).abs()\n    interior = F.avg_pool2d(support, 9, 1, 4) > 0.999\n    values = lap[interior]\n    return {\n        "color_moment_shift": float(color),\n        "coarse_rgb_drift": float(drift),\n        "L_change": float(luma),\n        "laplacian_median": float(values.median()) if values.numel() else 0.0,\n        "laplacian_p95": float(torch.quantile(values, 0.95)) if values.numel() else 0.0,\n    }\n\n\ndef masked_mean(error, weight):\n    return (error * weight).sum((1, 2, 3)) / (\n        weight.sum((1, 2, 3)) * error.shape[1]\n    ).clamp_min(1)\n\n\ndef region_weights(clean_rgb, condition):\n    mask = ((condition[:, :1] + 1) / 2).clamp(0, 1)\n    valid = tissue_support(clean_rgb).detach()\n    # Keep ROI losses off the FOV transition; epsilon still supervises the full image.\n    valid = (F.avg_pool2d(valid, 5, 1, 2) > 0.99).float()\n    return mask * valid, (1 - mask) * valid\n\n\ndef balanced_error(error, fg, bg):\n    a, b = masked_mean(error, fg), masked_mean(error, bg)\n    va = (fg.sum((1, 2, 3)) > 0).float()\n    vb = (bg.sum((1, 2, 3)) > 0).float()\n    return ((a * va + b * vb) / (va + vb).clamp_min(1)).mean()\n\n\ndef region_gradient(a, b, region):\n    terms = []\n    for dim in [2, 3]:\n        err = (a.diff(dim=dim) - b.diff(dim=dim)).abs()\n        if dim == 2:\n            w = region[:, :, :-1] * region[:, :, 1:]\n        else:\n            w = region[:, :, :, :-1] * region[:, :, :, 1:]\n        terms.append(masked_mean(err, w))\n    return sum(terms)\n\n\ndef roi_perceptual(per, a, b, region, side=128):\n    # Aligned paired patches; select a center whose patch contains the most ROI support.\n    side = min(side, a.shape[-1], a.shape[-2])\n    losses = []\n    for i in range(len(a)):\n        coords = torch.nonzero(region[i, 0] > 0.5, as_tuple=False)\n        if not len(coords):\n            continue\n        chosen = torch.linspace(\n            0, len(coords) - 1, min(8, len(coords)), device=a.device\n        ).long()\n        options = []\n        for point in coords[chosen]:\n            y = max(0, min(a.shape[-2] - side, int(point[0]) - side // 2))\n            x = max(0, min(a.shape[-1] - side, int(point[1]) - side // 2))\n            score = float(region[i, :, y : y + side, x : x + side].mean())\n            options.append((score, y, x))\n        _, y, x = max(options)\n        losses.append(\n            per.loss(\n                a[i : i + 1, :, y : y + side, x : x + side],\n                b[i : i + 1, :, y : y + side, x : x + side],\n            )\n        )\n    return torch.stack(losses).mean() if losses else a.sum() * 0\n\n\ndef degrade_real(clean, condition, seed):\n    """Paired blur+grain simulation, especially background. No geometry warp or color shift.\n\n    This small original degradation model is inspired by restoration training, not BSRGAN reproduction.\n    """\n    gen = torch.Generator(device=clean.device).manual_seed(int(seed))\n    result = []\n    rgb = (clean + 1) / 2\n    mask = ((condition[:, :1] + 1) / 2).clamp(0, 1)\n    for i in range(len(clean)):\n        sigma = 0.5 + 1.0 * float(torch.rand((), device=clean.device, generator=gen))\n        axis = torch.arange(-3, 4, device=clean.device, dtype=clean.dtype)\n        kernel = torch.exp(-axis.square() / (2 * sigma * sigma))\n        kernel /= kernel.sum()\n        k = (kernel[:, None] * kernel[None, :])[None, None].repeat(3, 1, 1, 1)\n        blur = F.conv2d(\n            F.pad(rgb[i : i + 1], (3, 3, 3, 3), mode="reflect"), k, groups=3\n        )\n        gaussian = torch.randn(rgb[i : i + 1].shape, device=clean.device, generator=gen)\n        grain = (\n            0.007 + 0.018 * float(torch.rand((), device=clean.device, generator=gen))\n        ) * gaussian\n        speckle = (\n            0.025\n            * torch.randn((1, 1, *rgb.shape[-2:]), device=clean.device, generator=gen)\n            * rgb[i : i + 1]\n        )\n        mix = (1 - 0.65 * mask[i : i + 1]) * tissue_support(rgb[i : i + 1])\n        # Preserve genuine black pixels; no claim that simulated artifacts exactly match DDPM outputs.\n        degraded = rgb[i : i + 1] + mix * (blur + grain + speckle - rgb[i : i + 1])\n        degraded = torch.where(\n            rgb[i : i + 1].amax(1, keepdim=True) < 0.025, rgb[i : i + 1], degraded\n        )\n        result.append(degraded.clamp(0, 1))\n    return torch.cat(result) * 2 - 1\n\n\ndef mixed_training_state(diff, clean, condition, t, cfg, seed):\n    gen = torch.Generator(device=clean.device).manual_seed(int(seed))\n    repair = (\n        torch.rand((len(clean), 1, 1, 1), device=clean.device, generator=gen)\n        < cfg["restoration_probability"]\n    )\n    repair = repair & (t >= cfg["restoration_t_min"])[:, None, None, None]\n    corrupted = degrade_real(clean, condition, seed + 1)\n    base = torch.where(repair, corrupted, clean)\n    noise = torch.randn(clean.shape, device=clean.device, generator=gen)\n    noisy = diff.q_sample(base, t, noise)\n    # A corrupted-real branch must NOT incorrectly use plain Gaussian noise as target.\n    alpha = extract(diff.sqrt_alphas_cumprod, t, clean.shape)\n    sigma = extract(diff.sqrt_one_minus_alphas_cumprod, t, clean.shape).clamp_min(1e-5)\n    effective_target = (noisy - alpha * clean) / sigma\n    return noisy, effective_target, repair.flatten()\n\n\ndef background_loss(\n    diff, eps, noisy, clean, target, t, condition, per, cfg, teacher_eps=None, pure=None\n):\n    prediction = diff.predict_start_from_noise(noisy, t, eps).clamp(-1, 1)\n    fg, bg = region_weights((clean + 1) / 2, condition)\n    err = (eps - target).abs()\n    epsilon = 0.5 * err.mean() + 0.5 * balanced_error(err, fg, bg)\n    eligible = t <= cfg["aux_t_max"]\n    grad = prediction.sum() * 0\n    patch = grad\n    rgb = grad\n    if eligible.any():\n        pred, real = prediction[eligible], clean[eligible]\n        f, b = fg[eligible], bg[eligible]\n        pl, rl = rgb_to_lab((pred + 1) / 2)[:, :1], rgb_to_lab((real + 1) / 2)[:, :1]\n        grad = (\n            0.5 * region_gradient(pl, rl, f).mean()\n            + 0.5 * region_gradient(pl, rl, b).mean()\n        )\n        patch = 0.5 * roi_perceptual(per, pred, real, f) + 0.5 * roi_perceptual(\n            per, pred, real, b\n        )\n        rgb = balanced_error((pred - real).abs(), f, b)\n    prior = prediction.sum() * 0\n    if teacher_eps is not None and pure is not None and pure.any():\n        prior = (eps[pure] - teacher_eps[pure]).abs().mean()\n    total = (\n        epsilon\n        + cfg["gradient_weight"] * grad\n        + cfg["lpips_weight"] * patch\n        + cfg["rgb_weight"] * rgb\n        + cfg["teacher_weight"] * prior\n    )\n    return total, torch.stack([epsilon, grad, patch, prior, rgb])\n\n\ndef palette_chroma_only(image, reference, strength=0.25):\n    # Use only a,b means from an unrelated train palette; never its spatial black/white mask.\n    lab = rgb_to_lab(image)\n    a = lab_moments(image)\n    b = lab_moments(reference)\n    shift = (b[:, 1:3] - a[:, 1:3]).clamp(-0.08, 0.08)[:, :, None, None] * strength\n    fade = ((lab[:, :1] - 0.02) / 0.1).clamp(0, 1)\n    out = lab_to_rgb(torch.cat([lab[:, :1], lab[:, 1:] + fade * shift], 1))\n    keep = (image.amax(1, keepdim=True) < 0.025) | (image.amin(1, keepdim=True) > 0.995)\n    return torch.where(keep, image, out)\n\n\n@torch.no_grad()\ndef background_reverse(diff, reference, condition, steps, seed, cfg, palette=None):\n    """Higher-noise background resynthesis with early, spatially selective coarse anchors."""\n    if not 1 <= steps <= diff.num_timesteps:\n        raise ValueError("Invalid reverse steps")\n    device = reference.device\n    clean = reference * 2 - 1\n    gen = torch.Generator(device=device).manual_seed(int(seed))\n    noise = torch.randn(clean.shape, device=device, generator=gen)\n    ts = torch.full((len(clean),), steps - 1, device=device, dtype=torch.long)\n    x = diff.q_sample(clean, ts, noise)\n    soft = F.avg_pool2d(((condition[:, :1] + 1) / 2).clamp(0, 1), 15, 1, 7)\n    for t in reversed(range(steps)):\n        ts = torch.full((len(x),), t, device=device, dtype=torch.long)\n        eps = diff.denoise_fn(torch.cat([x, condition], 1), ts)\n        x0 = diff.predict_start_from_noise(x, ts, eps).clamp(-1, 1)\n        mean, _, logvar = diff.q_posterior(x0, x, ts)\n        z = torch.randn(x.shape, device=device, generator=gen)\n        x = mean + (t > 0) * (0.5 * logvar).exp() * z\n        # Last detail steps are completely free of source reinjection.\n        fade = max(\n            0.0,\n            (t - cfg["release_last_steps"])\n            / max(steps - 1 - cfg["release_last_steps"], 1),\n        )\n        if fade:\n            alpha = diff.alphas_cumprod[t - 1] if t else x.new_tensor(1.0)\n            ref = alpha.sqrt() * clean + (1 - alpha).sqrt() * noise\n            fg = low_pass(ref, 32 if reference.shape[-1] >= 32 else 4) - low_pass(\n                x, 32 if reference.shape[-1] >= 32 else 4\n            )\n            bg = low_pass(ref, cfg["low_side"]) - low_pass(x, cfg["low_side"])\n            x = x + fade * (\n                cfg["foreground_anchor"] * soft * fg\n                + cfg["background_anchor"] * (1 - soft) * bg\n            )\n    out = ((x + 1) / 2).clamp(0, 1)\n    return palette_chroma_only(\n        out, reference if palette is None else palette, cfg["chroma_strength"]\n    )\n\n\n# --- v2 continuation: spatial protection and aligned boundary supervision ---\nv2_reverse = background_reverse\nv2_background_loss = background_loss\n\n\ndef protection_regions(condition, cfg):\n    mask = ((condition[:, :1] + 1) / 2).clamp(0, 1)\n    radius = cfg["guard_radius"]\n    dilated = F.max_pool2d(mask, 2 * radius + 1, 1, radius)\n    erode_radius = max(1, radius // 2)\n    eroded = -F.max_pool2d(-mask, 2 * erode_radius + 1, 1, erode_radius)\n    band = (dilated - eroded).clamp(0, 1)\n    soft = F.avg_pool2d(dilated, 9, 1, 4)\n    return mask, band, soft\n\n\ndef continuation_loss(\n    diff, eps, noisy, clean, target, t, condition, per, cfg, teacher_eps=None, pure=None\n):\n    # Teacher is the frozen selected v2 refiner, not a new author initialization.\n    total, parts = v2_background_loss(\n        diff, eps, noisy, clean, target, t, condition, per, cfg, None, None\n    )\n    mask, band, _ = protection_regions(condition, cfg)\n    valid = tissue_support((clean + 1) / 2).detach()\n    band = band * valid\n    prior = eps.sum() * 0\n    if teacher_eps is not None and pure is not None and pure.any():\n        weights = 1 + cfg["prior_fg_boost"] * mask + cfg["prior_band_boost"] * band\n        prior = masked_mean((eps[pure] - teacher_eps[pure]).abs(), weights[pure]).mean()\n    pred = diff.predict_start_from_noise(noisy, t, eps).clamp(-1, 1)\n    eligible = t <= cfg["aux_t_max"]\n    boundary_rgb = eps.sum() * 0\n    boundary_grad = boundary_rgb\n    if eligible.any():\n        a, b, w = pred[eligible], clean[eligible], band[eligible]\n        boundary_rgb = masked_mean((a - b).abs(), w).mean()\n        al = rgb_to_lab((a + 1) / 2)[:, :1]\n        bl = rgb_to_lab((b + 1) / 2)[:, :1]\n        boundary_grad = region_gradient(al, bl, w).mean()\n    total = (\n        total\n        + cfg["teacher_weight"] * prior\n        + cfg["boundary_rgb_weight"] * boundary_rgb\n        + cfg["boundary_gradient_weight"] * boundary_grad\n    )\n    parts = torch.cat(\n        [parts[:3], prior[None], parts[4:], boundary_rgb[None], boundary_grad[None]]\n    )\n    return total, parts\n\n\n@torch.no_grad()\ndef protected_reverse(diff, reference, condition, steps, seed, cfg, palette=None):\n    # Protect only coarse structure near the polyp; do not copy real-reference pixels.\n    if not 1 <= steps <= diff.num_timesteps:\n        raise ValueError("Invalid reverse steps")\n    clean = reference * 2 - 1\n    gen = torch.Generator(device=reference.device).manual_seed(int(seed))\n    noise = torch.randn(clean.shape, device=clean.device, generator=gen)\n    ts = torch.full((len(clean),), steps - 1, device=clean.device, dtype=torch.long)\n    x = diff.q_sample(clean, ts, noise)\n    mask, band, protected = protection_regions(condition, cfg)\n    side = 32 if reference.shape[-1] >= 32 else 4\n    for t in reversed(range(steps)):\n        ts = torch.full((len(x),), t, device=x.device, dtype=torch.long)\n        eps = diff.denoise_fn(torch.cat([x, condition], 1), ts)\n        x0 = diff.predict_start_from_noise(x, ts, eps).clamp(-1, 1)\n        mean, _, logvar = diff.q_posterior(x0, x, ts)\n        z = torch.randn(x.shape, device=x.device, generator=gen)\n        x = mean + (t > 0) * (0.5 * logvar).exp() * z\n        fade = max(\n            0.0,\n            (t - cfg["release_last_steps"])\n            / max(steps - 1 - cfg["release_last_steps"], 1),\n        )\n        alpha = diff.alphas_cumprod[t - 1] if t else x.new_tensor(1.0)\n        ref = alpha.sqrt() * clean + (1 - alpha).sqrt() * noise\n        coarse_delta = low_pass(ref, side) - low_pass(x, side)\n        bg_delta = low_pass(ref, cfg["low_side"]) - low_pass(x, cfg["low_side"])\n        fg_strength = (\n            fade * cfg["foreground_anchor"] + (1 - fade) * cfg["foreground_floor"]\n        )\n        guard_strength = (1 - fade) * cfg["boundary_floor"]\n        weight = (fg_strength * mask + guard_strength * band).clamp(0, 1)\n        # Background remains fully released in the last detail steps, as in v2.\n        x = (\n            x\n            + weight * coarse_delta\n            + fade * cfg["background_anchor"] * (1 - protected) * bg_delta\n        )\n    out = ((x + 1) / 2).clamp(0, 1)\n    return palette_chroma_only(\n        out, reference if palette is None else palette, cfg["chroma_strength"]\n    )\n\n\ndef background_reverse(diff, reference, condition, steps, seed, cfg, palette=None):\n    return protected_reverse(diff, reference, condition, steps, seed, cfg, palette)\n\n\ndef state_fingerprint(state):\n    h = hashlib.sha256()\n    for name, tensor in sorted(state.items()):\n        h.update(name.encode())\n        h.update(str(tensor.dtype).encode())\n        h.update(str(tuple(tensor.shape)).encode())\n        h.update(tensor.detach().cpu().contiguous().numpy().tobytes())\n    return h.hexdigest()\n', 'models.py': '# -*- coding:utf-8 -*-\n#\n# Original code is here: https://github.com/openai/guided-diffusion\n#\n\nfrom abc import abstractmethod\n\nimport math\n\nimport numpy as np\nimport torch as th\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef checkpoint(func, inputs, params, flag):\n    """\n    Evaluate a function without caching intermediate activations, allowing for\n    reduced memory at the expense of extra compute in the backward pass.\n    :param func: the function to evaluate.\n    :param inputs: the argument sequence to pass to `func`.\n    :param params: a sequence of parameters `func` depends on but does not\n                   explicitly take as arguments.\n    :param flag: if False, disable gradient checkpointing.\n    """\n    if flag:\n        args = tuple(inputs) + tuple(params)\n        return CheckpointFunction.apply(func, len(inputs), *args)\n    else:\n        return func(*inputs)\n\n\nclass CheckpointFunction(th.autograd.Function):\n    @staticmethod\n    def forward(ctx, run_function, length, *args):\n        ctx.run_function = run_function\n        ctx.input_tensors = list(args[:length])\n        ctx.input_params = list(args[length:])\n        with th.no_grad():\n            output_tensors = ctx.run_function(*ctx.input_tensors)\n        return output_tensors\n\n    @staticmethod\n    def backward(ctx, *output_grads):\n        ctx.input_tensors = [x.detach().requires_grad_(True) for x in ctx.input_tensors]\n        with th.enable_grad():\n            # Fixes a bug where the first op in run_function modifies the\n            # Tensor storage in place, which is not allowed for detach()\'d\n            # Tensors.\n            shallow_copies = [x.view_as(x) for x in ctx.input_tensors]\n            output_tensors = ctx.run_function(*shallow_copies)\n        input_grads = th.autograd.grad(\n            output_tensors,\n            ctx.input_tensors + ctx.input_params,\n            output_grads,\n            allow_unused=True,\n        )\n        del ctx.run_function\n        del ctx.input_tensors\n        del ctx.input_params\n        del output_tensors\n        return (None, None) + input_grads\n\n\ndef conv_nd(dims, *args, **kwargs):\n    """\n    Create a 1D, 2D, or 3D convolution module.\n    """\n    if dims == 1:\n        return nn.Conv1d(*args, **kwargs)\n    elif dims == 2:\n        return nn.Conv2d(*args, **kwargs)\n    elif dims == 3:\n        return nn.Conv3d(*args, **kwargs)\n    raise ValueError(f"unsupported dimensions: {dims}")\n\n\ndef linear(*args, **kwargs):\n    """\n    Create a linear module.\n    """\n    return nn.Linear(*args, **kwargs)\n\n\ndef zero_module(module):\n    """\n    Zero out the parameters of a module and return it.\n    """\n    for p in module.parameters():\n        p.detach().zero_()\n    return module\n\n\ndef scale_module(module, scale):\n    """\n    Scale the parameters of a module and return it.\n    """\n    for p in module.parameters():\n        p.detach().mul_(scale)\n    return module\n\n\ndef mean_flat(tensor):\n    """\n    Take the mean over all non-batch dimensions.\n    """\n    return tensor.mean(dim=list(range(1, len(tensor.shape))))\n\n\ndef normalization(channels):\n    """\n    Make a standard normalization layer.\n    :param channels: number of input channels.\n    :return: an nn.Module for normalization.\n    """\n    return GroupNorm32(32, channels)\n\n\ndef timestep_embedding(timesteps, dim, max_period=1000):\n    """\n    Create sinusoidal timestep embeddings.\n    :param timesteps: a 1-D Tensor of N indices, one per batch element.\n                      These may be fractional.\n    :param dim: the dimension of the output.\n    :param max_period: controls the minimum frequency of the embeddings.\n    :return: an [N x dim] Tensor of positional embeddings.\n    """\n    half = dim // 2\n    freqs = th.exp(\n        -math.log(max_period) * th.arange(start=0, end=half, dtype=th.float32) / half\n    ).to(device=timesteps.device)\n    args = timesteps[:, None].float() * freqs[None]\n    embedding = th.cat([th.cos(args), th.sin(args)], dim=-1)\n    if dim % 2:\n        embedding = th.cat([embedding, th.zeros_like(embedding[:, :1])], dim=-1)\n    return embedding\n\n\nclass SiLU(nn.Module):\n    def forward(self, x):\n        return x * th.sigmoid(x)\n\n\nclass GroupNorm32(nn.GroupNorm):\n    def forward(self, x):\n        return super().forward(x.float()).type(x.dtype)\n\n\nclass AttentionPool2d(nn.Module):\n    """\n    Adapted from CLIP: https://github.com/openai/CLIP/blob/main/clip/model.py\n    """\n\n    def __init__(\n        self,\n        spacial_dim: int,\n        embed_dim: int,\n        num_heads_channels: int,\n        output_dim: int = None,\n    ):\n        super().__init__()\n        self.positional_embedding = nn.Parameter(\n            th.randn(embed_dim, spacial_dim**2 + 1) / embed_dim**0.5\n        )\n        self.qkv_proj = conv_nd(1, embed_dim, 3 * embed_dim, 1)\n        self.c_proj = conv_nd(1, embed_dim, output_dim or embed_dim, 1)\n        self.num_heads = embed_dim // num_heads_channels\n        self.attention = QKVAttention(self.num_heads)\n\n    def forward(self, x):\n        b, c, *_spatial = x.shape\n        x = x.reshape(b, c, -1)  # NC(HW)\n        x = th.cat([x.mean(dim=-1, keepdim=True), x], dim=-1)  # NC(HW+1)\n        x = x + self.positional_embedding[None, :, :].to(x.dtype)  # NC(HW+1)\n        x = self.qkv_proj(x)\n        x = self.attention(x)\n        x = self.c_proj(x)\n        return x[:, :, 0]\n\n\nclass TimestepBlock(nn.Module):\n    """\n    Any module where forward() takes timestep embeddings as a second argument.\n    """\n\n    @abstractmethod\n    def forward(self, x, emb):\n        """\n        Apply the module to `x` given `emb` timestep embeddings.\n        """\n\n\nclass TimestepEmbedSequential(nn.Sequential, TimestepBlock):\n    """\n    A sequential module that passes timestep embeddings to the children that\n    support it as an extra input.\n    """\n\n    def forward(self, x, emb):\n        for layer in self:\n            if isinstance(layer, TimestepBlock):\n                x = layer(x, emb)\n            else:\n                x = layer(x)\n        return x\n\n\nclass Upsample(nn.Module):\n    """\n    An upsampling layer with an optional convolution.\n    :param channels: channels in the inputs and outputs.\n    :param use_conv: a bool determining if a convolution is applied.\n    :param dims: determines if the signal is 1D, 2D, or 3D. If 3D, then\n                 upsampling occurs in the inner-two dimensions.\n    """\n\n    def __init__(self, channels, use_conv, dims=2, out_channels=None):\n        super().__init__()\n        self.channels = channels\n        self.out_channels = out_channels or channels\n        self.use_conv = use_conv\n        self.dims = dims\n        if use_conv:\n            self.conv = conv_nd(dims, self.channels, self.out_channels, 3, padding=1)\n\n    def forward(self, x):\n        assert x.shape[1] == self.channels\n        if self.dims == 3:\n            x = F.interpolate(\n                x, (x.shape[2], x.shape[3] * 2, x.shape[4] * 2), mode="nearest"\n            )\n        else:\n            x = F.interpolate(x, scale_factor=2, mode="nearest")\n        if self.use_conv:\n            x = self.conv(x)\n        return x\n\n\nclass Downsample(nn.Module):\n    """\n    A downsampling layer with an optional convolution.\n    :param channels: channels in the inputs and outputs.\n    :param use_conv: a bool determining if a convolution is applied.\n    :param dims: determines if the signal is 1D, 2D, or 3D. If 3D, then\n                 downsampling occurs in the inner-two dimensions.\n    """\n\n    def __init__(self, channels, use_conv, dims=2, out_channels=None):\n        super().__init__()\n        self.channels = channels\n        self.out_channels = out_channels or channels\n        self.use_conv = use_conv\n        self.dims = dims\n        stride = 2 if dims != 3 else (1, 2, 2)\n        if use_conv:\n            self.op = conv_nd(\n                dims, self.channels, self.out_channels, 3, stride=stride, padding=1\n            )\n        else:\n            assert self.channels == self.out_channels\n            self.op = avg_pool_nd(dims, kernel_size=stride, stride=stride)\n\n    def forward(self, x):\n        assert x.shape[1] == self.channels\n        return self.op(x)\n\n\nclass ResBlock(TimestepBlock):\n    """\n    A residual block that can optionally change the number of channels.\n    :param channels: the number of input channels.\n    :param emb_channels: the number of timestep embedding channels.\n    :param dropout: the rate of dropout.\n    :param out_channels: if specified, the number of out channels.\n    :param use_conv: if True and out_channels is specified, use a spatial\n        convolution instead of a smaller 1x1 convolution to change the\n        channels in the skip connection.\n    :param dims: determines if the signal is 1D, 2D, or 3D.\n    :param use_checkpoint: if True, use gradient checkpointing on this module.\n    :param up: if True, use this block for upsampling.\n    :param down: if True, use this block for downsampling.\n    """\n\n    def __init__(\n        self,\n        channels,\n        emb_channels,\n        dropout,\n        out_channels=None,\n        use_conv=False,\n        use_scale_shift_norm=False,\n        dims=2,\n        use_checkpoint=False,\n        up=False,\n        down=False,\n    ):\n        super().__init__()\n        self.channels = channels\n        self.emb_channels = emb_channels\n        self.dropout = dropout\n        self.out_channels = out_channels or channels\n        self.use_conv = use_conv\n        self.use_checkpoint = use_checkpoint\n        self.use_scale_shift_norm = use_scale_shift_norm\n\n        self.in_layers = nn.Sequential(\n            normalization(channels),\n            nn.SiLU(),\n            conv_nd(dims, channels, self.out_channels, 3, padding=1),\n        )\n\n        self.updown = up or down\n\n        if up:\n            self.h_upd = Upsample(channels, False, dims)\n            self.x_upd = Upsample(channels, False, dims)\n        elif down:\n            self.h_upd = Downsample(channels, False, dims)\n            self.x_upd = Downsample(channels, False, dims)\n        else:\n            self.h_upd = self.x_upd = nn.Identity()\n\n        self.emb_layers = nn.Sequential(\n            nn.SiLU(),\n            linear(\n                emb_channels,\n                2 * self.out_channels if use_scale_shift_norm else self.out_channels,\n            ),\n        )\n        self.out_layers = nn.Sequential(\n            normalization(self.out_channels),\n            nn.SiLU(),\n            nn.Dropout(p=dropout),\n            zero_module(\n                conv_nd(dims, self.out_channels, self.out_channels, 3, padding=1)\n            ),\n        )\n\n        if self.out_channels == channels:\n            self.skip_connection = nn.Identity()\n        elif use_conv:\n            self.skip_connection = conv_nd(\n                dims, channels, self.out_channels, 3, padding=1\n            )\n        else:\n            self.skip_connection = conv_nd(dims, channels, self.out_channels, 1)\n\n    def forward(self, x, emb):\n        """\n        Apply the block to a Tensor, conditioned on a timestep embedding.\n        :param x: an [N x C x ...] Tensor of features.\n        :param emb: an [N x emb_channels] Tensor of timestep embeddings.\n        :return: an [N x C x ...] Tensor of outputs.\n        """\n        return checkpoint(\n            self._forward, (x, emb), self.parameters(), self.use_checkpoint\n        )\n\n    def _forward(self, x, emb):\n        if self.updown:\n            in_rest, in_conv = self.in_layers[:-1], self.in_layers[-1]\n            h = in_rest(x)\n            h = self.h_upd(h)\n            x = self.x_upd(x)\n            h = in_conv(h)\n        else:\n            h = self.in_layers(x)\n        emb_out = self.emb_layers(emb).type(h.dtype)\n        while len(emb_out.shape) < len(h.shape):\n            emb_out = emb_out[..., None]\n        if self.use_scale_shift_norm:\n            out_norm, out_rest = self.out_layers[0], self.out_layers[1:]\n            scale, shift = th.chunk(emb_out, 2, dim=1)\n            h = out_norm(h) * (1 + scale) + shift\n            h = out_rest(h)\n        else:\n            h = h + emb_out\n            h = self.out_layers(h)\n        return self.skip_connection(x) + h\n\n\nclass AttentionBlock(nn.Module):\n    """\n    An attention block that allows spatial positions to attend to each other.\n    Originally ported from here, but adapted to the N-d case.\n    https://github.com/hojonathanho/diffusion/blob/1e0dceb3b3495bbe19116a5e1b3596cd0706c543/diffusion_tf/models/unet.py#L66.\n    """\n\n    def __init__(\n        self,\n        channels,\n        num_heads=1,\n        num_head_channels=-1,\n        use_checkpoint=False,\n        use_new_attention_order=False,\n    ):\n        super().__init__()\n        self.channels = channels\n        if num_head_channels == -1:\n            self.num_heads = num_heads\n        else:\n            assert (\n                channels % num_head_channels == 0\n            ), f"q,k,v channels {channels} is not divisible by num_head_channels {num_head_channels}"\n            self.num_heads = channels // num_head_channels\n        self.use_checkpoint = use_checkpoint\n        self.norm = normalization(channels)\n        self.qkv = conv_nd(1, channels, channels * 3, 1)\n        if use_new_attention_order:\n            # split qkv before split heads\n            self.attention = QKVAttention(self.num_heads)\n        else:\n            # split heads before split qkv\n            self.attention = QKVAttentionLegacy(self.num_heads)\n\n        self.proj_out = zero_module(conv_nd(1, channels, channels, 1))\n\n    def forward(self, x):\n        return checkpoint(self._forward, (x,), self.parameters(), True)\n\n    def _forward(self, x):\n        b, c, *spatial = x.shape\n        x = x.reshape(b, c, -1)\n        qkv = self.qkv(self.norm(x))\n        h = self.attention(qkv)\n        h = self.proj_out(h)\n        return (x + h).reshape(b, c, *spatial)\n\n\ndef count_flops_attn(model, _x, y):\n    """\n    A counter for the `thop` package to count the operations in an\n    attention operation.\n    Meant to be used like:\n        macs, params = thop.profile(\n            model,\n            inputs=(inputs, timestamps),\n            custom_ops={QKVAttention: QKVAttention.count_flops},\n        )\n    """\n    b, c, *spatial = y[0].shape\n    num_spatial = int(np.prod(spatial))\n    # We perform two matmuls with the same number of ops.\n    # The first computes the weight matrix, the second computes\n    # the combination of the value vectors.\n    matmul_ops = 2 * b * (num_spatial**2) * c\n    model.total_ops += th.DoubleTensor([matmul_ops])\n\n\nclass QKVAttentionLegacy(nn.Module):\n    """\n    A module which performs QKV attention. Matches legacy QKVAttention + input/ouput heads shaping\n    """\n\n    def __init__(self, n_heads):\n        super().__init__()\n        self.n_heads = n_heads\n\n    def forward(self, qkv):\n        """\n        Apply QKV attention.\n        :param qkv: an [N x (H * 3 * C) x T] tensor of Qs, Ks, and Vs.\n        :return: an [N x (H * C) x T] tensor after attention.\n        """\n        bs, width, length = qkv.shape\n        assert width % (3 * self.n_heads) == 0\n        ch = width // (3 * self.n_heads)\n        q, k, v = qkv.reshape(bs * self.n_heads, ch * 3, length).split(ch, dim=1)\n        scale = 1 / math.sqrt(math.sqrt(ch))\n        weight = th.einsum(\n            "bct,bcs->bts", q * scale, k * scale\n        )  # More stable with f16 than dividing afterwards\n        weight = th.softmax(weight.float(), dim=-1).type(weight.dtype)\n        a = th.einsum("bts,bcs->bct", weight, v)\n        return a.reshape(bs, -1, length)\n\n    @staticmethod\n    def count_flops(model, _x, y):\n        return count_flops_attn(model, _x, y)\n\n\nclass QKVAttention(nn.Module):\n    """\n    A module which performs QKV attention and splits in a different order.\n    """\n\n    def __init__(self, n_heads):\n        super().__init__()\n        self.n_heads = n_heads\n\n    def forward(self, qkv):\n        """\n        Apply QKV attention.\n        :param qkv: an [N x (3 * H * C) x T] tensor of Qs, Ks, and Vs.\n        :return: an [N x (H * C) x T] tensor after attention.\n        """\n        bs, width, length = qkv.shape\n        assert width % (3 * self.n_heads) == 0\n        ch = width // (3 * self.n_heads)\n        q, k, v = qkv.chunk(3, dim=1)\n        scale = 1 / math.sqrt(math.sqrt(ch))\n        weight = th.einsum(\n            "bct,bcs->bts",\n            (q * scale).view(bs * self.n_heads, ch, length),\n            (k * scale).view(bs * self.n_heads, ch, length),\n        )  # More stable with f16 than dividing afterwards\n        weight = th.softmax(weight.float(), dim=-1).type(weight.dtype)\n        a = th.einsum("bts,bcs->bct", weight, v.reshape(bs * self.n_heads, ch, length))\n        return a.reshape(bs, -1, length)\n\n    @staticmethod\n    def count_flops(model, _x, y):\n        return count_flops_attn(model, _x, y)\n\n\ndef convert_module_to_f16(l):\n    """\n    Convert primitive modules to float16.\n    """\n    if isinstance(l, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):\n        l.weight.data = l.weight.data.half()\n        if l.bias is not None:\n            l.bias.data = l.bias.data.half()\n\n\ndef convert_module_to_f32(l):\n    """\n    Convert primitive modules to float32, undoing convert_module_to_f16().\n    """\n    if isinstance(l, (nn.Conv1d, nn.Conv2d, nn.Conv3d)):\n        l.weight.data = l.weight.data.float()\n        if l.bias is not None:\n            l.bias.data = l.bias.data.float()\n\n\n# -*- coding:utf-8 -*-\n#\n# Original code is here: https://github.com/openai/guided-diffusion\n#\n\nimport numpy as np\nimport torch as th\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nNUM_CLASSES = 1\n\n\nclass UNetModel(nn.Module):\n    """\n    The full UNet model with attention and timestep embedding.\n    :param in_channels: channels in the input Tensor.\n    :param model_channels: base channel count for the model.\n    :param out_channels: channels in the output Tensor.\n    :param num_res_blocks: number of residual blocks per downsample.\n    :param attention_resolutions: a collection of downsample rates at which\n        attention will take place. May be a set, list, or tuple.\n        For example, if this contains 4, then at 4x downsampling, attention\n        will be used.\n    :param dropout: the dropout probability.\n    :param channel_mult: channel multiplier for each level of the UNet.\n    :param conv_resample: if True, use learned convolutions for upsampling and\n        downsampling.\n    :param dims: determines if the signal is 1D, 2D, or 3D.\n    :param num_classes: if specified (as an int), then this model will be\n        class-conditional with `num_classes` classes.\n    :param use_checkpoint: use gradient checkpointing to reduce memory usage.\n    :param num_heads: the number of attention heads in each attention layer.\n    :param num_heads_channels: if specified, ignore num_heads and instead use\n                               a fixed channel width per attention head.\n    :param num_heads_upsample: works with num_heads to set a different number\n                               of heads for upsampling. Deprecated.\n    :param use_scale_shift_norm: use a FiLM-like conditioning mechanism.\n    :param resblock_updown: use residual blocks for up/downsampling.\n    :param use_new_attention_order: use a different attention pattern for potentially\n                                    increased efficiency.\n    """\n\n    def __init__(\n        self,\n        image_size,\n        in_channels,\n        model_channels,\n        out_channels,\n        num_res_blocks,\n        attention_resolutions,\n        dropout=0,\n        channel_mult=(1, 2, 4, 8),\n        conv_resample=True,\n        dims=2,\n        num_classes=None,\n        use_checkpoint=False,\n        use_fp16=False,\n        num_heads=1,\n        num_head_channels=-1,\n        num_heads_upsample=-1,\n        use_scale_shift_norm=False,\n        resblock_updown=False,\n        use_new_attention_order=False,\n    ):\n        super().__init__()\n\n        if num_heads_upsample == -1:\n            num_heads_upsample = num_heads\n\n        self.image_size = image_size\n        self.in_channels = in_channels\n        self.model_channels = model_channels\n        self.out_channels = out_channels\n        self.num_res_blocks = num_res_blocks\n        self.attention_resolutions = attention_resolutions\n        self.dropout = dropout\n        self.channel_mult = channel_mult\n        self.conv_resample = conv_resample\n        self.num_classes = num_classes\n        self.use_checkpoint = use_checkpoint\n        self.dtype = th.float16 if use_fp16 else th.float32\n        self.num_heads = num_heads\n        self.num_head_channels = num_head_channels\n        self.num_heads_upsample = num_heads_upsample\n\n        time_embed_dim = model_channels * 4\n        self.time_embed = nn.Sequential(\n            linear(model_channels, time_embed_dim),\n            nn.SiLU(),\n            linear(time_embed_dim, time_embed_dim),\n        )\n\n        if self.num_classes is not None:\n            self.label_emb = nn.Embedding(num_classes, time_embed_dim)\n\n        ch = input_ch = int(channel_mult[0] * model_channels)\n        self.input_blocks = nn.ModuleList(\n            [TimestepEmbedSequential(conv_nd(dims, in_channels, ch, 3, padding=1))]\n        )\n        self._feature_size = ch\n        input_block_chans = [ch]\n        ds = 1\n        for level, mult in enumerate(channel_mult):\n            for _ in range(num_res_blocks):\n                layers = [\n                    ResBlock(\n                        ch,\n                        time_embed_dim,\n                        dropout,\n                        out_channels=int(mult * model_channels),\n                        dims=dims,\n                        use_checkpoint=use_checkpoint,\n                        use_scale_shift_norm=use_scale_shift_norm,\n                    )\n                ]\n                ch = int(mult * model_channels)\n                if ds in attention_resolutions:\n                    layers.append(\n                        AttentionBlock(\n                            ch,\n                            use_checkpoint=use_checkpoint,\n                            num_heads=num_heads,\n                            num_head_channels=num_head_channels,\n                            use_new_attention_order=use_new_attention_order,\n                        )\n                    )\n                self.input_blocks.append(TimestepEmbedSequential(*layers))\n                self._feature_size += ch\n                input_block_chans.append(ch)\n            if level != len(channel_mult) - 1:\n                out_ch = ch\n                self.input_blocks.append(\n                    TimestepEmbedSequential(\n                        ResBlock(\n                            ch,\n                            time_embed_dim,\n                            dropout,\n                            out_channels=out_ch,\n                            dims=dims,\n                            use_checkpoint=use_checkpoint,\n                            use_scale_shift_norm=use_scale_shift_norm,\n                            down=True,\n                        )\n                        if resblock_updown\n                        else Downsample(\n                            ch, conv_resample, dims=dims, out_channels=out_ch\n                        )\n                    )\n                )\n                ch = out_ch\n                input_block_chans.append(ch)\n                ds *= 2\n                self._feature_size += ch\n\n        self.middle_block = TimestepEmbedSequential(\n            ResBlock(\n                ch,\n                time_embed_dim,\n                dropout,\n                dims=dims,\n                use_checkpoint=use_checkpoint,\n                use_scale_shift_norm=use_scale_shift_norm,\n            ),\n            AttentionBlock(\n                ch,\n                use_checkpoint=use_checkpoint,\n                num_heads=num_heads,\n                num_head_channels=num_head_channels,\n                use_new_attention_order=use_new_attention_order,\n            ),\n            ResBlock(\n                ch,\n                time_embed_dim,\n                dropout,\n                dims=dims,\n                use_checkpoint=use_checkpoint,\n                use_scale_shift_norm=use_scale_shift_norm,\n            ),\n        )\n        self._feature_size += ch\n\n        self.output_blocks = nn.ModuleList([])\n        for level, mult in list(enumerate(channel_mult))[::-1]:\n            for i in range(num_res_blocks + 1):\n                ich = input_block_chans.pop()\n                layers = [\n                    ResBlock(\n                        ch + ich,\n                        time_embed_dim,\n                        dropout,\n                        out_channels=int(model_channels * mult),\n                        dims=dims,\n                        use_checkpoint=use_checkpoint,\n                        use_scale_shift_norm=use_scale_shift_norm,\n                    )\n                ]\n                ch = int(model_channels * mult)\n                if ds in attention_resolutions:\n                    layers.append(\n                        AttentionBlock(\n                            ch,\n                            use_checkpoint=use_checkpoint,\n                            num_heads=num_heads_upsample,\n                            num_head_channels=num_head_channels,\n                            use_new_attention_order=use_new_attention_order,\n                        )\n                    )\n                if level and i == num_res_blocks:\n                    out_ch = ch\n                    layers.append(\n                        ResBlock(\n                            ch,\n                            time_embed_dim,\n                            dropout,\n                            out_channels=out_ch,\n                            dims=dims,\n                            use_checkpoint=use_checkpoint,\n                            use_scale_shift_norm=use_scale_shift_norm,\n                            up=True,\n                        )\n                        if resblock_updown\n                        else Upsample(ch, conv_resample, dims=dims, out_channels=out_ch)\n                    )\n                    ds //= 2\n                self.output_blocks.append(TimestepEmbedSequential(*layers))\n                self._feature_size += ch\n\n        self.out = nn.Sequential(\n            normalization(ch),\n            nn.SiLU(),\n            zero_module(conv_nd(dims, input_ch, out_channels, 3, padding=1)),\n        )\n\n    def convert_to_fp16(self):\n        """\n        Convert the torso of the model to float16.\n        """\n        self.input_blocks.apply(convert_module_to_f16)\n        self.middle_block.apply(convert_module_to_f16)\n        self.output_blocks.apply(convert_module_to_f16)\n\n    def convert_to_fp32(self):\n        """\n        Convert the torso of the model to float32.\n        """\n        self.input_blocks.apply(convert_module_to_f32)\n        self.middle_block.apply(convert_module_to_f32)\n        self.output_blocks.apply(convert_module_to_f32)\n\n    def forward(self, x, timesteps, y=None):\n        """\n        Apply the model to an input batch.\n        :param x: an [N x C x ...] Tensor of inputs.\n        :param timesteps: a 1-D batch of timesteps.\n        :param y: an [N] Tensor of labels, if class-conditional.\n        :return: an [N x C x ...] Tensor of outputs.\n        """\n        assert (y is not None) == (\n            self.num_classes is not None\n        ), "must specify y if and only if the model is class-conditional"\n\n        hs = []\n        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))\n\n        if self.num_classes is not None:\n            assert y.shape == (x.shape[0],)\n            emb = emb + self.label_emb(y)\n\n        h = x.type(self.dtype)\n        for module in self.input_blocks:\n            h = module(h, emb)\n            hs.append(h)\n        h = self.middle_block(h, emb)\n        for module in self.output_blocks:\n            h = th.cat([h, hs.pop()], dim=1)\n            h = module(h, emb)\n        h = h.type(x.dtype)\n        return self.out(h)\n\n\ndef create_model(\n    image_size,\n    num_channels,\n    num_res_blocks,\n    channel_mult="",\n    learn_sigma=False,\n    class_cond=False,\n    use_checkpoint=False,\n    attention_resolutions="16",\n    num_heads=1,\n    num_head_channels=-1,\n    num_heads_upsample=-1,\n    use_scale_shift_norm=False,\n    dropout=0,\n    resblock_updown=False,\n    use_fp16=False,\n    use_new_attention_order=False,\n    in_channels=6,\n    out_channels=3,\n):\n    if channel_mult == "":\n        if image_size == 512:\n            channel_mult = (0.5, 1, 1, 2, 2, 4, 4)\n        elif image_size == 256:\n            channel_mult = (1, 1, 2, 2, 4, 4)\n        elif image_size == 128:\n            channel_mult = (1, 1, 2, 3, 4)\n        elif image_size == 64:\n            channel_mult = (1, 2, 3, 4)\n        else:\n            raise ValueError(f"unsupported image size: {image_size}")\n    else:\n        channel_mult = tuple(int(ch_mult) for ch_mult in channel_mult.split(","))\n\n    attention_ds = []\n    for res in attention_resolutions.split(","):\n        attention_ds.append(image_size // int(res))\n\n    return UNetModel(\n        image_size=image_size,\n        in_channels=in_channels,\n        model_channels=num_channels,\n        out_channels=(1 * out_channels if not learn_sigma else 2 * out_channels),\n        num_res_blocks=num_res_blocks,\n        attention_resolutions=tuple(attention_ds),\n        dropout=dropout,\n        channel_mult=channel_mult,\n        num_classes=(NUM_CLASSES if class_cond else None),\n        use_checkpoint=use_checkpoint,\n        use_fp16=use_fp16,\n        num_heads=num_heads,\n        num_head_channels=num_head_channels,\n        num_heads_upsample=num_heads_upsample,\n        use_scale_shift_norm=use_scale_shift_norm,\n        resblock_updown=resblock_updown,\n        use_new_attention_order=use_new_attention_order,\n    )\n\n\nimport torch\nimport numpy as np\nimport math\nfrom torch import nn\nimport torch.nn.functional as F\nfrom inspect import isfunction\nfrom functools import partial\nfrom tqdm.auto import tqdm\n\n\ndef exists(x):\n    return x is not None\n\n\ndef default(val, d):\n    if exists(val):\n        return val\n    return d() if isfunction(d) else d\n\n\ndef extract(a, t, x_shape):\n    b, *_ = t.shape\n    out = a.gather(-1, t)\n    return out.reshape(b, *((1,) * (len(x_shape) - 1)))\n\n\ndef noise_like(shape, device, repeat=False):\n    repeat_noise = lambda: torch.randn((1, *shape[1:]), device=device).repeat(\n        shape[0], *((1,) * (len(shape) - 1))\n    )\n    noise = lambda: torch.randn(shape, device=device)\n    return repeat_noise() if repeat else noise()\n\n\ndef cosine_beta_schedule(timesteps, s=0.008):\n    """\n    cosine schedule\n    as proposed in https://openreview.net/forum?id=-NEXDKk8gZ\n    """\n    steps = timesteps + 1\n    x = np.linspace(0, steps, steps)\n    alphas_cumprod = np.cos(((x / steps) + s) / (1 + s) * np.pi * 0.5) ** 2\n    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]\n    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])\n    return np.clip(betas, a_min=0, a_max=0.999)\n\n\nclass GaussianDiffusion(nn.Module):\n    def __init__(\n        self,\n        denoise_fn,\n        *,\n        image_size,\n        channels=3,\n        timesteps=1000,\n        loss_type="l1",\n        betas=None,\n    ):\n        super().__init__()\n        self.channels = channels\n        self.image_size = image_size\n        self.denoise_fn = denoise_fn\n\n        if exists(betas):\n            betas = (\n                betas.detach().cpu().numpy()\n                if isinstance(betas, torch.Tensor)\n                else betas\n            )\n        else:\n            betas = cosine_beta_schedule(timesteps)\n\n        alphas = 1.0 - betas\n        alphas_cumprod = np.cumprod(alphas, axis=0)\n        alphas_cumprod_prev = np.append(1.0, alphas_cumprod[:-1])\n\n        (timesteps,) = betas.shape\n        self.num_timesteps = int(timesteps)\n        self.loss_type = loss_type\n\n        to_torch = partial(torch.tensor, dtype=torch.float32)\n\n        self.register_buffer("betas", to_torch(betas))\n        self.register_buffer("alphas_cumprod", to_torch(alphas_cumprod))\n        self.register_buffer("alphas_cumprod_prev", to_torch(alphas_cumprod_prev))\n\n        # calculations for diffusion q(x_t | x_{t-1}) and others\n        self.register_buffer("sqrt_alphas_cumprod", to_torch(np.sqrt(alphas_cumprod)))\n        self.register_buffer(\n            "sqrt_one_minus_alphas_cumprod", to_torch(np.sqrt(1.0 - alphas_cumprod))\n        )\n        self.register_buffer(\n            "log_one_minus_alphas_cumprod", to_torch(np.log(1.0 - alphas_cumprod))\n        )\n        self.register_buffer(\n            "sqrt_recip_alphas_cumprod", to_torch(np.sqrt(1.0 / alphas_cumprod))\n        )\n        self.register_buffer(\n            "sqrt_recipm1_alphas_cumprod", to_torch(np.sqrt(1.0 / alphas_cumprod - 1))\n        )\n\n        # calculations for posterior q(x_{t-1} | x_t, x_0)\n        posterior_variance = (\n            betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)\n        )\n        # above: equal to 1. / (1. / (1. - alpha_cumprod_tm1) + alpha_t / beta_t)\n        self.register_buffer("posterior_variance", to_torch(posterior_variance))\n        # below: log calculation clipped because the posterior variance is 0 at the beginning of the diffusion chain\n        self.register_buffer(\n            "posterior_log_variance_clipped",\n            to_torch(np.log(np.maximum(posterior_variance, 1e-20))),\n        )\n        self.register_buffer(\n            "posterior_mean_coef1",\n            to_torch(betas * np.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)),\n        )\n        self.register_buffer(\n            "posterior_mean_coef2",\n            to_torch(\n                (1.0 - alphas_cumprod_prev) * np.sqrt(alphas) / (1.0 - alphas_cumprod)\n            ),\n        )\n\n    def q_mean_variance(self, x_start, t):\n        mean = extract(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start\n        variance = extract(1.0 - self.alphas_cumprod, t, x_start.shape)\n        log_variance = extract(self.log_one_minus_alphas_cumprod, t, x_start.shape)\n        return mean, variance, log_variance\n\n    def predict_start_from_noise(self, x_t, t, noise):\n        return (\n            extract(self.sqrt_recip_alphas_cumprod, t, x_t.shape) * x_t\n            - extract(self.sqrt_recipm1_alphas_cumprod, t, x_t.shape) * noise\n        )\n\n    def q_posterior(self, x_start, x_t, t, c=None):\n        posterior_mean = (\n            extract(self.posterior_mean_coef1, t, x_t.shape) * x_start\n            + extract(self.posterior_mean_coef2, t, x_t.shape) * x_t\n        )\n        posterior_variance = extract(self.posterior_variance, t, x_t.shape)\n        posterior_log_variance_clipped = extract(\n            self.posterior_log_variance_clipped, t, x_t.shape\n        )\n        return posterior_mean, posterior_variance, posterior_log_variance_clipped\n\n    def p_mean_variance(self, x, t, clip_denoised: bool, c=None):\n        x_recon = self.predict_start_from_noise(\n            x, t=t, noise=self.denoise_fn(torch.cat([x, c], 1), t)\n        )\n\n        if clip_denoised:\n            x_recon.clamp_(-1.0, 1.0)\n\n        model_mean, posterior_variance, posterior_log_variance = self.q_posterior(\n            x_start=x_recon, x_t=x, t=t, c=c\n        )\n        return model_mean, posterior_variance, posterior_log_variance\n\n    @torch.no_grad()\n    def p_sample(\n        self, x, t, condition_tensors=None, clip_denoised=True, repeat_noise=False\n    ):\n        b, *_, device = *x.shape, x.device\n        model_mean, _, model_log_variance = self.p_mean_variance(\n            x=x, t=t, c=condition_tensors, clip_denoised=clip_denoised\n        )\n        noise = noise_like(x.shape, device, repeat_noise)\n        # no noise when t == 0\n        nonzero_mask = (1 - (t == 0).float()).reshape(b, *((1,) * (len(x.shape) - 1)))\n        return model_mean + nonzero_mask * (0.5 * model_log_variance).exp() * noise\n\n    @torch.no_grad()\n    def p_sample_loop(self, shape, condition_tensors=None):\n        device = self.betas.device\n        b = shape[0]\n        img = torch.randn(shape, device=device)\n\n        # Looping through timesteps in reverse order\n        for i in tqdm(\n            reversed(range(self.num_timesteps)),\n            desc="sampling loop time step",\n            total=self.num_timesteps,\n        ):\n            t = torch.full((b,), i, device=device, dtype=torch.long)\n            img = self.p_sample(img, t, condition_tensors=condition_tensors)\n\n        return img\n\n    @torch.no_grad()\n    def sample(self, batch_size=16, condition_tensors=None):\n        image_size = self.image_size\n        channels = self.channels\n\n        return self.p_sample_loop(\n            (batch_size, channels, image_size, image_size),\n            condition_tensors=condition_tensors,\n        )\n\n    @torch.no_grad()\n    def interpolate(self, x1, x2, t=None, lam=0.5):\n        b, *_, device = *x1.shape, x1.device\n        t = default(t, self.num_timesteps - 1)\n\n        assert x1.shape == x2.shape\n\n        t_batched = torch.stack([torch.tensor(t, device=device)] * b)\n        xt1, xt2 = map(lambda x: self.q_sample(x, t=t_batched), (x1, x2))\n\n        img = (1 - lam) * xt1 + lam * xt2\n        for i in tqdm(\n            reversed(range(0, t)), desc="interpolation sample time step", total=t\n        ):\n            img = self.p_sample(\n                img, torch.full((b,), i, device=device, dtype=torch.long)\n            )\n\n        return img\n\n    def q_sample(self, x_start, t, noise=None):\n        noise = default(noise, lambda: torch.randn_like(x_start))\n\n        return (\n            extract(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start\n            + extract(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape) * noise\n        )\n\n    def p_losses(self, x_start, t, condition_tensors=None, noise=None):\n        b, c, h, w = x_start.shape\n        noise = default(noise, lambda: torch.randn_like(x_start))\n\n        x_noisy = self.q_sample(x_start=x_start, t=t, noise=noise)\n        x_recon = self.denoise_fn(torch.cat([x_noisy, condition_tensors], 1), t)\n\n        if self.loss_type == "l1":\n            loss = (noise - x_recon).abs().mean()\n        elif self.loss_type == "l2":\n            loss = F.mse_loss(x_recon, noise)\n        else:\n            raise NotImplementedError()\n\n        return loss\n\n    def forward(self, x, condition_tensors=None, *args, **kwargs):\n        (\n            b,\n            c,\n            h,\n            w,\n            device,\n            img_size,\n        ) = (\n            *x.shape,\n            x.device,\n            self.image_size,\n        )\n        assert (\n            h == img_size and w == img_size\n        ), f"height and width of image must be {img_size}"\n        t = torch.randint(0, self.num_timesteps, (b,), device=device).long()\n        return self.p_losses(x, t, condition_tensors=condition_tensors, *args, **kwargs)\n', 'color_base.py': '"""Trainable color calibration of own DDPM output. No spatial resampling of output pixels."""\n\nimport math\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\n\n\ndef rgb_to_lab(x):\n    x = x.clamp(0, 1)\n    linear = torch.where(\n        x > 0.04045, ((x + 0.055) / 1.055).clamp_min(1e-8).pow(2.4), x / 12.92\n    )\n    matrix = x.new_tensor(\n        [\n            [0.4124564, 0.3575761, 0.1804375],\n            [0.2126729, 0.7151522, 0.0721750],\n            [0.0193339, 0.1191920, 0.9503041],\n        ]\n    )\n    xyz = torch.einsum("ij,bjhw->bihw", matrix, linear)\n    xyz = xyz / x.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]\n    delta = 6 / 29\n    f = torch.where(\n        xyz > delta**3, xyz.clamp_min(1e-8).pow(1 / 3), xyz / (3 * delta**2) + 4 / 29\n    )\n    return torch.stack(\n        [\n            (116 * f[:, 1] - 16) / 100,\n            500 * (f[:, 0] - f[:, 1]) / 128,\n            200 * (f[:, 1] - f[:, 2]) / 128,\n        ],\n        dim=1,\n    )\n\n\ndef lab_to_rgb(lab, clip=True):\n    fy = (lab[:, 0] * 100 + 16) / 116\n    fx, fz = fy + lab[:, 1] * 128 / 500, fy - lab[:, 2] * 128 / 200\n    f = torch.stack([fx, fy, fz], dim=1)\n    delta = 6 / 29\n    xyz = torch.where(f > delta, f**3, 3 * delta**2 * (f - 4 / 29))\n    xyz = xyz * lab.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]\n    matrix = lab.new_tensor(\n        [\n            [3.2404542, -1.5371385, -0.4985314],\n            [-0.9692660, 1.8760108, 0.0415560],\n            [0.0556434, -0.2040259, 1.0572252],\n        ]\n    )\n    linear = torch.einsum("ij,bjhw->bihw", matrix, xyz)\n    x = torch.where(\n        linear > 0.0031308,\n        1.055 * linear.clamp_min(1e-8).pow(1 / 2.4) - 0.055,\n        12.92 * linear,\n    )\n    return x.clamp(0, 1) if clip else x\n\n\ndef tissue_support(x):\n    # Exclude black FOV and nearly pure white specular pixels from color statistics.\n    keep = (x.amax(1, keepdim=True) > 0.06) & (x.amin(1, keepdim=True) < 0.98)\n    fallback = torch.ones_like(keep)\n    return torch.where(keep.sum((2, 3), keepdim=True) >= 16, keep, fallback).float()\n\n\ndef lab_moments(x, support=None):\n    lab = rgb_to_lab(x)\n    w = tissue_support(x).detach() if support is None else support.detach()\n    n = w.sum((2, 3)).clamp_min(1)\n    mu = (lab * w).sum((2, 3)) / n\n    variance = ((lab - mu[:, :, None, None]) ** 2 * w).sum((2, 3)) / n\n    std = (variance + 1e-6).sqrt()\n    return torch.cat([mu, std], 1)\n\n\ndef palette_projection(x, palette, gain_limit=3.0):\n    """Global diagonal Lab color transfer, an explicit first/second moment constraint.\n\n    It has no learned geometry. RGB gamut clipping means final moments need not\n    exactly match the target; actual post-gamut errors are always measured.\n    """\n    support = tissue_support(x)\n    source = lab_moments(x, support)\n    lab = rgb_to_lab(x)\n    floor = x.new_tensor([0.035, 0.015, 0.015])[None]\n    src_std = torch.maximum(source[:, 3:], floor)\n    target_std = torch.maximum(palette[:, 3:], floor)\n    scale = (target_std / src_std).clamp(1 / gain_limit, gain_limit)\n    corrected = (lab - source[:, :3, None, None]) * scale[:, :, None, None] + palette[\n        :, :3, None, None\n    ]\n    # Preserve deep shadows continuously; no threshold-based hole cutting/FOV segmentation.\n    lightness = torch.minimum(corrected[:, :1], lab[:, :1].clamp_min(0) * 3.0)\n    corrected = torch.cat([lightness, corrected[:, 1:]], dim=1)\n    raw = lab_to_rgb(corrected, clip=False)\n    result = raw.clamp(0, 1)\n    # Keep existing black borders and pure white highlights at the same locations.\n    preserve = (x.amax(1, keepdim=True) <= 0.025) | (x.amin(1, keepdim=True) >= 0.995)\n    result = torch.where(preserve, x, result)\n    penalty_mask = (~preserve).float()\n    excess = ((F.relu(-raw) + F.relu(raw - 1)) * penalty_mask).sum() / (\n        penalty_mask.sum() * 3\n    ).clamp_min(1)\n    return result, excess\n\n\ndef identity_lut(dim, device=None):\n    axis = torch.linspace(0, 1, dim, device=device)\n    blue, green, red = torch.meshgrid(axis, axis, axis, indexing="ij")\n    return torch.stack([red, green, blue])\n\n\ndef apply_lut(x, lut):\n    # 5-D grid_sample: D=B, H=G, W=R. Grid coordinates are (R,G,B).\n    grid = x.permute(0, 2, 3, 1).unsqueeze(1) * 2 - 1\n    return F.grid_sample(\n        lut, grid, mode="bilinear", padding_mode="border", align_corners=True\n    ).squeeze(2)\n\n\nclass PaletteLUT(nn.Module):\n    """Small image/palette-conditioned LUT model, inspired by image-adaptive 3DLUT.\n\n    Original implementation using native PyTorch. This is an adaptation, not\n    the published authors\' architecture or released weights.\n    """\n\n    def __init__(self, dim=17, bases=3):\n        super().__init__()\n        self.dim, self.bases = dim, bases\n        self.encoder = nn.Sequential(\n            nn.Conv2d(3, 16, 3, 2, 1),\n            nn.SiLU(),\n            nn.Conv2d(16, 32, 3, 2, 1),\n            nn.SiLU(),\n            nn.Conv2d(32, 48, 3, 2, 1),\n            nn.SiLU(),\n            nn.AdaptiveAvgPool2d(1),\n        )\n        self.predictor = nn.Sequential(\n            nn.Linear(48 + 6, 48), nn.SiLU(), nn.Linear(48, bases)\n        )\n        nn.init.zeros_(self.predictor[-1].weight)\n        nn.init.zeros_(self.predictor[-1].bias)\n        values = torch.randn(bases, 3, dim, dim, dim) * 0.002\n        values -= values.mean(0, keepdim=True)\n        self.residuals = nn.Parameter(values)\n        self.register_buffer("identity", identity_lut(dim))\n\n    def forward(self, x, palette, project=True):\n        features = self.encoder(\n            F.interpolate(x, size=(64, 64), mode="bilinear", align_corners=False)\n        ).flatten(1)\n        weights = self.predictor(torch.cat([features, palette], 1)).softmax(1)\n        tables = self.identity[None] + 0.75 * self.residuals.tanh()\n        lut = torch.einsum("bm,mcijk->bcijk", weights, tables)\n        raw = apply_lut(x, lut)\n        clip_penalty = (F.relu(-raw) + F.relu(raw - 1)).mean()\n        rendered = raw.clamp(0, 1)\n        if project:\n            rendered, gamut = palette_projection(rendered, palette)\n            clip_penalty = clip_penalty + gamut\n            baseline, _ = palette_projection(x, palette)\n            # Learned increment is bounded to avoid large new chromatic speckles.\n            rendered = baseline + 0.04 * torch.tanh((rendered - baseline) / 0.04)\n            rendered = rendered.clamp(0, 1)\n            preserve = (x.amax(1, keepdim=True) <= 0.025) | (\n                x.amin(1, keepdim=True) >= 0.995\n            )\n            rendered = torch.where(preserve, x, rendered)\n        return rendered, clip_penalty\n\n    def regularization(self):\n        tables = self.identity[None] + 0.75 * self.residuals.tanh()\n        tv = sum((self.residuals.diff(dim=axis) ** 2).mean() for axis in [2, 3, 4])\n        # Monotonicity in the corresponding color axis; cross-channel mixing is allowed.\n        mono = F.relu(-tables[:, 0].diff(dim=3)).mean()\n        mono += F.relu(-tables[:, 1].diff(dim=2)).mean()\n        mono += F.relu(-tables[:, 2].diff(dim=1)).mean()\n        return tv, mono\n\n\ndef color_corruption(clean, seed, severe=True):\n    """Paired supervision: invertible logit color/exposure shifts, no spatial warp."""\n    g = torch.Generator(device=clean.device).manual_seed(int(seed))\n    b = len(clean)\n    bias = torch.randn(b, 3, 1, 1, device=clean.device, generator=g) * (\n        1.0 if severe else 0.25\n    )\n    exposure = torch.randn(b, 1, 1, 1, device=clean.device, generator=g) * 0.7\n    gamma = (\n        (0.25 * torch.randn(b, 1, 1, 1, device=clean.device, generator=g))\n        .exp()\n        .clamp(0.65, 1.55)\n    )\n    logit = torch.logit(clean.clamp(0.005, 0.995))\n    noisy = torch.sigmoid(gamma * logit + bias + exposure)\n    black = clean.amax(1, keepdim=True) <= 0.025\n    return torch.where(black, clean, noisy)\n\n\ndef gradient_match(a, b):\n    # Aligned paired target gradients, not an energy statistic that rewards speckles.\n    return (a.diff(dim=2) - b.diff(dim=2)).abs().mean() + (\n        a.diff(dim=3) - b.diff(dim=3)\n    ).abs().mean()\n\n\ndef normalized_luma(x):\n    l = rgb_to_lab(x)[:, :1]\n    mu, std = l.mean((2, 3), keepdim=True), l.std((2, 3), keepdim=True).clamp_min(0.05)\n    return (l - mu) / std\n\n\ndef palette_swd(x, y, seed=123, pixels=256, directions=16):\n    """Spatially unpaired Lab sliced Wasserstein loss; never pixel L1 to author."""\n    g = torch.Generator(device=x.device).manual_seed(seed)\n    a, b = rgb_to_lab(x).flatten(2), rgb_to_lab(y).flatten(2)\n    valid_a, valid_b = (\n        tissue_support(x).flatten(2).bool(),\n        tissue_support(y).flatten(2).bool(),\n    )\n    theta = torch.randn(3, directions, device=x.device, generator=g)\n    theta = theta / theta.norm(dim=0, keepdim=True).clamp_min(1e-6)\n    losses = []\n    for i in range(len(x)):\n        ia = torch.nonzero(valid_a[i, 0], as_tuple=True)[0]\n        ib = torch.nonzero(valid_b[i, 0], as_tuple=True)[0]\n        n = min(pixels, len(ia), len(ib))\n        ia = ia[torch.linspace(0, len(ia) - 1, n, device=x.device).long()]\n        ib = ib[torch.linspace(0, len(ib) - 1, n, device=x.device).long()]\n        aa, bb = theta.T @ a[i, :, ia], theta.T @ b[i, :, ib]\n        losses.append((aa.sort(-1).values - bb.sort(-1).values).abs().mean())\n    return torch.stack(losses).mean()\n\n\ndef delta_e76(a, b):\n    difference = (rgb_to_lab(a) - rgb_to_lab(b)) * a.new_tensor([100, 128, 128])[\n        None, :, None, None\n    ]\n    return difference.square().sum(1).clamp_min(1e-10).sqrt().mean()\n\n\ndef restoration_metrics(candidate, target):\n    mse = (candidate - target).square().mean().clamp_min(1e-10)\n    return {\n        "deltaE76": float(delta_e76(candidate, target)),\n        "PSNR": float(-10 * torch.log10(mse)),\n        "gradient_L1": float(gradient_match(candidate, target)),\n    }\n', 'extension.py': '"""New evaluation study. Original master artifacts are immutable inputs."""\nimport os,sys,json,time,hashlib,copy,argparse,gc\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\nfrom PIL import Image\n\ndef identity(x):return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(\',\',\':\')).encode()).hexdigest()\ndef sha(p):\n h=hashlib.sha256()\n with open(p,\'rb\') as f:\n  for b in iter(lambda:f.read(1<<20),b\'\'):h.update(b)\n return h.hexdigest()\ndef write(p,x):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_suffix(p.suffix+\'.tmp\');t.write_text(json.dumps(x,indent=2));os.replace(t,p)\ndef images(folder):\n if not folder or not Path(folder).is_dir():return {}\n result={}\n for p in sorted(Path(folder).rglob(\'*\')):\n  if p.suffix.lower() in [\'.png\',\'.jpg\',\'.jpeg\',\'.bmp\',\'.tif\',\'.tiff\']:\n   if p.stem in result:raise ValueError(\'Duplicate image ID: \'+p.stem)\n   result[p.stem]=str(p)\n return result\n\ndef external(spec):\n im=images(spec.get(\'images\'));ma=images(spec.get(\'masks\'))\n if not im or not ma:return []\n prefix=spec.get(\'mask_prefix\',\'\')\n if prefix:ma={k[len(prefix):] if k.startswith(prefix) else k:v for k,v in ma.items()}\n if set(im)!=set(ma):raise ValueError(\'External image/mask IDs differ. Configure paired leaf directories.\')\n expected=spec.get(\'expected_n\')\n if expected and len(im)!=expected:raise ValueError(f\'Expected {expected} annotated external pairs, found {len(im)}\')\n for sid in im:\n  with Image.open(im[sid]) as a,Image.open(ma[sid]) as b:\n   if a.size!=b.size:raise ValueError(\'External image/mask native dimensions differ: \'+sid)\n return [{\'id\':i,\'image\':im[i],\'mask\':ma[i]} for i in sorted(im)]\n\ndef file_identity(paths):return [{\'name\':Path(p).name,\'sha256\':sha(p)} for p in paths]\n\ndef resolve(c):\n """Record exact input identities once per notebook launch, not each GPU batch."""\n p=c[\'parent\'];root=Path(p[\'out\']);records={r[\'id\']:r for r in p[\'records\']}\n banks={\'deployment\':{i:str(root/\'author/images\'/(i+\'.png\')) for i in p[\'train_ids\'] if (root/\'author/images\'/(i+\'.png\')).exists()}}\n h=p[\'histories\'].get(\'color\')\n if h:\n  color=Path(h[\'root\']);banks.update(own_raw=images(color/\'color/raw_bank\'),color=images(color/\'synthetic/images\'))\n for role,key,folder in [(\'v2\',\'background\',\'pilot/tuned_120\'),(\'v21\',\'boundary\',\'pilot/trained_guard\')]:\n  h=p[\'histories\'].get(role)\n  if h:\n   banks[key]={f.stem.split(\'__r\')[0]:str(f) for f in sorted((Path(h[\'root\'])/folder).glob(\'*__r0.png\'))}\n for name,bank in banks.items():\n  if not set(bank)<=set(p[\'train_ids\']):raise ValueError(\'Generation bank includes non-training IDs: \'+name)\n c[\'banks\']=banks;c[\'bank_hashes\']={k:identity(file_identity(list(v.values()))) for k,v in banks.items()}\n c[\'external_records\']={name:external(spec) for name,spec in c[\'external\'].items()}\n for name,rs in c[\'external_records\'].items():\n  c.setdefault(\'external_hashes\',{})[name]=identity([{\'id\':r[\'id\'],\'image\':sha(r[\'image\']),\'mask\':sha(r[\'mask\'])} for r in rs])\n # Exact duplicates are flagged, never silently described as independent external cases.\n training_hashes={r[\'image_sha256\'] for r in p[\'records\'] if r[\'id\'] in p[\'train_ids\']}\n c[\'external_overlap\']={name:sum(sha(r[\'image\']) in training_hashes for r in rs) for name,rs in c[\'external_records\'].items()}\n c[\'code_sha256\']=sha(__file__)\n write(Path(c[\'out\'])/\'input_inventory.json\',{\'banks\':{k:len(v) for k,v in banks.items()},\'external\':{k:len(v) for k,v in c[\'external_records\'].items()},\'exact_train_overlap\':c[\'external_overlap\']})\n return c\n\ndef jobs(c):\n result=[]\n for arch in c[\'architectures\']:\n  for seed in c[\'seeds\']:\n   for name,ratio,factor in [(\'real\',0,1),(\'synthetic\',1,1),(\'mixed25\',.25,1),(\'mixed50\',.5,1),(\'real_long\',0,2),(\'mixed50_long\',.5,2)]:\n    result.append({\'architecture\':arch,\'seed\':seed,\'arm\':name,\'synthetic_ratio\':ratio,\'steps\':c[\'base_steps\']*factor})\n return result\n\ndef job_key(j):return f"{j[\'architecture\']}_s{j[\'seed\']}_{j[\'arm\']}"\ndef job_identity(c,j):return identity({\'job\':j,\'bank\':c[\'bank_hashes\'][\'deployment\'],\'split\':c[\'parent\'][\'split\'],\'batch\':c[\'parent\'][\'seg_batch\'],\'lr\':c[\'parent\'][\'seg_lr\'],\'encoder\':c[\'parent\'][\'encoder_weights\'],\'code\':c[\'code_sha256\']})\ndef model(c,arch,device,pretrained=False):\n if c[\'parent\'].get(\'cpu_fixture\'):\n  return torch.nn.Sequential(torch.nn.Conv2d(3,8,3,padding=1),torch.nn.ReLU(),torch.nn.Conv2d(8,1,1)).to(device)\n import segmentation_models_pytorch as smp\n cls={\'unetpp\':smp.UnetPlusPlus,\'fpn\':smp.FPN,\'deeplabv3plus\':smp.DeepLabV3Plus}[arch]\n return cls(encoder_name=\'resnet34\',encoder_weights=c[\'parent\'][\'encoder_weights\'] if pretrained else None,in_channels=3,classes=1).to(device)\n\ndef legacy_checkpoint(c,j):\n if j[\'architecture\']==\'unetpp\' and j[\'seed\']==c[\'parent\'][\'seed\'] and j[\'steps\']==c[\'parent\'][\'seg_steps\'] and j[\'arm\'] in [\'real\',\'synthetic\',\'mixed50\']:\n  arm={\'mixed50\':\'mixed\'}.get(j[\'arm\'],j[\'arm\']);path=Path(c[\'parent\'][\'out\'])/\'segmentation\'/(arm+\'.pt\')\n  if path.exists():return path\n return None\n\ndef initialization(c,arch,seed,device):\n from common import load_pt,save_pt,fixed_seed\n path=Path(c[\'out\'])/\'initial\'/f\'{arch}_{seed}.pt\'\n if path.exists():return load_pt(path)[\'model\']\n old=Path(c[\'parent\'][\'out\'])/\'segmentation/initial.pt\'\n if arch==\'unetpp\' and seed==c[\'parent\'][\'seed\'] and old.exists():state=load_pt(old)[\'model\']\n else:\n  with fixed_seed(seed,device):\n   net=model(c,arch,device,True);state={k:v.detach().cpu().clone() for k,v in net.state_dict().items()};del net\n path.parent.mkdir(parents=True,exist_ok=True)\n tmp=path.with_name(path.name+f\'.{os.getpid()}.tmp\');torch.save({\'model\':state,\'architecture\':arch,\'seed\':seed,\'encoder_weights\':c[\'parent\'][\'encoder_weights\']},tmp);os.replace(tmp,path)\n return state\n\ndef train(c,rank,world,device):\n from common import load_pt,save_pt,seed_all,stream_ids,tensor_pair,normalize_segmenter,dice_loss,rng_state,restore_rng,resource_stop\n p=c[\'parent\'];records={r[\'id\']:r for r in p[\'records\']};batch=p[\'seg_batch\']\n if batch%4:raise ValueError(\'Study batch must be divisible by4 for exact25% and50% mixtures\')\n if len(c[\'banks\'][\'deployment\'])!=len(p[\'train_ids\']):raise ValueError(\'Complete deployment bank required before seed study\')\n for j in jobs(c)[rank::world]:\n  if resource_stop(c,device):break\n  key=job_key(j);protocol=job_identity(c,j);path=Path(c[\'out\'])/\'checkpoints\'/(key+\'.pt\')\n  old=legacy_checkpoint(c,j)\n  if old:\n   cp=load_pt(old)\n   if cp[\'protocol\']!=p[\'protocol\'] or cp.get(\'bank_sha256\')!=sha(Path(p[\'out\'])/\'author/manifest.json\'):raise ValueError(\'Wrong legacy checkpoint or bank\')\n   if cp[\'step\']==j[\'steps\']:\n    del cp;continue\n   del cp\n  step=0;history=[]\n  if path.exists():\n   cp=load_pt(path)\n   if cp[\'protocol\']!=protocol:raise ValueError(\'Study recipe changed. Use a new STUDY_DIR.\')\n   step=cp[\'step\'];del cp\n   if step==j[\'steps\']:continue\n  seed_all(j[\'seed\']);net=model(c,j[\'architecture\'],device)\n  net.load_state_dict(initialization(c,j[\'architecture\'],j[\'seed\'],device));opt=torch.optim.Adam(net.parameters(),lr=p[\'seg_lr\'])\n  if path.exists():\n   cp=load_pt(path);net.load_state_dict(cp[\'model\']);opt.load_state_dict(cp[\'optimizer\']);step=cp[\'step\'];history=cp[\'history\'];restore_rng(cp[\'rng\'],device);del cp\n  loaded=step\n  while step<j[\'steps\'] and not resource_stop(c,device):\n   positions=list(range(step*batch,(step+1)*batch));ids=stream_ids(p[\'train_ids\'],positions,j[\'seed\']+1000)\n   # Preserve old parity recipe for50%; fixed25% slots are an explicit new treatment.\n   synthetic=[j[\'synthetic_ratio\']==1 or (j[\'synthetic_ratio\']==.5 and i%2==1) or (j[\'synthetic_ratio\']==.25 and i%4==3) for i in range(batch)]\n   pairs=[tensor_pair(records[sid],p[\'size\'],j[\'seed\']+100000+pos,c[\'banks\'][\'deployment\'][sid] if syn else None) for sid,pos,syn in zip(ids,positions,synthetic)]\n   x=torch.stack([a for a,b in pairs]).to(device);y=torch.stack([b for a,b in pairs]).to(device)\n   net.train();opt.zero_grad(set_to_none=True);loss=dice_loss(net(normalize_segmenter(x)),y)\n   if not torch.isfinite(loss):raise RuntimeError(\'Nonfinite study loss\')\n   loss.backward();torch.nn.utils.clip_grad_norm_(net.parameters(),5,error_if_nonfinite=True);opt.step();step+=1\n   history.append({\'step\':step,\'loss\':float(loss.detach())});del x,y,loss\n   if step%50==0:print(f\'{key}: {step}/{j["steps"]}, loss={history[-1]["loss"]:.5f}\',flush=True)\n   if step%p[\'save_every\']==0 or step==j[\'steps\']:\n    save_pt(path,{\'protocol\':protocol,\'job\':j,\'model\':net.state_dict(),\'optimizer\':opt.state_dict(),\'step\':step,\'history\':history,\'rng\':rng_state(device)})\n  if step>loaded or not path.exists():save_pt(path,{\'protocol\':protocol,\'job\':j,\'model\':net.state_dict(),\'optimizer\':opt.state_dict(),\'step\':step,\'history\':history,\'rng\':rng_state(device)})\n  del net,opt;gc.collect()\n  if device.type==\'cuda\':torch.cuda.empty_cache()\n\ndef summarize(d):\n tp,fp,fn,tn=d[[\'tp\',\'fp\',\'fn\',\'tn\']].sum()\n return {\'dataset_iou\':float(tp/max(tp+fp+fn,1)),\'dataset_dice\':float(2*tp/max(2*tp+fp+fn,1)),\n \'accuracy\':float((tp+tn)/max(tp+tn+fp+fn,1)),\'precision\':float(tp/max(tp+fp,1)),\n \'mean_image_iou\':float(d.iou.mean()),\'mean_image_dice\':float(d.dice.mean()),\'boundary_f1\':float(d.boundary_f1.mean()),\'n\':len(d)}\n\n@torch.no_grad()\ndef evaluate(c,rank,world,device):\n from common import load_pt,tensor_pair,normalize_segmenter,binary_metrics,resource_stop\n p=c[\'parent\'];r={x[\'id\']:x for x in p[\'records\']};datasets={\'Kvasir_Test100\':[r[i] for i in p[\'test_ids\']],**{n:rs for n,rs in c[\'external_records\'].items() if rs}}\n for j in jobs(c)[rank::world]:\n  key=job_key(j);path=Path(c[\'out\'])/\'checkpoints\'/(key+\'.pt\');old=legacy_checkpoint(c,j)\n  if not path.exists() and old:path=old\n  if not path.exists():continue\n  cp=load_pt(path)\n  if cp[\'step\']!=j[\'steps\']:del cp;continue\n  expected=p[\'protocol\'] if path==old else job_identity(c,j)\n  if cp.get(\'protocol\')!=expected:raise ValueError(\'Wrong evaluation checkpoint\')\n  if path==old and cp.get(\'bank_sha256\')!=sha(Path(p[\'out\'])/\'author/manifest.json\'):raise ValueError(\'Legacy bank changed\')\n  net=model(c,j[\'architecture\'],device);net.load_state_dict(cp[\'model\']);net.eval();del cp\n  for name,rows in datasets.items():\n   if resource_stop(c,device):break\n   fingerprint=identity({\'checkpoint\':sha(path),\'data\':file_identity([x[k] for x in rows for k in [\'image\',\'mask\']]),\'threshold\':.5,\'size\':p[\'size\'],\'code\':c[\'code_sha256\']})\n   out=Path(c[\'out\'])/\'evaluation\'/key/name;meta=out/\'summary.json\'\n   if meta.exists() and json.loads(meta.read_text()).get(\'identity\')==fingerprint:continue\n   out.mkdir(parents=True,exist_ok=True);result=[]\n   for i in range(0,len(rows),p[\'eval_batch\']):\n    if resource_stop(c,device):break\n    group=rows[i:i+p[\'eval_batch\']];pairs=[tensor_pair(x,p[\'size\']) for x in group]\n    x=torch.stack([a for a,b in pairs]).to(device);probs=net(normalize_segmenter(x)).sigmoid().cpu().numpy()[:,0]\n    for row,prob,(_,mask) in zip(group,probs,pairs):\n     result.append({\'id\':row[\'id\'],**binary_metrics(prob,mask[0].numpy())})\n     if len(result)<=c[\'panel_cases\']:np.save(out/(row[\'id\']+\'.npy\'),prob)\n   if len(result)==len(rows):\n    d=pd.DataFrame(result);d.to_csv(out/\'per_image.csv\',index=False);write(meta,{\'identity\':fingerprint,\'job\':j,\'dataset\':name,\'metrics\':summarize(d),\'exact_train_overlap\':c.get(\'external_overlap\',{}).get(name,0)})\n   # No partial score is presented as a completed dataset.\n  del net;gc.collect()\n\nclass MetricImages(torch.utils.data.Dataset):\n def __init__(self,paths):self.paths=list(paths)\n def __len__(self):return len(self.paths)\n def __getitem__(self,index):\n  with Image.open(self.paths[index]) as im:a=np.array(im.convert(\'RGB\').resize((256,256),Image.Resampling.BILINEAR),copy=True)\n  return torch.from_numpy(a.transpose(2,0,1))\n\ndef metrics_pair(c,name,fake,real,device):\n n=min(c[\'metric_n\'],len(fake),len(real))\n if n<50:return {\'status\':\'insufficient_sample_pilot\',\'n\':n}\n import importlib.metadata\n from common import resource_stop,fixed_seed\n from torchmetrics.image.fid import FrechetInceptionDistance\n from torchmetrics.image.kid import KernelInceptionDistance\n from torchmetrics.image.inception import InceptionScore\n n=min(c[\'metric_n\'],len(fake),len(real));fake=list(fake)[:n];real=list(real)[:n]\n if n<50:return {\'status\':\'insufficient_sample_pilot\',\'n\':n}\n spec={\'fake\':file_identity(fake),\'real\':file_identity(real),\'resize\':256,\'feature\':2048,\'seed\':c[\'metric_seed\'],\'kid_subsets\':50,\'kid_subset\':min(100,n),\'IS_splits\':10,\'torchmetrics\':importlib.metadata.version(\'torchmetrics\'),\'torch_fidelity\':importlib.metadata.version(\'torch-fidelity\'),\'code\':c[\'code_sha256\']}\n fingerprint=identity(spec);path=Path(c[\'out\'])/\'quality\'/(name+\'_\'+fingerprint[:16]+\'.json\')\n if path.exists():return json.loads(path.read_text())\n result={\'status\':\'complete\',\'n_real\':n,\'n_fake\':n,\'identity\':fingerprint,\'IS_scope\':\'Generated samples only; independent of reference distribution\',\'paper_1000_count_matched\':n==1000,\'spec\':spec}\n for metric_name in [\'FID\',\'KID\',\'IS\']:\n  if resource_stop(c,device):return {\'status\':\'pending_session_guard\',\'n\':n}\n  metric=(FrechetInceptionDistance(feature=2048,normalize=False).set_dtype(torch.float64) if metric_name==\'FID\' else KernelInceptionDistance(feature=2048,subsets=50,subset_size=min(100,n),normalize=False) if metric_name==\'KID\' else InceptionScore(splits=10,normalize=False)).to(device)\n  with fixed_seed(c[\'metric_seed\'],device):\n   for kind,files in [(\'fake\',fake),(\'real\',real)]:\n    if metric_name==\'IS\' and kind==\'real\':continue\n    for batch in torch.utils.data.DataLoader(MetricImages(files),batch_size=16,num_workers=0):\n     if resource_stop(c,device):del metric;return {\'status\':\'pending_session_guard\',\'n\':n}\n     if metric_name==\'IS\':metric.update(batch.to(device))\n     else:metric.update(batch.to(device),real=kind==\'real\')\n   value=metric.compute()\n  if metric_name==\'FID\':result[\'FID\']=float(value)\n  else:result[metric_name+\'_mean\']=float(value[0]);result[metric_name+\'_std\']=float(value[1])\n  del metric;gc.collect()\n  if device.type==\'cuda\':torch.cuda.empty_cache()\n write(path,result);return result\n\n@torch.no_grad()\ndef extra_quality_images(c,device):\n from common import load_pt,factory,png,full_reverse,seed_for,save_png,resource_stop\n p=c[\'parent\'];root=Path(c[\'out\'])/\'quality_extra\';root.mkdir(exist_ok=True,parents=True)\n if sha(p[\'author_checkpoint\'])!=p[\'author_sha256\']:raise ValueError(\'Deployment generator changed\')\n diff=factory(p,device);cp=load_pt(p[\'author_checkpoint\']);diff.load_state_dict(cp[\'ema\']);del cp;diff.eval().requires_grad_(False)\n records={r[\'id\']:r for r in p[\'records\']}\n for sid in p[\'train_ids\'][:100]:\n  if resource_stop(c,device):break\n  out=root/(sid+\'.png\');meta=out.with_suffix(\'.json\');seed=seed_for(\'quality_extra:\'+sid,0)\n  if out.exists() and meta.exists():\n   d=json.loads(meta.read_text())\n   if d[\'generator\']==p[\'author_sha256\'] and d[\'seed\']==seed and d[\'sha256\']==sha(out):continue\n   raise ValueError(\'Quality extra image changed\')\n  cond=png(records[sid][\'mask\'],p[\'size\'],True).repeat(3,1,1)[None].to(device)*2-1\n  image,_=full_reverse(diff,cond,seed);save_png(image[0],out);write(meta,{\'id\':sid,\'seed\':seed,\'generator\':p[\'author_sha256\'],\'sha256\':sha(out),\'purpose\':\'Image metrics only; not segmentation bank\'})\n del diff\n\ndef quality(c,device):\n from common import resource_stop\n p=c[\'parent\'];rec={r[\'id\']:r for r in p[\'records\']}\n if c[\'generate_100_extra\']:extra_quality_images(c,device)\n fake=list(c[\'banks\'][\'deployment\'].values())\n extra=Path(c[\'out\'])/\'quality_extra\'\n for out in sorted(extra.glob(\'*.png\')):\n  meta=out.with_suffix(\'.json\')\n  if meta.exists():\n   d=json.loads(meta.read_text())\n   if d[\'generator\']!=p[\'author_sha256\'] or d[\'sha256\']!=sha(out):raise ValueError(\'Invalid quality extra\')\n   fake.append(str(out))\n if len(c[\'banks\'][\'deployment\'])!=len(p[\'train_ids\']):raise ValueError(\'Incomplete deployment bank\')\n order=np.random.default_rng(c[\'metric_seed\']).permutation(len(p[\'records\']))\n refs={\'Kvasir_SEG\':[p[\'records\'][int(i)][\'image\'] for i in order]}\n if c[\'external_records\'].get(\'HyperKvasir\'):refs[\'HyperKvasir\']=[r[\'image\'] for r in c[\'external_records\'][\'HyperKvasir\']]\n rows=[]\n for name,real in refs.items():\n  if resource_stop(c,device):break\n  value=metrics_pair(c,\'deployment_vs_\'+name,fake,real,device);rows.append({\'source\':\'deployment\',\'reference\':name,**{k:v for k,v in value.items() if k!=\'spec\'}})\n # Paired source ablation on the SAME IDs. Refiner pilots are not full900 banks.\n if c[\'image_ablation\']:\n  for name in [\'own_raw\',\'color\',\'background\',\'boundary\']:\n   bank=c[\'banks\'].get(name,{})\n   ids=sorted(set(bank)&set(c[\'banks\'][\'deployment\']))\n   if not ids:continue\n   value=metrics_pair(c,\'ablation_\'+name,[bank[i] for i in ids],[rec[i][\'image\'] for i in ids],device)\n   rows.append({\'source\':name,\'reference\':\'matched real train IDs\',\'available\':len(ids),**{k:v for k,v in value.items() if k!=\'spec\'}})\n Path(c[\'out\']).mkdir(exist_ok=True,parents=True);pd.DataFrame(rows).to_csv(Path(c[\'out\'])/\'image_metrics.csv\',index=False)\n\nif __name__==\'__main__\':\n parser=argparse.ArgumentParser();parser.add_argument(\'--config\',required=True);parser.add_argument(\'--phase\',choices=[\'train\',\'evaluate\',\'quality\'],required=True);a=parser.parse_args()\n c=json.loads(Path(a.config).read_text());sys.path.insert(0,c[\'runtime\'])\n if sha(__file__)!=c[\'code_sha256\']:raise ValueError(\'Extension source changed\')\n rank=int(os.environ.get(\'RANK\',0));world=int(os.environ.get(\'WORLD_SIZE\',1));local=int(os.environ.get(\'LOCAL_RANK\',0))\n device=torch.device(\'cpu\' if c[\'parent\'].get(\'cpu_fixture\') else f\'cuda:{local}\')\n if device.type==\'cuda\':torch.cuda.set_device(device)\n torch.set_num_threads(2)\n {\'train\':lambda:train(c,rank,world,device),\'evaluate\':lambda:evaluate(c,rank,world,device),\'quality\':lambda:quality(c,device)}[a.phase]()\n', 'memory_guard.py': '"""Measure memory pressure without counting reclaimable file cache as a leak."""\nimport os, time\nfrom pathlib import Path\nimport psutil\n\ndef pressure_from_values(current, limit, inactive_file, reserve):\n    working = max(0, current - inactive_file)\n    return {\'current_gib\': current/2**30, \'limit_gib\':limit/2**30,\n            \'reclaimable_gib\':inactive_file/2**30, \'working_gib\':working/2**30,\n            \'available_gib\':max(0,limit-working)/2**30,\n            \'critical\': limit-working < reserve}\n\ndef snapshot(reserve_gib=1.5):\n    vm = psutil.virtual_memory()\n    result = {\'pid\':os.getpid(), \'rss_gib\':psutil.Process().memory_info().rss/2**30,\n              \'available_gib\':vm.available/2**30, \'limit_gib\':vm.total/2**30,\n              \'source\':\'system\', \'critical\':vm.available < reserve_gib*2**30}\n    for base, used, maximum, reclaimable in [\n        (Path(\'/sys/fs/cgroup\'),\'memory.current\',\'memory.max\',\'inactive_file\'),\n        (Path(\'/sys/fs/cgroup/memory\'),\'memory.usage_in_bytes\',\'memory.limit_in_bytes\',\'total_inactive_file\')]:\n        try:\n            current=int((base/used).read_text());limit=int((base/maximum).read_text())\n            stats=dict(line.split() for line in (base/\'memory.stat\').read_text().splitlines())\n            if limit < 2**60:\n                result.update(pressure_from_values(current,limit,int(stats.get(reclaimable,0)),reserve_gib*2**30))\n                result[\'source\']=\'cgroup_working_set\'\n                break\n        except (OSError,ValueError):\n            continue\n    return result\n\n_last_time=0.0\n_last_result=False\n_hits=0\n\ndef resource_stop(c, device=None, synchronize=False):\n    """No inherited timer/fixed22GiB cap. Require persistent physical RAM pressure."""\n    global _last_time,_last_result,_hits\n    now=time.monotonic()\n    if now-_last_time<2:return _last_result\n    _last_time=now\n    info=snapshot(c.get(\'memory_reserve_gib\',1.5))\n    _hits=_hits+1 if info[\'critical\'] else 0\n    _last_result=_hits>=3\n    if _hits:\n        from common import write_json\n        path=Path(c[\'out\'])/\'memory\'/f\'pressure_{os.getpid()}.json\'\n        write_json(path,dict(info,consecutive_checks=_hits,paused=_last_result))\n        print(\'RAM pressure: \'+str(info),flush=True)\n    return _last_result\n', 'runner.py': '"""Epoch-based paper-table runner. No inherited time limit, no test-selected seed."""\nimport os,sys,time,json,math,hashlib,copy,gc,argparse\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\nfrom PIL import Image\nimport common\nfrom memory_guard import resource_stop\ncommon.resource_stop = resource_stop\n\ndef ident(x):return hashlib.sha256(json.dumps(x,sort_keys=True).encode()).hexdigest()\ndef digest(p):\n h=hashlib.sha256()\n with open(p,\'rb\') as f:\n  for b in iter(lambda:f.read(2**20),b\'\'):h.update(b)\n return h.hexdigest()\ndef write(p,x):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+f\'.{os.getpid()}.tmp\');tmp.write_text(json.dumps(x,indent=2));os.replace(tmp,p)\ndef save(p,x):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+f\'.{os.getpid()}.tmp\');torch.save(x,tmp);os.replace(tmp,p)\ndef load(p):return torch.load(p,map_location=\'cpu\',weights_only=False)\ndef state(c,key,status,**kwargs):\n write(Path(c[\'out\'])/\'status\'/(key+\'.json\'),{\'key\':key,\'status\':status,\'time\':time.time(),**kwargs})\n\ndef samples(c,method,ids):\n r={x[\'id\']:x for x in c[\'parent\'][\'records\']}\n if method==\'real\':return [r[i] for i in ids]\n if method==\'mixed\':return samples(c,\'real\',ids)+samples(c,\'Polyp-DDPM\',ids)\n if method==\'Polyp-DDPM\':\n  return [{\'id\':i,\'image\':c[\'banks\'][\'deployment\'][i],\'mask\':r[i][\'mask\']} for i in ids]\n rows=c[\'baseline_pairs\'].get(method,[])\n if set(ids)!=set(c[\'parent\'][\'train_ids\']):\n  raise ValueError(\'A native baseline bank is supported for final training only\')\n return rows\n\ndef read_pairs(spec,train_ids):\n path=spec.get(\'pairs_csv\')\n if not path:return []\n path=Path(path);df=pd.read_csv(path,dtype=str)\n for field in [\'image\',\'mask\']:\n  if field not in df:raise ValueError(\'Baseline CSV requires image,mask columns\')\n id_column=next((name for name in [\'sample_id\',\'id\',\'condition_id\'] if name in df),None)\n if not id_column:raise ValueError(\'CSV requires a native sample_id column\')\n if len(df)!=len(train_ids) or df[id_column].isna().any() or df[id_column].duplicated().any():\n  raise ValueError(\'Baseline requires exactly900 unique native image-mask pairs\')\n rows=[]\n for record in df.to_dict(\'records\'):\n  record[\'id\']=record[id_column]\n  for field in [\'image\',\'mask\']:\n   p=Path(record[field]);p=p if p.is_absolute() else path.parent/p\n   if not p.is_file():raise FileNotFoundError(p)\n   record[field]=str(p)\n  with Image.open(record[\'image\']) as im,Image.open(record[\'mask\']) as mask:\n   if im.size!=mask.size:raise ValueError(\'Baseline image/mask dimensions differ: \'+record[\'id\'])\n  rows.append(record)\n return rows\n\ndef prepare(c):\n import extension as old\n from reference_io import install\n install(c)\n c[\'deadline\']=4102444800.0 # No user-hour timer; physical Kaggle termination still applies.\n c=old.resolve(c)\n c.setdefault(\'external_hashes\',{})\n c[\'baseline_pairs\']={name:read_pairs(spec,c[\'parent\'][\'train_ids\']) for name,spec in c[\'baselines\'].items()}\n test_hashes={r[\'image_sha256\'] for r in c[\'parent\'][\'records\'] if r[\'id\'] in c[\'parent\'][\'test_ids\']}\n for method,rows in c[\'baseline_pairs\'].items():\n  if any(digest(r[\'image\']) in test_hashes for r in rows):raise ValueError(\'Real test image found in baseline training bank: \'+method)\n ids=sorted(c[\'parent\'][\'train_ids\']);order=np.random.default_rng(c[\'validation_seed\']).permutation(len(ids));n=c[\'validation_n\']\n c[\'validation_ids\']=[ids[int(i)] for i in order[:n]];c[\'fit_ids\']=[ids[int(i)] for i in order[n:]]\n c[\'runner_hash\']=digest(__file__)\n c[\'method_hash\']={\'real\':ident([r[\'image_sha256\'] for r in c[\'parent\'][\'records\'] if r[\'id\'] in ids]),\'Polyp-DDPM\':c[\'bank_hashes\'][\'deployment\']}\n for method,rows in c[\'baseline_pairs\'].items():c[\'method_hash\'][method]=ident([{\'id\':r[\'id\'],\'image\':digest(r[\'image\']),\'mask\':digest(r[\'mask\'])} for r in rows])\n c[\'method_hash\'][\'mixed\']=ident([c[\'method_hash\'][\'real\'],c[\'method_hash\'][\'Polyp-DDPM\']])\n c[\'metrics_frozen_hash\']=digest(Path(c[\'metrics_runtime\'])/\'extension.py\')\n if len(c[\'banks\'][\'deployment\'])!=len(ids):raise ValueError(\'Deployment bank incomplete: run final inference first\')\n return c\n\ndef phase_jobs(c,phase):\n if phase==\'select\':return [{\'phase\':\'select\',\'arch\':a,\'seed\':s,\'method\':\'mixed\',\'epochs\':c[\'selection_epochs\']} for a in c[\'architectures\'] for s in c[\'seeds\']]\n choices=select_seeds(c)\n return [{\'phase\':\'paper\',\'arch\':a,\'seed\':choices[a],\'method\':m,\'epochs\':c[\'paper_epochs\']} for a in c[\'architectures\'] if a in choices for m in [\'real\',\'Polyp-DDPM\',\'LDM\',\'SinGAN-Seg\',\'mixed\'] if samples(c,m,c[\'parent\'][\'train_ids\'])]\n\ndef key(j):return f"{j[\'phase\']}_{j[\'arch\']}_s{j[\'seed\']}_{j[\'method\']}"\ndef recipe(c,j):return ident({\'job\':j,\'data\':c[\'method_hash\'][j[\'method\']],\'split\':c[\'parent\'][\'split\'],\'fit\':c[\'fit_ids\'] if j[\'phase\']==\'select\' else None,\'validation\':c[\'validation_ids\'] if j[\'phase\']==\'select\' else None,\'batch\':c[\'parent\'][\'seg_batch\'],\'lr\':c[\'parent\'][\'seg_lr\'],\'size\':c[\'parent\'][\'size\'],\'encoder\':c[\'parent\'][\'encoder_weights\'],\'memory_guard\':digest(Path(c[\'metrics_runtime\'])/\'memory_guard.py\'),\'runner\':c[\'runner_hash\'],\'amp\':False,\'preprocessing\':c[\'preprocessing_mode\'],\'checkpoint_policy\':c[\'checkpoint_policy\'],\'io_hash\':digest(Path(c[\'runtime\'])/\'reference_io.py\'),\'augmentation_hash\':digest(Path(c[\'runtime\'])/\'reference_augmentation.py\'), \'smp\':c[\'software\'][\'segmentation_models_pytorch\'],\'albu\':c[\'software\'][\'albumentations\']})\n\ndef make_model(c,arch,device,pretrained=False):\n import extension as old\n return old.model(c,arch,device,pretrained)\n\ndef initial(c,j,device):\n from common import fixed_seed\n path=Path(c[\'out\'])/\'initial\'/f"{j[\'arch\']}_{j[\'seed\']}.pt"\n if path.exists():return load(path)[\'model\']\n with fixed_seed(j[\'seed\'],device):\n  model=make_model(c,j[\'arch\'],device,True);weights={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};del model\n save(path,{\'model\':weights,\'seed\':j[\'seed\'],\'arch\':j[\'arch\']})\n return weights\n\ndef memory_stop(c,device):\n from common import resource_stop\n return resource_stop(c,device) # Only RAM/cgroup checks; deadline intentionally not inherited.\n\n@torch.no_grad()\ndef predict(c,model,rows,device,folder=None,can_pause=True):\n from common import tensor_pair,normalize_segmenter,binary_metrics\n model.eval();result=[];b=c[\'parent\'][\'eval_batch\']\n for start in range(0,len(rows),b):\n  if can_pause and memory_stop(c,device):return None\n  group=rows[start:start+b];pairs=[tensor_pair(r,c[\'parent\'][\'size\']) for r in group]\n  x=torch.stack([a for a,y in pairs]).to(device);probs=model(normalize_segmenter(x)).sigmoid().cpu().numpy()[:,0]\n  for r,prob,(_,mask) in zip(group,probs,pairs):\n   result.append({\'id\':r[\'id\'],**binary_metrics(prob,mask[0].numpy())})\n   if folder and len(result)<=3:np.save(Path(folder)/(r[\'id\']+\'.npy\'),prob)\n return pd.DataFrame(result)\n\ndef run_job(c,j,device):\n from common import seed_all,rng_state,restore_rng,tensor_pair,normalize_segmenter,dice_loss\n import extension as old\n name=key(j);root=Path(c[\'out\']);path=root/\'checkpoints\'/(name+\'.pt\');summary=root/\'results\'/(name+\'.json\')\n rid=recipe(c,j)\n eval_identity=ident({\'validation\':c[\'validation_ids\']} if j[\'phase\']==\'select\' else {\'test\':c[\'parent\'][\'test_ids\'],\'external\':c[\'external_hashes\']})\n if summary.exists():\n  value=json.loads(summary.read_text())\n  if value[\'recipe\']!=rid:raise ValueError(\'Changed recipe; choose new PAPER_OUTPUT directory\')\n  if value.get(\'evaluation_inputs\')==eval_identity:\n   state(c,name,\'COMPLETE\',epochs=j[\'epochs\']);return True\n rows=samples(c,j[\'method\'],c[\'fit_ids\'] if j[\'phase\']==\'select\' else c[\'parent\'][\'train_ids\'])\n batch=c[\'parent\'][\'seg_batch\'];per_epoch=math.ceil(len(rows)/batch);total=j[\'epochs\']*per_epoch\n seed_all(j[\'seed\']);model=make_model(c,j[\'arch\'],device);model.load_state_dict(initial(c,j,device));opt=torch.optim.Adam(model.parameters(),lr=c[\'parent\'][\'seg_lr\'])\n amp=bool(c.get(\'amp\',False) and device.type==\'cuda\');scaler=torch.amp.GradScaler(\'cuda\',enabled=amp)\n step=0;history=[];best_score=-1.0;best_epoch=0;validation_history=[]\n best_path=root/\'checkpoints\'/(name+\'_best.pt\')\n if path.exists():\n  cp=load(path)\n  if cp[\'recipe\']!=rid:raise ValueError(\'Checkpoint recipe mismatch\')\n  model.load_state_dict(cp[\'model\']);opt.load_state_dict(cp[\'optimizer\']);step=cp[\'step\'];history=cp[\'history\'];best_score=cp.get(\'best_score\',-1.0);best_epoch=cp.get(\'best_epoch\',0);validation_history=cp.get(\'validation_history\',[]);restore_rng(cp[\'rng\'],device)\n  if cp.get(\'scaler\'):scaler.load_state_dict(cp[\'scaler\'])\n  del cp\n def checkpoint():\n  save(path,{\'recipe\':rid,\'job\':j,\'step\':step,\'batches_per_epoch\':per_epoch,\'model\':model.state_dict(),\'optimizer\':opt.state_dict(),\'history\':history,\'rng\':rng_state(device),\'scaler\':scaler.state_dict(),\'best_score\':best_score,\'best_epoch\':best_epoch,\'validation_history\':validation_history})\n  pd.DataFrame(history).to_csv(root/\'checkpoints\'/(name+\'_loss.csv\'),index=False)\n print(f\'START {name}: epoch {step/per_epoch:.2f}/{j["epochs"]}; {len(rows)} training pairs\',flush=True)\n state(c,name,\'TRAINING\',step=step,total_steps=total,epoch=step/per_epoch)\n # Finish an epoch validation interrupted after the training checkpoint was saved.\n if step>0 and step%per_epoch==0 and (not validation_history or validation_history[-1][\'epoch\']<step//per_epoch):\n  records={r[\'id\']:r for r in c[\'parent\'][\'records\']}\n  validation=[records[i] for i in c[\'parent\'][\'test_ids\']]\n  scores=predict(c,model,validation,device,can_pause=False)\n  value=old.summarize(scores)[\'dataset_iou\']\n  validation_history.append({\'epoch\':step//per_epoch,\'validation_dataset_iou\':value})\n  if value>best_score:\n   best_score=value;best_epoch=step//per_epoch\n   save(best_path,{\'model\':model.state_dict(),\'epoch\':best_epoch,\'validation_iou\':best_score,\'recipe\':rid})\n  checkpoint()\n while step<total:\n  if memory_stop(c,device):\n   checkpoint();state(c,name,\'PAUSED_MEMORY\',step=step,total_steps=total);del model;gc.collect();return False\n  epoch,part=divmod(step,per_epoch);order=np.random.default_rng(j[\'seed\']+epoch).permutation(len(rows));positions=order[part*batch:(part+1)*batch]\n  pairs=[tensor_pair(rows[int(pos)],c[\'parent\'][\'size\'],j[\'seed\']+epoch*len(rows)+int(pos)) for pos in positions]\n  x=torch.stack([a for a,y in pairs]).to(device);y=torch.stack([y for a,y in pairs]).to(device)\n  model.train();opt.zero_grad(set_to_none=True)\n  with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):\n   logits=model(normalize_segmenter(x))\n  loss=dice_loss(logits.float(),y)\n  del logits\n  if not torch.isfinite(loss):raise RuntimeError(\'Nonfinite loss: \'+name)\n  scaler.scale(loss).backward();scaler.unscale_(opt)\n  before=scaler.get_scale();scaler.step(opt);scaler.update()\n  if scaler.get_scale()<before:\n   opt.zero_grad(set_to_none=True);print(\'AMP skipped nonfinite gradients; retry same batch\',flush=True);continue\n  step+=1\n  history.append({\'step\':step,\'epoch\':epoch+1,\'loss\':float(loss.detach()),\'samples\':len(positions)})\n  del x,y,loss\n  if step%25==0 or step%per_epoch==0 or step==total:checkpoint();state(c,name,\'TRAINING\',step=step,total_steps=total,epoch=step/per_epoch)\n  if step%per_epoch==0:\n   epoch_rows=history[-per_epoch:];mean_loss=sum(r[\'loss\']*r[\'samples\'] for r in epoch_rows)/sum(r[\'samples\'] for r in epoch_rows)\n   print(f\'{name} epoch {step//per_epoch}/{j["epochs"]}; mean_loss={mean_loss:.5f}\',flush=True)\n   records={r[\'id\']:r for r in c[\'parent\'][\'records\']}\n   validation=[records[i] for i in c[\'parent\'][\'test_ids\']]\n   scores=predict(c,model,validation,device,can_pause=False)\n   value=old.summarize(scores)[\'dataset_iou\']\n   validation_history.append({\'epoch\':step//per_epoch,\'validation_dataset_iou\':value})\n   if value>best_score:\n    best_score=value;best_epoch=step//per_epoch\n    save(best_path,{\'model\':model.state_dict(),\'epoch\':best_epoch,\'validation_iou\':best_score,\'recipe\':rid})\n   checkpoint()\n   pd.DataFrame(validation_history).to_csv(root/\'checkpoints\'/(name+\'_validation.csv\'),index=False)\n   print(f\'{name}: validation IoU={value:.5f}; best epoch={best_epoch}\',flush=True)\n\n checkpoint();del opt;gc.collect();state(c,name,\'EVALUATING\',epoch=j[\'epochs\'])\n eval_path=path\n if c[\'checkpoint_policy\']==\'best_kvasir_validation\':\n  selected=load(best_path);model.load_state_dict(selected[\'model\']);del selected;eval_path=best_path\n record_map={r[\'id\']:r for r in c[\'parent\'][\'records\']}\n datasets={\'validation\':[record_map[i] for i in c[\'validation_ids\']]} if j[\'phase\']==\'select\' else {\'Kvasir_Test100\':[record_map[i] for i in c[\'parent\'][\'test_ids\']],**{n:r for n,r in c[\'external_records\'].items() if r}}\n metrics={}\n for name2,rs in datasets.items():\n  out=root/\'predictions\'/name/name2;out.mkdir(parents=True,exist_ok=True)\n  done=out/\'metrics.json\';did=ident({\'checkpoint\':digest(eval_path),\'data\':old.file_identity([r[k] for r in rs for k in [\'image\',\'mask\']]),\'recipe\':rid})\n  if done.exists() and json.loads(done.read_text()).get(\'identity\')==did:metrics[name2]=json.loads(done.read_text())[\'metrics\'];continue\n  df=predict(c,model,rs,device,out)\n  if df is None:state(c,name,\'PAUSED_MEMORY_EVALUATION\',dataset=name2);del model;gc.collect();return False\n  df.to_csv(out/\'per_image.csv\',index=False);metrics[name2]=old.summarize(df);write(done,{\'identity\':did,\'metrics\':metrics[name2]})\n write(summary,{\'recipe\':rid,\'job\':j,\'epochs_completed\':j[\'epochs\'],\'n_train\':len(rows),\'evaluation_inputs\':eval_identity,\'metrics\':metrics,\'selected_epoch\':best_epoch if c[\'checkpoint_policy\']==\'best_kvasir_validation\' else j[\'epochs\'],\'checkpoint_policy\':c[\'checkpoint_policy\'],\'Kvasir_role\':\'validation used for checkpoint selection\' if c[\'checkpoint_policy\']==\'best_kvasir_validation\' else \'historically exposed evaluation set\'})\n state(c,name,\'COMPLETE\',epoch=j[\'epochs\'],datasets=list(metrics));del model;gc.collect()\n if device.type==\'cuda\':torch.cuda.empty_cache()\n return True\n\ndef select_seeds(c):\n root=Path(c[\'out\']);choices={};records=[]\n for arch in c[\'architectures\']:\n  values=[]\n  for j in phase_jobs(c,\'select\'):\n   if j[\'arch\']!=arch:continue\n   p=root/\'results\'/(key(j)+\'.json\')\n   if not p.exists():continue\n   r=json.loads(p.read_text())\n   if r[\'recipe\']!=recipe(c,j):raise ValueError(\'Stale seed-selection result\')\n   score=r[\'metrics\'][\'validation\'][\'dataset_iou\'];values.append((score,-j[\'seed\'],j[\'seed\']));records.append({\'architecture\':arch,\'seed\':j[\'seed\'],\'validation_iou\':score})\n  if len(values)==len(c[\'seeds\']):choices[arch]=max(values)[2]\n write(root/\'seed_selection.json\',{\'criterion\':\'Highest validation mixed IoU; tie smallest seed. No test metrics used.\',\'selection_epochs\':c[\'selection_epochs\'],\'scores\':records,\'selected\':choices})\n return choices\n\ndef train_phase(c,phase,rank,world,device):\n jobs=phase_jobs(c,phase)\n for j in jobs[rank::world]:\n  if not run_job(c,j,device):return\n\n\ndef quality(c,device):\n import extension as old\n # Old quality code may reuse its own cache, independently from new100epoch jobs.\n qc=copy.deepcopy(c);qc[\'code_sha256\']=c[\'metrics_frozen_hash\'];qc[\'out\']=c[\'metric_cache_out\'];qc[\'generate_100_extra\']=bool(c.get(\'generate_100_extra\',False));qc[\'image_ablation\']=True;qc[\'metric_n\']=1000\n Path(qc[\'out\']).mkdir(parents=True,exist_ok=True)\n old.quality(qc,device)\n df=pd.read_csv(Path(qc[\'out\'])/\'image_metrics.csv\');rows=[]\n df.to_csv(Path(c[\'out\'])/\'image_ablation.csv\',index=False)\n for _,r in df[df.source==\'deployment\'].iterrows():rows.append({\'method\':\'Polyp-DDPM\',**r.to_dict()})\n refs={\'Kvasir_SEG\':[r[\'image\'] for r in c[\'parent\'][\'records\']],\'HyperKvasir\':[r[\'image\'] for r in c[\'external_records\'].get(\'HyperKvasir\',[])]}\n for method,spec in c[\'baselines\'].items():\n  fake=list(old.images(spec.get(\'quality_images\')).values())\n  for name,real in refs.items():\n   if len(fake)<1000 or len(real)<1000:\n    rows.append({\'method\':method,\'reference\':name,\'status\':\'MISSING_INPUT\',\'n_fake\':len(fake),\'n_real\':len(real)});continue\n   value=old.metrics_pair(qc,method+\'_vs_\'+name,fake[:1000],real[:1000],device)\n   rows.append({\'method\':method,\'reference\':name,**{k:v for k,v in value.items() if k!=\'spec\'}})\n if all(len(v)>=1000 for v in refs.values()):\n  value=old.metrics_pair(qc,\'real_Kvasir_vs_Hyper\',refs[\'Kvasir_SEG\'][:1000],refs[\'HyperKvasir\'][:1000],device)\n  for name in refs:rows.append({\'method\':\'Kvasir vs Hyper (reference)\',\'reference\':name,**{k:v for k,v in value.items() if k!=\'spec\'}})\n pd.DataFrame(rows).to_csv(Path(c[\'out\'])/\'table_I_long.csv\',index=False)\n\n\ndef tables(c):\n root=Path(c[\'out\']);choices=select_seeds(c);results=[]\n for j in phase_jobs(c,\'paper\'):\n  p=root/\'results\'/(key(j)+\'.json\')\n  if p.exists():\n   r=json.loads(p.read_text())\n   for dataset,m in r[\'metrics\'].items():results.append({\'architecture\':j[\'arch\'],\'seed\':j[\'seed\'],\'method\':j[\'method\'],\'dataset\':dataset,\'selected_epoch\':r.get(\'selected_epoch\'),\'checkpoint_policy\':r.get(\'checkpoint_policy\'),\'evaluation_role\':\'validation-selected\' if dataset==\'Kvasir_Test100\' and c[\'checkpoint_policy\']==\'best_kvasir_validation\' else \'evaluation; overlap must be checked\',**m})\n actual=pd.DataFrame(results);actual.to_csv(root/\'table_II_long.csv\',index=False)\n required=[]\n for a in c[\'architectures\']:\n  for d in [\'Kvasir_Test100\',\'HyperKvasir\',\'ETIS\']:\n   for method in [\'real\',\'LDM\',\'SinGAN-Seg\',\'Polyp-DDPM\']:\n    match=actual[(actual.architecture==a)&(actual.dataset==d)&(actual.method==method)] if not actual.empty else pd.DataFrame()\n    required.append({\'table\':\'II\',\'architecture\':a,\'dataset\':d,\'method\':method,\'status\':\'COMPLETE\' if len(match)==1 else \'MISSING_RESULT\'})\n t1=root/\'table_I_long.csv\';im=pd.read_csv(t1) if t1.exists() else pd.DataFrame()\n for method in [\'LDM\',\'SinGAN-Seg\',\'Polyp-DDPM\',\'Kvasir vs Hyper (reference)\']:\n  for ref in [\'Kvasir_SEG\',\'HyperKvasir\']:\n   match=im[(im.method==method)&(im.reference==ref)&(im.status==\'complete\')&(im.n_fake==1000)&(im.n_real==1000)] if not im.empty else pd.DataFrame()\n   required.append({\'table\':\'I\',\'architecture\':\'\',\'dataset\':ref,\'method\':method,\'status\':\'COMPLETE\' if len(match)==1 else \'MISSING_RESULT\'})\n for a in c[\'architectures\']:\n  if a not in choices:required.append({\'table\':\'selection\',\'architecture\':a,\'dataset\':\'validation\',\'method\':\'all seeds\',\'status\':\'MISSING_RESULT\'})\n  for d in [\'Kvasir_Test100\',\'HyperKvasir\',\'ETIS\']:\n   match=actual[(actual.architecture==a)&(actual.dataset==d)&(actual.method==\'mixed\')] if not actual.empty else pd.DataFrame()\n   required.append({\'table\':\'mixed\',\'architecture\':a,\'dataset\':d,\'method\':\'mixed\',\'status\':\'COMPLETE\' if len(match)==1 else \'MISSING_RESULT\'})\n coverage=pd.DataFrame(required);coverage.to_csv(root/\'coverage.csv\',index=False);write(root/\'completion.json\',{\'complete\':bool((coverage.status==\'COMPLETE\').all()),\'missing\':coverage[coverage.status!=\'COMPLETE\'].to_dict(\'records\')})\n return actual,im,coverage\n\nif __name__==\'__main__\':\n p=argparse.ArgumentParser();p.add_argument(\'--config\',required=True);p.add_argument(\'--phase\',choices=[\'select\',\'paper\',\'quality\'],required=True);p.add_argument(\'--rank\',type=int,default=0);p.add_argument(\'--world\',type=int,default=1);a=p.parse_args();c=json.loads(Path(a.config).read_text())\n sys.path[:0]=[c[\'runtime\'],c[\'metrics_runtime\']]\n if digest(__file__)!=c[\'runner_hash\']:raise ValueError(\'Runner changed; a new study folder is required\')\n device=torch.device(\'cpu\' if c[\'parent\'].get(\'cpu_fixture\') else f\'cuda:{a.rank}\')\n if device.type==\'cuda\':torch.cuda.set_device(device)\n torch.set_num_threads(2)\n from reference_io import install\n install(c)\n try:\n  if a.phase==\'quality\':quality(c,device)\n  else:train_phase(c,a.phase,a.rank,a.world,device)\n except Exception as e:\n  state(c,f\'worker_{a.phase}_{a.rank}\',\'ERROR\',error=repr(e));raise\n', 'bootstrap.py': '"""Resolve read-only master data and independent evaluation checkpoints."""\nimport copy,json,shutil\nfrom pathlib import Path\nfrom runner import digest,write\nimport extension\n\ndef resolve_master(master_root, input_root):\n    if master_root:\n        root=Path(master_root)\n        if not (root/\'run_config.json\').is_file():raise FileNotFoundError(root/\'run_config.json\')\n        return root\n    candidates=[]\n    for path in Path(input_root).rglob(\'run_config.json\'):\n        if (path.parent/\'author/images\').is_dir():candidates.append(path.parent)\n    if len(candidates)!=1:\n        raise ValueError(\'Set MASTER_ROOT to one attached master output. Candidates: \'+str([str(p) for p in candidates]))\n    return candidates[0]\n\ndef load_parent(root, input_root, images=None, masks=None):\n    root=Path(root);parent=json.loads((root/\'run_config.json\').read_text());parent[\'out\']=str(root)\n    ids={r[\'id\'] for r in parent[\'records\']}\n    if not images:\n        matches=[]\n        for path in Path(input_root).rglob(\'images\'):\n            if (path.parent/\'masks\').is_dir() and \'kvasir\' in str(path).lower():\n                found=extension.images(path)\n                if set(found)==ids:matches.append(path)\n        if len(matches)!=1:raise ValueError(\'Set REAL_IMAGES and REAL_MASKS. Candidate real datasets: \'+str(matches))\n        images=str(matches[0]);masks=str(matches[0].parent/\'masks\')\n    im=extension.images(images);ma=extension.images(masks)\n    if set(im)!=ids or set(ma)!=ids:raise ValueError(\'Dataset differs from original master IDs\')\n    for row in parent[\'records\']:\n        row[\'image\']=im[row[\'id\']];row[\'mask\']=ma[row[\'id\']]\n        for kind in [\'image\',\'mask\']:\n            expected=row.get(kind+\'_sha256\')\n            if expected and digest(row[kind])!=expected:raise ValueError(\'Changed original data: \'+row[\'id\']+\' \'+kind)\n    for role,history in parent.get(\'histories\',{}).items():\n        if not history:continue\n        original=Path(history[\'root\'])\n        if original.is_dir():continue\n        matches=[p for p in Path(input_root).rglob(original.name) if p.is_dir()]\n        if len(matches)==1:history[\'root\']=str(matches[0])\n        else:parent[\'histories\'][role]=None # Optional ablation unavailable; core remains runnable.\n    # Do not construct the generator or optimizer in the notebook process.\n    checkpoint=Path(parent[\'author_checkpoint\'])\n    if not checkpoint.is_file():\n        matches=[p for p in Path(input_root).rglob(checkpoint.name) if p.is_file()]\n        matches=[p for p in matches if digest(p)==parent[\'author_sha256\']]\n        parent[\'author_checkpoint\']=str(matches[0]) if len(matches)==1 else \'\'\n    return parent\n\ndef recover_output(out, resume, input_root):\n    out=Path(out)\n    if (out/\'config.json\').exists():return \'local\'\n    if not resume:\n        candidates=[p.parent for p in Path(input_root).rglob(\'config.json\') if p.parent.name==\'polyp_ddpm_evaluation_v8\']\n        if len(candidates)>1:raise ValueError(\'Multiple evaluation outputs. Set RESUME_EVALUATION: \'+str(candidates))\n        resume=str(candidates[0]) if candidates else None\n    if resume:\n        src=Path(resume)\n        previous=json.loads((src/\'config.json\').read_text())\n        if \'preprocessing_mode\' not in previous:raise ValueError(\'Cannot resume v7 training into v8; clear RESUME_EVALUATION\')\n        if not (src/\'config.json\').is_file():raise FileNotFoundError(src/\'config.json\')\n        shutil.copytree(src,out,dirs_exist_ok=True,ignore=shutil.ignore_patterns(\'runtime\',\'__pycache__\'))\n    out.mkdir(parents=True,exist_ok=True)\n    return resume or \'new\'\n\ndef recover_metric_cache(root, out):\n    dest=Path(out)/\'metrics\'\n    for src in [Path(root)/\'evaluation_v5\',Path(root)/\'paper_tables_v6/metrics\']:\n        for name in [\'quality\',\'quality_extra\']:\n            if (src/name).is_dir() and not (dest/name).exists():\n                shutil.copytree(src/name,dest/name)\n\ndef discover_external(specs,input_root,master_root):\n    result=copy.deepcopy(specs)\n    old=Path(master_root)/\'evaluation_v5/study_config.json\'\n    saved=json.loads(old.read_text()).get(\'external\',{}) if old.exists() else {}\n    for name,spec in result.items():\n        if spec.get(\'images\'):continue\n        previous=saved.get(name,{})\n        if previous.get(\'images\') and Path(previous[\'images\']).is_dir():result[name]=previous;continue\n        token=\'hyper\' if name==\'HyperKvasir\' else \'etis\'\n        candidates=[p for p in Path(input_root).rglob(\'images\') if token in str(p).lower() and (p.parent/\'masks\').is_dir()]\n        if len(candidates)==1:\n            spec.update(images=str(candidates[0]),masks=str(candidates[0].parent/\'masks\'))\n    return result\n', 'mixed_addon.py': '"""Add repaired-bank arms without changing the original v7 training identities."""\nimport argparse,copy,json,sys\nfrom pathlib import Path\nimport torch\nimport pandas as pd\nimport runner as base\n\n_samples=base.samples\n_jobs=base.phase_jobs\n\ndef samples(c,method,ids):\n    if method==\'mixed_repaired\':\n        return _samples(c,\'real\',ids)+_samples(c,\'TextureBackground\',ids)\n    return _samples(c,method,ids)\n\ndef phase_jobs(c,phase):\n    jobs=_jobs(c,phase)\n    if phase==\'paper\' and c.get(\'baseline_pairs\',{}).get(\'TextureBackground\'):\n        for arch,seed in base.select_seeds(c).items():\n            jobs.extend({\'phase\':\'paper\',\'arch\':arch,\'seed\':seed,\'method\':method,\'epochs\':c[\'paper_epochs\']}\n                        for method in [\'TextureBackground\',\'mixed_repaired\'])\n    return jobs\n\nbase.samples=samples\nbase.phase_jobs=phase_jobs\n\ndef prepare(c):\n    c=base.prepare(c)\n    rows=base.read_pairs({\'pairs_csv\':c.get(\'repaired_pairs_csv\')},c[\'parent\'][\'train_ids\'])\n    if rows:\n        # Repaired images originate from precisely the current training masks.\n        if {r[\'id\'] for r in rows}!=set(c[\'parent\'][\'train_ids\']):\n            raise ValueError(\'Repaired sample_id must match the current train900 IDs\')\n        native={r[\'id\']:r for r in c[\'parent\'][\'records\']}\n        for row in rows:\n            if base.digest(row[\'mask\'])!=native[row[\'id\']][\'mask_sha256\']:\n                raise ValueError(\'Repaired mask differs from original condition mask: \'+row[\'id\'])\n        tests={r[\'image_sha256\'] for r in c[\'parent\'][\'records\'] if r[\'id\'] in c[\'parent\'][\'test_ids\']}\n        if any(base.digest(r[\'image\']) in tests for r in rows):raise ValueError(\'Test image in repaired bank\')\n    c[\'baseline_pairs\'][\'TextureBackground\']=rows\n    if rows:\n        bank=[{\'id\':r[\'id\'],\'image\':base.digest(r[\'image\']),\'mask\':base.digest(r[\'mask\'])} for r in rows]\n        c[\'method_hash\'][\'TextureBackground\']=base.ident({\'bank\':bank,\'adapter\':base.digest(__file__)})\n        c[\'method_hash\'][\'mixed_repaired\']=base.ident([c[\'method_hash\'][\'real\'],c[\'method_hash\'][\'TextureBackground\']])\n    return c\n\n# Expose unchanged base helpers to notebook cells.\nwrite=base.write\ntables=base.tables\nselect_seeds=base.select_seeds\nkey=base.key\n\ndef quality(c,device):\n    base.quality(c,device)\n    rows=c.get(\'baseline_pairs\',{}).get(\'TextureBackground\',[])\n    if not rows:return\n    import extension\n    qc=copy.deepcopy(c);qc[\'out\']=c[\'metric_cache_out\'];qc[\'metric_n\']=len(rows)\n    qc[\'code_sha256\']=c[\'metrics_frozen_hash\']\n    native={r[\'id\']:r for r in c[\'parent\'][\'records\']}\n    value=extension.metrics_pair(qc,\'repaired_vs_matched_real\',\n        [r[\'image\'] for r in rows],[native[r[\'id\']][\'image\'] for r in rows],device)\n    path=Path(c[\'out\'])/\'image_ablation.csv\'\n    old=pd.read_csv(path)\n    row={\'source\':\'texture_background\',\'reference\':\'matched real train IDs\',\'available\':len(rows),\n         **{k:v for k,v in value.items() if k!=\'spec\'}}\n    pd.concat([old,pd.DataFrame([row])],ignore_index=True).to_csv(path,index=False)\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--config\',required=True)\n    parser.add_argument(\'--phase\',choices=[\'select\',\'paper\',\'quality\'],required=True)\n    parser.add_argument(\'--rank\',type=int,default=0)\n    parser.add_argument(\'--world\',type=int,default=1)\n    a=parser.parse_args();c=json.loads(Path(a.config).read_text())\n    if base.digest(base.__file__)!=c[\'runner_hash\']:raise ValueError(\'Base runner changed\')\n    device=torch.device(\'cpu\' if c[\'parent\'].get(\'cpu_fixture\') else f\'cuda:{a.rank}\')\n    if device.type==\'cuda\':torch.cuda.set_device(device)\n    torch.set_num_threads(2)\n    try:\n        if a.phase==\'quality\':quality(c,device)\n        else:base.train_phase(c,a.phase,a.rank,a.world,device)\n    except Exception as exc:\n        base.state(c,f\'worker_{a.phase}_{a.rank}\',\'ERROR\',error=repr(exc));raise\n', 'direct_protocol.py': '"""Direct100epoch protocol. No20epoch selection; original v7 jobs remain reusable."""\nimport argparse,json\nfrom pathlib import Path\nimport torch\nimport mixed_addon as addon\nbase=addon.base\n\ndef fixed_seeds(c):\n    seeds=c[\'seeds\']\n    if len(seeds)!=1:raise ValueError(\'Direct protocol requires one declared seed; use separate studies for repeats\')\n    return {arch:int(seeds[0]) for arch in c[\'architectures\']}\n\ndef phase_jobs(c,phase):\n    if phase==\'select\':return []\n    if phase!=\'paper\':return []\n    jobs=[]\n    for arch,seed in fixed_seeds(c).items():\n        methods=[\'real\',\'Polyp-DDPM\',\'mixed\']\n        methods += [m for m in [\'LDM\',\'SinGAN-Seg\'] if c[\'baseline_pairs\'].get(m)]\n        if c[\'baseline_pairs\'].get(\'TextureBackground\'):methods+=[\'TextureBackground\',\'mixed_repaired\']\n        jobs.extend({\'phase\':\'paper\',\'arch\':arch,\'seed\':seed,\'method\':m,\'epochs\':c[\'paper_epochs\']} for m in methods)\n    return jobs\n\nbase.select_seeds=fixed_seeds\nbase.phase_jobs=phase_jobs\nprepare=addon.prepare\nwrite=base.write\ntables=base.tables\nselect_seeds=fixed_seeds\nkey=base.key\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--config\',required=True)\n    p.add_argument(\'--phase\',choices=[\'paper\',\'quality\'],required=True)\n    p.add_argument(\'--rank\',type=int,default=0);p.add_argument(\'--world\',type=int,default=1)\n    a=p.parse_args();c=json.loads(Path(a.config).read_text())\n    if base.digest(base.__file__)!=c[\'runner_hash\']:raise ValueError(\'Base runner changed\')\n    if c[\'paper_epochs\']!=100 and not c[\'parent\'].get(\'cpu_fixture\'):raise ValueError(\'Paper specifies100epochs\')\n    device=torch.device(\'cpu\' if c[\'parent\'].get(\'cpu_fixture\') else f\'cuda:{a.rank}\')\n    if device.type==\'cuda\':torch.cuda.set_device(device)\n    torch.set_num_threads(2)\n    from reference_io import install\n    install(c)\n    try:\n        if a.phase==\'quality\':addon.quality(c,device)\n        else:base.train_phase(c,\'paper\',a.rank,a.world,device)\n    except Exception as exc:\n        base.state(c,f\'worker_{a.phase}_{a.rank}\',\'ERROR\',error=repr(exc));raise\n', 'reference_io.py': '"""Input/metric adapters derived from reference[9]; source snapshots embedded."""\nimport random\nimport numpy as np\nimport torch\nfrom PIL import Image\nimport albumentations as albu\nimport common\n\n_train=None\n_valid=None\nMODE=\'reference_code\'\ndef configure(c):\n    global _train,_valid,MODE\n    from reference_augmentation import get_training_augmentation,get_validation_augmentation\n    _train=get_training_augmentation();_valid=get_validation_augmentation()\n    MODE=c.get(\'preprocessing_mode\',\'reference_code\')\n\ndef tensor_pair(record,size,seed=None,synthetic=None):\n    with Image.open(synthetic or record[\'image\']) as handle:image=np.array(handle.convert(\'RGB\'),copy=True)\n    with Image.open(record[\'mask\']) as handle:mask=(np.array(handle.convert(\'RGB\'))[:,:,0]>128).astype(float)\n    py=random.getstate();nr=np.random.get_state()\n    try:\n        if seed is not None:random.seed(seed);np.random.seed(seed%(2**32))\n        result=(_train if seed is not None else _valid)(image=image,mask=mask)\n    finally:random.setstate(py);np.random.set_state(nr)\n    x=torch.from_numpy(result[\'image\'].copy().transpose(2,0,1)).float()/255\n    y=torch.from_numpy(result[\'mask\'].copy()).float()[None]\n    if MODE==\'reference_code\':x=common.normalize_segmenter(x[None])[0]\n    return x,y\n\ndef install(c):\n    configure(c)\n    common.tensor_pair=tensor_pair\n    original=common.binary_metrics\n    def metrics(prob,mask):\n        # Reference uses >0.5, whereas old notebook used >=0.5.\n        return original((prob>0.5).astype(np.float32),mask)\n    common.binary_metrics=metrics\n    if not c[\'parent\'].get(\'cpu_fixture\'):\n        import segmentation_models_pytorch as smp\n        common.dice_loss=smp.losses.DiceLoss(smp.losses.BINARY_MODE,from_logits=True)\n', 'reference_augmentation.py': 'import albumentations as albu\n\ndef get_training_augmentation():\n    train_transform = [\n\n        albu.HorizontalFlip(p=0.5),\n\n        albu.ShiftScaleRotate(scale_limit=0.5, rotate_limit=0, shift_limit=0.1, p=1, border_mode=0),\n\n        albu.PadIfNeeded(min_height=256, min_width=256, always_apply=True, border_mode=0),\n        albu.Resize(height=256, width=256, always_apply=True),\n\n       # albu.IAAAdditiveGaussianNoise(p=0.2),\n        albu.GaussNoise(p=0.2),\n        albu.Perspective(p=0.5),\n\n        albu.OneOf(\n            [\n                albu.CLAHE(p=1),\n                albu.RandomBrightnessContrast(p=1),\n                albu.RandomGamma(p=1),\n            ],\n            p=0.9,\n        ),\n\n        albu.OneOf(\n            [\n                albu.Sharpen(p=1),\n                albu.Blur(blur_limit=3, p=1),\n                albu.MotionBlur(blur_limit=3, p=1),\n            ],\n            p=0.9,\n        ),\n\n        albu.OneOf(\n            [\n                albu.RandomContrast(p=1),\n                albu.HueSaturationValue(p=1),\n            ],\n            p=0.9,\n        ),\n    ]\n    return albu.Compose(train_transform)\n\ndef get_validation_augmentation():\n    """Add paddings to make image shape divisible by 32"""\n    test_transform = [\n        albu.PadIfNeeded(min_height=256, min_width=256, always_apply=True, border_mode=0),\n        albu.Resize(height=256, width=256, always_apply=True),\n    ]\n    return albu.Compose(test_transform)\n', 'reference_snapshot/train.py': '#=========================================================\n# Developer: Vajira Thambawita (PhD)\n# References: * https://github.com/qubvel/segmentation_models.pytorch/blob/master/examples/binary_segmentation_intro.ipynb\n#             * https://pytorch-lightning.readthedocs.io/en/stable/\n#=========================================================\n\nimport argparse\nfrom omegaconf import OmegaConf\nfrom datetime import datetime\nimport os\nimport copy\nfrom tqdm import tqdm\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom PIL import Image\n\n#Pytorch\nimport torch\nimport torch.optim as optim\nfrom torch.optim import lr_scheduler\nimport torch.nn as nn\nfrom torch.utils.data import DataLoader\nfrom torchvision import models, transforms,datasets, utils\n\nfrom torch.autograd import Variable\n\n\nimport segmentation_models_pytorch as smp\nimport pytorch_lightning as pl\nfrom pytorch_lightning.cli import LightningCLI\n#from einops import rearrange\nimport pandas as pd\n\nfrom data.prepare_data import PolypDataModule # custom data module\nimport segmentation_models_pytorch as smp\nimport wandb\nfrom pytorch_lightning.loggers import WandbLogger\n\n\n\n\n# Pytorch lightning training\nclass PolypModel(pl.LightningModule):\n\n    def __init__(self, \n                 arch="UnetPlusPlus", \n                 encoder_name= "resnet34", \n                 in_channels=3, \n                 out_classes=1, \n                 lr=0.0001, \n                 test_print_batch_id=0, \n                 test_print_num=5, \n                 output_dir="",\n                 wandb_name="",\n                 **kwargs):\n        super().__init__()\n        \n        \n        \n        # "UnetPlusPlus", "resnet34", in_channels=3, out_classes=2\n        self.arch = arch\n        self.encoder_name = encoder_name\n        self.in_channels = in_channels\n        self.out_classes = out_classes\n        self.test_print_batch_id = test_print_batch_id\n        self.test_print_num = test_print_num\n        self.lr = lr\n        self.wandb_name = "test"\n        self.output_dir = output_dir\n        self.wandb_name = wandb_name\n        \n        \n        \n        print(self.output_dir)\n        \n        #exit(0)\n        os.makedirs(self.output_dir, exist_ok=True)\n\n        self.model = smp.create_model(\n            self.arch, encoder_name=self.encoder_name, in_channels=self.in_channels, classes=self.out_classes, **kwargs)\n\n        # preprocessing parameteres for image\n        params = smp.encoders.get_preprocessing_params(encoder_name)\n        self.register_buffer("std", torch.tensor(params["std"]).view(1, 3, 1, 1)) # self.std\n        self.register_buffer("mean", torch.tensor(params["mean"]).view(1, 3, 1, 1)) # self.mean\n\n        # for image segmentation dice loss could be the best first choice\n        self.loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)\n\n    def forward(self, image):\n        # normalize image here\n        image = (image - self.mean) / self.std\n        mask = self.model(image)\n        return mask\n\n    def shared_step(self, batch, stage):\n        \n        image = batch["image"]\n\n        # Shape of the image should be (batch_size, num_channels, height, width)\n        # if you work with grayscale images, expand channels dim to have [batch_size, 1, height, width]\n        assert image.ndim == 4\n\n        # Check that image dimensions are divisible by 32, \n        # encoder and decoder connected by `skip connections` and usually encoder have 5 stages of \n        # downsampling by factor 2 (2 ^ 5 = 32); e.g. if we have image with shape 65x65 we will have \n        # following shapes of features in encoder and decoder: 84, 42, 21, 10, 5 -> 5, 10, 20, 40, 80\n        # and we will get an error trying to concat these features\n        h, w = image.shape[2:]\n        #print(f"h={h}, w={w}")\n        assert h % 32 == 0 and w % 32 == 0\n\n        mask = batch["mask"]\n\n        # Shape of the mask should be [batch_size, num_classes, height, width]\n        # for binary segmentation num_classes = 1\n        assert mask.ndim == 4\n\n        # Check that mask values in between 0 and 1, NOT 0 and 255 for binary segmentation\n        assert mask.max() <= 1.0 and mask.min() >= 0\n\n        logits_mask = self.forward(image)\n        \n        # Predicted mask contains logits, and loss_fn param `from_logits` is set to True\n        loss = self.loss_fn(logits_mask, mask)\n\n        # Lets compute metrics for some threshold\n        # first convert mask values to probabilities, then \n        # apply thresholding\n        prob_mask = logits_mask.sigmoid()\n        pred_mask = (prob_mask > 0.5).float()\n\n        # We will compute IoU metric by two ways\n        #   1. dataset-wise\n        #   2. image-wise\n        # but for now we just compute true positive, false positive, false negative and\n        # true negative \'pixels\' for each image and class\n        # these values will be aggregated in the end of an epoch\n        tp, fp, fn, tn = smp.metrics.get_stats(pred_mask.long(), mask.long(), mode="binary")\n\n        return {\n            "loss": loss,\n            "tp": tp,\n            "fp": fp,\n            "fn": fn,\n            "tn": tn,\n        }\n\n    def shared_epoch_end(self, outputs, stage):\n        # aggregate step metics\n        tp = torch.cat([x["tp"] for x in outputs])\n        fp = torch.cat([x["fp"] for x in outputs])\n        fn = torch.cat([x["fn"] for x in outputs])\n        tn = torch.cat([x["tn"] for x in outputs])\n\n        # per image IoU means that we first calculate IoU score for each image \n        # and then compute mean over these scores\n        per_image_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro-imagewise")\n        \n        # dataset IoU means that we aggregate intersection and union over whole dataset\n        # and then compute IoU score. The difference between dataset_iou and per_image_iou scores\n        # in this particular case will not be much, however for dataset \n        # with "empty" images (images without target class) a large gap could be observed. \n        # Empty images influence a lot on per_image_iou and much less on dataset_iou.\n        dataset_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")\n\n        metrics = {\n            f"{stage}_per_image_iou": per_image_iou,\n            f"{stage}_dataset_iou": dataset_iou,\n        }\n        \n        #if stage == "valid":\n        #    print(metrics)\n        \n        self.log_dict(metrics, prog_bar=True)\n\n    def training_step(self, batch, batch_idx):\n        return self.shared_step(batch, "train")            \n\n    def training_epoch_end(self, outputs):\n        return self.shared_epoch_end(outputs, "train")\n\n    def validation_step(self, batch, batch_idx):\n        return self.shared_step(batch, "valid")\n\n    def validation_epoch_end(self, outputs):\n        return self.shared_epoch_end(outputs, "valid")\n\n    def test_step(self, batch, batch_idx):\n        image = batch["image"]\n        #print(image.shape)\n        image_before = image\n        #print(image_before.shape)\n        image_origin = batch["image_origin"]\n        mask = batch["mask"]\n        \n        image = (image - self.mean) / self.std\n        mask_p = self.model(image)\n        \n        prob_mask = mask_p.sigmoid()\n        pred_mask = (prob_mask > 0.5).float()\n        \n        loss = self.loss_fn(mask_p, mask)\n        #print(mask_p.shape)\n        tp, fp, fn, tn = smp.metrics.get_stats(pred_mask.long(), mask.long(), mode="binary")\n        \n        return {"image_origin": image_origin, \n                "image_before":image_before,\n                "mask": mask, \n                "prob_mask": prob_mask, \n                "pred_mask": pred_mask,\n                "loss": loss,\n                "tp": tp,\n                "fp": fp,\n                "fn": fn,\n                "tn": tn,}  \n\n    def test_epoch_end(self, outputs):\n        \n        # aggregate step metics\n        tp = torch.cat([x["tp"] for x in outputs])\n        fp = torch.cat([x["fp"] for x in outputs])\n        fn = torch.cat([x["fn"] for x in outputs])\n        tn = torch.cat([x["tn"] for x in outputs])\n        \n        per_image_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro-imagewise")\n        dataset_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")\n        \n        per_image_f1 = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro-imagewise")\n        dataset_f1 = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")\n        \n        per_image_accuray = smp.metrics.accuracy(tp, fp, fn, tn, reduction="micro-imagewise")\n        dataset_accuracy = smp.metrics.accuracy(tp, fp, fn, tn, reduction="micro")\n        \n        per_image_precision = smp.metrics.precision(tp, fp, fn, tn, reduction="micro-imagewise")\n        dataset_precision = smp.metrics.precision(tp, fp, fn, tn, reduction="micro")\n        \n        metrics = {\n            "per_image_iou": per_image_iou,\n            "per_image_f1": per_image_f1,\n            "per_image_accuray": per_image_accuray,\n            "per_image_precision": per_image_precision,\n            "dataset_iou": dataset_iou,\n            "dataset_f1": dataset_f1,\n            "dataset_accuracy": dataset_accuracy,\n            "dataset_precision": dataset_precision \n        }\n        \n        self.log_dict(metrics, prog_bar=True)\n        \n        with open(f"{self.output_dir}/metrics.txt", "w") as f:\n          \n            for key, value in metrics.items():\n                f.write(f"{key}\\t={value}")\n                f.write("\\n")\n\n        #print("test config=", self.ckpt_path)\n        \n        for i in range(self.test_print_num):\n            img = outputs[0]["image_origin"][i].cpu().numpy()\n            #img_before = rearrange(outputs[0]["image_before"][i], \'c h w -> h w c\').cpu().numpy()\n            #print("img before max=", img_before.max())\n            #print("img before min=", img_before.min())\n            mask = outputs[0]["pred_mask"][i][0,:, :].cpu().numpy()\n            mask_gt = outputs[0]["mask"][i][0,:, :].cpu().numpy()\n            plt.imsave(f"{self.output_dir}/image_{i}.png",img) \n            plt.imsave(f"{self.output_dir}/mask_pred_{i}.png", mask) \n            plt.imsave(f"{self.output_dir}/mask_gt_{i}.png", mask_gt) \n            #plt.imsave(f"{self.output_dir}/image_before_{i}.png", img_before) \n            \n        \n    \n    def predict_step(self, batch, batch_idx):\n        image = batch["image"]\n        mask = batch["mask"]\n        image = (image - self.mean) / self.std\n        mask_p = self.model(image)\n        prob_mask = mask_p.sigmoid()\n        pred_mask = (prob_mask > 0.5).float()\n        print(mask_p.shape)\n        \n        return {"image": image, "mask": mask, "prob_mask": prob_mask, "pred_mask": pred_mask}\n    \n\n    def configure_optimizers(self):\n        return torch.optim.Adam(self.parameters(), lr=self.lr)\n\n\n\n    \nclass MyLightningCLI(LightningCLI):\n    def add_arguments_to_parser(self, parser):\n        parser.add_argument("--wandb_name", default="unet_plus_plus_1")\n        parser.add_argument("--wandb_entity", default="simulamet_mlc")\n        parser.add_argument("--wandb_project", default="diffusion_polyp")\n        parser.add_argument("--output_dir", default="output/new_3")\n        parser.link_arguments("output_dir", "model.output_dir")\n        parser.link_arguments("wandb_name", "model.wandb_name")\n        \n    #def add_default_arguments_to_parser(self, parser):\n        \n        \n        \n    def instantiate_classes(self):\n        #print(self.config[self.config.subcommand])\n        \n        # Call to the logger before initiate other clases, because Trainer class init logger if we didn´t do it\n        logger = WandbLogger(entity=self.config[self.config.subcommand].wandb_entity, \n                             project=self.config[self.config.subcommand].wandb_project,\n                             name=self.config[self.config.subcommand].wandb_name)\n        super().instantiate_classes() # call to super class instatiate_classes()\n      \n        \n\n# Implementing a CLI\ndef cli_main():\n  \n    \n    cli = MyLightningCLI(PolypModel, PolypDataModule, \n                       save_config_kwargs={"config_filename": "test_config.yaml", \'overwrite\':True}\n                       )\n    \n\n\nif __name__ == "__main__":\n\n    cli_main() # Pytorch lightning command line interface \n    wandb.finish() # Finish Wandb\n\n   \n\n', 'reference_snapshot/data/prepare_data.py': '\nimport pandas as pd\nimport albumentations as albu\nfrom torch.utils.data import DataLoader, Subset, ConcatDataset\nfrom omegaconf import OmegaConf\nimport segmentation_models_pytorch as smp\n\nfrom data.dataset import Dataset\nimport pytorch_lightning as pl\n\ndef get_training_augmentation():\n    train_transform = [\n\n        albu.HorizontalFlip(p=0.5),\n\n        albu.ShiftScaleRotate(scale_limit=0.5, rotate_limit=0, shift_limit=0.1, p=1, border_mode=0),\n\n        albu.PadIfNeeded(min_height=256, min_width=256, always_apply=True, border_mode=0),\n        albu.Resize(height=256, width=256, always_apply=True),\n\n       # albu.IAAAdditiveGaussianNoise(p=0.2),\n        albu.GaussNoise(p=0.2),\n        albu.Perspective(p=0.5),\n\n        albu.OneOf(\n            [\n                albu.CLAHE(p=1),\n                albu.RandomBrightnessContrast(p=1),\n                albu.RandomGamma(p=1),\n            ],\n            p=0.9,\n        ),\n\n        albu.OneOf(\n            [\n                albu.Sharpen(p=1),\n                albu.Blur(blur_limit=3, p=1),\n                albu.MotionBlur(blur_limit=3, p=1),\n            ],\n            p=0.9,\n        ),\n\n        albu.OneOf(\n            [\n                albu.RandomContrast(p=1),\n                albu.HueSaturationValue(p=1),\n            ],\n            p=0.9,\n        ),\n    ]\n    return albu.Compose(train_transform)\n\n\n\ndef get_validation_augmentation():\n    """Add paddings to make image shape divisible by 32"""\n    test_transform = [\n        albu.PadIfNeeded(min_height=256, min_width=256, always_apply=True, border_mode=0),\n        albu.Resize(height=256, width=256, always_apply=True),\n    ]\n    return albu.Compose(test_transform)\n\n\n\ndef to_tensor(x, **kwargs):\n    return x.transpose(2, 0, 1).astype(\'float32\')\n\ndef to_float(x, **kwargs):\n    return x.astype(\'float32\')\n\n\n\ndef get_preprocessing(preprocessing_fn):\n    """Construct preprocessing transform\n    \n    Args:\n        preprocessing_fn (callbale): data normalization function \n            (can be specific for each pretrained neural network)\n    Return:\n        transform: albumentations.Compose\n    \n    """\n    \n    _transform = []\n    if preprocessing_fn:\n        _transform.append(albu.Lambda(image=preprocessing_fn))\n    _transform.append(albu.Lambda(image=to_tensor, mask=to_tensor))\n\n    \n    return albu.Compose(_transform)\n\n\n\n\n# Pytorch lightning DataModule\n\nclass PolypDataModule(pl.LightningDataModule):\n    \n    def __init__(self,  encoder = "se_resnext50_32x4d", \n                 encoder_weight = "imagenet", bs=8, num_workers=2, \n                 data = {\n                     "train":[{"img_dir":"", "mask_dir": "", "num_samples": ""},],\n                     "validation":[{"img_dir":"", "mask_dir": "", "num_samples": ""},]\n                 }\n                ):\n        \n        super().__init__()\n        self.conf = OmegaConf.create(data) # create an OmegaConf from input data dictionary\n        self.encoder = encoder\n        self.encoder_weight = encoder_weight\n        self.preprocessing_fn = smp.encoders.get_preprocessing_fn(self.encoder, self.encoder_weight)\n        self.bs = bs\n        self.num_workers = num_workers\n        \n        #print(self.conf)\n        \n        \n    def setup(self, stage: str):\n        # Train data\n        datasets = []\n        #print(self.conf.train)\n        for d in self.conf.train:\n            data_sub = Dataset( d.img_dir, d.mask_dir, \n                               augmentation=get_training_augmentation(), \n                               preprocessing=get_preprocessing(self.preprocessing_fn))\n            if d.num_samples >= 0:\n                data_sub = Subset(data_sub, [i for i in range(d.num_samples)])\n            datasets.append(data_sub)\n        self.train_dataset = ConcatDataset(datasets)\n        print("train_dataset_size=", len(self.train_dataset))\n\n        # Valiation data\n        datasets = []\n        #print(self.conf.validation)\n        for d in self.conf.validation:\n            data_sub = Dataset(d.img_dir,d.mask_dir,\n                               augmentation=get_validation_augmentation(), \n                               preprocessing=get_preprocessing(self.preprocessing_fn))\n            if d.num_samples >= 0:\n                data_sub = Subset(data_sub, [i for i in range(d.num_samples)])\n            datasets.append(data_sub)\n\n        self.validation_dataset = ConcatDataset(datasets)\n        print("validation dataset size=", len(self.validation_dataset))\n        \n    def train_dataloader(self):\n        return DataLoader(self.train_dataset, batch_size=self.bs, shuffle=True, num_workers= self.num_workers)\n    \n    def val_dataloader(self):\n        return DataLoader(self.validation_dataset, batch_size=self.bs, shuffle=False, num_workers= self.num_workers)\n    \n    def test_dataloader(self):\n        return DataLoader(self.validation_dataset, batch_size=self.bs, shuffle=False, num_workers= self.num_workers)\n    \n    def predict_dataloader(self):\n        return DataLoader(self.validation_dataset, batch_size=self.bs, shuffle=False, num_workers= self.num_workers)\n    \n    \n        \n', 'reference_snapshot/data/dataset.py': 'from torch.utils.data import Dataset as BaseDataset\nimport numpy as np\nimport glob\nimport os\nfrom PIL import Image\nimport matplotlib.pyplot as plt\n\n\nclass Dataset(BaseDataset):\n    """CamVid Dataset. Read images, apply augmentation and preprocessing transformations.\n    \n    Args:\n        images_dir (str): path to images folder\n        masks_dir (str): path to segmentation masks folder\n        class_values (list): values of classes to extract from segmentation mask\n        augmentation (albumentations.Compose): data transfromation pipeline \n            (e.g. flip, scale, etc.)\n        preprocessing (albumentations.Compose): data preprocessing \n            (e.g. noralization, shape manipulation, etc.)\n    \n    """\n    \n    \n    def __init__(\n            self, \n            img_dir,\n            mask_dir,\n            img_exts = ["jpg", "png"],\n            mask_exts = ["jpg", "png"],\n            classes=[0,255], \n            augmentation=None, \n            preprocessing=None,\n            mask_clean=True\n    ):\n       \n        self.img_dir = img_dir\n        self.mask_dir  = mask_dir\n        \n        self.img_exts = img_exts\n        self.mask_exts = mask_exts\n        \n        # Get all image and mask paths using given extensions\n        self.images = []\n        self.masks = []\n        \n        for img_ext in self.img_exts:\n            self.images.extend(glob.glob(img_dir + f"/*.{img_ext}")) \n            \n        for mask_ext in self.mask_exts:\n            self.masks.extend(glob.glob(mask_dir + f"/*.{mask_ext}")) \n        \n        # convert str names to class values on masks\n        self.class_values = classes\n        \n        self.mask_clean = mask_clean\n        self.augmentation = augmentation\n        self.preprocessing = preprocessing\n    \n    def __getitem__(self, i):\n        \n        \n        # Getting images and masks without any mismatch\n        img_path = self.images[i]\n        img_name = img_path.split("/")[-1] #.split(".")[0]\n        mask_path = os.path.join(self.mask_dir, img_name)\n        img = Image.open(img_path)\n        image_t = np.array(img) # to print, but without preprocessing\n        image = np.array(img)\n        \n        mask = np.array(Image.open(mask_path))[:,:, 0] # take only one channel from 3-channels mask image\n        \n        #print(mask.min())\n        #print(mask.max())\n        # Mask clearning (to remove some artifacts coming from labeling)\n        if self.mask_clean:\n            mask = (mask > 128) # * 255 # if mask value > 128, set it to 255 (this will remove 254, 253 values and 1,2 etc)\n        mask = mask.astype("float")\n        mask = np.expand_dims(mask, axis=2)\n        # extract certain classes from mask (e.g. cars)\n        #masks = [(mask == v) for v in self.class_values]\n       # mask = np.stack(masks, axis=-1).astype(\'float\')\n\n        \n        # apply augmentations\n        if self.augmentation:\n            sample = self.augmentation(image=image, mask=mask)\n            ori_img_smpl = self.augmentation(image=image_t, mask=mask) # here mask is dummy. not required. but need to pass to work\n            image, mask = sample[\'image\'], sample[\'mask\']\n            image_t = ori_img_smpl["image"]\n            \n\n        # apply preprocessing\n        if self.preprocessing:\n            sample = self.preprocessing(image=image, mask=mask)\n            image, mask = sample[\'image\'], sample[\'mask\']\n            \n  \n            \n        return {"image":image, "mask":mask, "image_origin": image_t}\n        \n    def __len__(self):\n        return len(self.images)\n    \n# This section is just for testing.\nif __name__=="__main__":\n    test_dataset = Dataset(img_dir="/work/vajira/DL/roman_diffusion_model/Models/results_135/samples",\n                              mask_dir = "/work/vajira/DL/roman_diffusion_model/Models/200")\n    print(len(test_dataset))\n    #img, mask = \n    sample = test_dataset[500]\n    print(sample["image"].shape)\n    print(sample["mask"].shape)\n    print(sample["image_origin"].shape)\n    print(sample["mask"].max())\n    print(sample["mask"].min())\n    plt.imsave("mask.jpeg", sample["mask"][:,:,0])\n    plt.imsave("img.jpeg", sample["image_origin"])', 'reference_snapshot/config_RALLF000.yaml': '# pytorch_lightning==1.9.2\nseed_everything: true\ntrainer:\n  logger: pytorch_lightning.loggers.WandbLogger\n  enable_checkpointing: true\n  callbacks: \n      - class_path: pytorch_lightning.callbacks.ModelCheckpoint\n        init_args:\n            save_top_k: 1\n            monitor: "valid_dataset_iou"\n            mode: "max"\n            dirpath: "output/05_config"\n            filename: "best"\n  default_root_dir: null\n  gradient_clip_val: null\n  gradient_clip_algorithm: null\n  num_nodes: 1\n  num_processes: null\n  devices: [1]\n  gpus: null\n  auto_select_gpus: null\n  tpu_cores: null\n  ipus: null\n  enable_progress_bar: true\n  overfit_batches: 0.0\n  track_grad_norm: -1\n  check_val_every_n_epoch: 1\n  fast_dev_run: false\n  accumulate_grad_batches: null\n  max_epochs: 50\n  min_epochs: null\n  max_steps: -1\n  min_steps: null\n  max_time: null\n  limit_train_batches: null\n  limit_val_batches: null\n  limit_test_batches: null\n  limit_predict_batches: null\n  val_check_interval: null\n  log_every_n_steps: 50\n  accelerator: "gpu"\n  strategy: null\n  sync_batchnorm: false\n  precision: 32\n  enable_model_summary: true\n  num_sanity_val_steps: 2\n  resume_from_checkpoint: null\n  profiler: null\n  benchmark: null\n  deterministic: null\n  reload_dataloaders_every_n_epochs: 0\n  auto_lr_find: false\n  replace_sampler_ddp: true\n  detect_anomaly: false\n  auto_scale_batch_size: false\n  plugins: null\n  amp_backend: null\n  amp_level: null\n  move_metrics_to_cpu: false\n  multiple_trainloader_mode: max_size_cycle\n  inference_mode: true\nmodel:\n  arch: UnetPlusPlus\n  encoder_name: resnet34\n  in_channels: 3\n  out_classes: 1\n  lr: 0.0001\n  test_print_batch_id: 0\n  test_print_num: 5\n  encoder_weights: imagenet\ndata:\n  encoder: se_resnext50_32x4d\n  encoder_weight: imagenet\n  bs: 8\n  num_workers: 4\n  data:\n    train:\n    - img_dir: "/work/vajira/DL/roman_diffusion_model/conditional-polyp-diffusion/Kvasir-SEG/train/images"\n      mask_dir: "/work/vajira/DL/roman_diffusion_model/conditional-polyp-diffusion/Kvasir-SEG/train/masks"\n      num_samples: -1\n   \n    - img_dir: "/work/vajira/DL/roman_diffusion_model/Models/results_135/samples" # fake samples\n      mask_dir: "/work/vajira/DL/roman_diffusion_model/Models/200" # input masks are here\n      num_samples: 0\n    validation:\n    - img_dir: "/work/vajira/DL/roman_diffusion_model/conditional-polyp-diffusion/Kvasir-SEG/test/images"\n      mask_dir: "/work/vajira/DL/roman_diffusion_model/conditional-polyp-diffusion/Kvasir-SEG/test/masks"\n      num_samples: -1\n    - img_dir: "/work/vajira/DL/roman_diffusion_model/conditional-polyp-diffusion/Kvasir-SEG/val/images"\n      mask_dir: "/work/vajira/DL/roman_diffusion_model/conditional-polyp-diffusion/Kvasir-SEG/val/masks"\n      num_samples: -1\nwandb_name: 05_config\nwandb_entity: simulamet_mlc\nwandb_project: diffusion_polyp\noutput_dir: output/05_config\nckpt_path: null\n', 'reference_snapshot/requirements.txt': 'albumentations==1.3.0\nmatplotlib==3.5.2\nnumpy==1.21.5\nomegaconf==2.3.0\npandas==1.4.4\nPillow==9.4.0\npytorch_lightning==1.9.4\nsegmentation_models_pytorch==0.3.2\ntorch==1.13.1\ntorchvision==0.14.1\ntqdm==4.64.1\nwandb==0.13.11\n'}
for name, source in SOURCES.items():
    path=RUNTIME/name
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(source)
sys.path.insert(0, str(RUNTIME))
import direct_protocol as study
import bootstrap
from memory_guard import snapshot
del SOURCES


## 2. Input và resume v8
Không resume training v7 vào v8 vì preprocessing/augmentation/precision/checkpoint đã đổi. Output master vẫn dùng làm nguồn bank và split. RESUME_EVALUATION chỉ nhận outputv8. Các kết quả cũ vẫn giữ làm đối chứng, không đổi nhãn thành kết quả mới.


In [ ]:
resume_from=bootstrap.recover_output(OUTPUT,RESUME_EVALUATION,INPUT_ROOT)
master=bootstrap.resolve_master(MASTER_ROOT,INPUT_ROOT)
parent=bootstrap.load_parent(master,INPUT_ROOT,REAL_IMAGES,REAL_MASKS)
parent["seg_batch"]=BATCH_SIZE
assert BATCH_SIZE==8
AMP=False
parent["eval_batch"]=min(BATCH_SIZE,4)
external=bootstrap.discover_external(EXTERNAL,INPUT_ROOT,master)
bootstrap.recover_metric_cache(master,OUTPUT)

PAPER={
    "parent":parent, "out":OUTPUT, "runtime":str(RUNTIME), "metrics_runtime":str(RUNTIME),
    "metric_cache_out":str(Path(OUTPUT)/"metrics"), "external":external, "baselines":BASELINES,
    "architectures":ARCHITECTURES, "seeds":SEEDS, "selection_epochs":0,
    "paper_epochs":TRAIN_EPOCHS, "validation_seed":20261120, "validation_n":90,
    "metric_seed":20261120, "metric_n":1000, "amp":AMP, "memory_reserve_gib":1.5,
    "generate_100_extra":False,
}
PAPER["repaired_pairs_csv"]=REPAIRED_PAIRS_CSV
PAPER["repaired_label"]=REPAIRED_LABEL
PAPER["preprocessing_mode"]=PREPROCESSING_MODE
PAPER["checkpoint_policy"]=CHECKPOINT_POLICY
PAPER["software"]={n:importlib.metadata.version(n) for n in ["torch","segmentation_models_pytorch","albumentations","numpy"]}
PAPER=study.prepare(PAPER)
# Chỉ sinh thêm 100 ảnh metric nếu chưa có đủ và đã tìm được checkpoint đúng hash.
extra=list((Path(OUTPUT)/"metrics/quality_extra").glob("*.png"))
PAPER["generate_100_extra"] = len(extra)<100 and bool(parent.get("author_checkpoint"))
CONFIG=Path(OUTPUT)/"config.json"
study.write(CONFIG,PAPER)
display(pd.DataFrame([snapshot()]))

assert TRAIN_EPOCHS==100, "Paper downstream:100epochs"
assert len(SEEDS)==1, "Dùng một seed cố định; lặp seed trong study riêng, không chọn seed thắng."
assert abs(parent["seg_lr"]-1e-4)<1e-12, "Learning rate khác1e-4 của tài liệu tham chiếu."
study.write(Path(OUTPUT)/"protocol_alignment.json",{
    "exact_reproduction_verified":False,
    "pilot20epoch_used":False,"seed":SEEDS[0],"epochs":TRAIN_EPOCHS,
    "checkpoint_policy":CHECKPOINT_POLICY,
    "unverified":["original900/100split","Polyp-DDPM unpublished segmentation config",
                  "original best-checkpoint criterion","pretraining overlap"],
})


In [ ]:
inventory=[{"input":"Kvasir train", "n":len(parent["train_ids"]), "required":900},
           {"input":"Kvasir test", "n":len(parent["test_ids"]), "required":100},
           {"input":"Polyp-DDPM bank", "n":len(PAPER["banks"]["deployment"]), "required":900}]
for name, rows in PAPER["external_records"].items():
    inventory.append({"input":name,"n":len(rows),"required":1000 if name=="HyperKvasir" else 196})
for name, rows in PAPER["baseline_pairs"].items():
    inventory.append({"input":name,"n":len(rows),"required":900})
display(pd.DataFrame(inventory))
study.write(Path(OUTPUT)/"run_manifest.json",{
    "master":str(master), "resume_from":str(resume_from), "amp":AMP,
    "split":parent["split"], "generator_sha256":parent["author_sha256"],
    "runner_hash":PAPER["runner_hash"], "memory_policy":"cgroup working set; no inherited timer",
})

display(pd.DataFrame([
    {"arm":"real","real_pairs":900,"synthetic_pairs":0,"source":"Real dataset"},
    {"arm":"Polyp-DDPM","real_pairs":0,"synthetic_pairs":900,"source":"Published checkpoint"},
    {"arm":"mixed","real_pairs":900,"synthetic_pairs":900,"source":"Published checkpoint"},
    {"arm":"mixed_repaired","real_pairs":900,"synthetic_pairs":len(PAPER["baseline_pairs"].get("TextureBackground",[])),
     "source":REPAIRED_LABEL,"status":"ENABLED" if REPAIRED_PAIRS_CSV else "NOT_CONFIGURED"},
]))


**Dữ liệu và vai trò đánh giá.** Training dùng đủ900real hoặc900synthetic; mixed1800. Không tách810/90 hoặc chọnseed bằngpilot. Kvasir100 dùng làmvalidation khi policy best; kết quả không độc lập với chọncheckpoint. Kiểm tra trùng nội dung trước khi coi Hyper là đánh giá ngoài độc lập; SHA khác không chứng minh khác cảnh quay/bệnh nhân.


## 3. Bộ chạy có log và cơ chế giải phóng RAM
Mỗi worker kết thúc thì model/optimizer được hệ điều hành giải phóng. Mỗi 25 update và cuối epoch lưu checkpoint; CSV loss lưu từng update. Nếu song song gặp áp lực RAM thật, bộ chạy thử lại một worker bằng cùng checkpoint và recipe. Không tự giảm số epoch hoặc bỏ qua seed.

In [ ]:
def run_phase(phase):
    def launch(world):
        failures=[]
        def worker(rank):
            log=Path(OUTPUT)/f"{phase}_gpu{rank}.log"
            command=[sys.executable,str(RUNTIME/"direct_protocol.py"),"--config",str(CONFIG),
                     "--phase",phase,"--rank",str(rank),"--world",str(world)]
            env=os.environ.copy()
            env.update(PYTHONUNBUFFERED="1",MALLOC_ARENA_MAX="2",OMP_NUM_THREADS="2")
            with log.open("a") as handle:
                process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                         text=True,bufsize=1,env=env)
                for line in process.stdout:
                    print(f"[GPU{rank}] {line}",end="",flush=True)
                    handle.write(line);handle.flush()
                exit_code=process.wait()
            if exit_code:failures.append((rank,exit_code,str(log)))
        threads=[threading.Thread(target=worker,args=(rank,)) for rank in range(world)]
        for thread in threads:thread.start()
        for thread in threads:thread.join()
        if failures:raise RuntimeError(f"Worker lỗi, checkpoint được giữ: {failures}")

    def pending():
        if phase=="quality":return []
        result=[]
        for job in study.phase_jobs(PAPER,phase):
            path=Path(OUTPUT)/"status"/(study.key(job)+".json")
            result.append(json.loads(path.read_text()) if path.exists() else {"key":study.key(job),"status":"NOT_STARTED"})
        return [row for row in result if row["status"]!="COMPLETE"]

    world=1 if phase=="quality" else min(MAX_WORKERS,torch.cuda.device_count())
    launch(world)
    if world>1 and pending():
        print("Tiếp tục các job còn thiếu bằng một worker để giảm RAM.")
        display(pd.DataFrame([snapshot()]))
        launch(1)
    left=pending()
    if left:
        display(pd.DataFrame(left))
        for path in (Path(OUTPUT)/"memory").glob("pressure_*.json"):
            display(pd.DataFrame([json.loads(path.read_text())]))
        raise RuntimeError("RAM vẫn thiếu thật. Checkpoint đã lưu; resume trong phiên mới. Không bỏ qua lượt train.")


## 4. Công bố seed và lượt training trực tiếp
Paper Polyp-DDPM quy định100epoch downstream, không quy định bước chọn seed ở20epoch. Bản này không chạy bước đó. Mọi arm của cả ba kiến trúc dùng cùng seed42 được công bố trước lượt chạy; không chọn lại theo kết quả test.

Các kết quả100epoch cũ tương thích seed/recipe được dùng lại. Checkpoint20epoch pilot không được dùng để warm-start hoặc thay thế checkpoint100epoch. Đây vẫn là study exploratory vì các test đã được xem ở những lượt trước.


In [ ]:
selected=study.select_seeds(PAPER)
display(pd.DataFrame([{"architecture":a,"declared_seed":s,"epochs":TRAIN_EPOCHS,
                      "selection":"fixed; no20epoch pilot"} for a,s in selected.items()]))


In [ ]:
import matplotlib.pyplot as plt
planned=pd.DataFrame(study.phase_jobs(PAPER,"paper"))
display(planned)
planned.to_csv(Path(OUTPUT)/"direct100epoch_jobs.csv",index=False)


## 5. Downstream100epoch theo mã tham chiếu
ResNet34/ImageNet, Adam1e-4, batch8, SMP binaryDiceLoss, FP32, không clipgradient. Augmentation lấy nguyên hàm từ snapshot[9]: flip; shift/scale; pad/resize256; noise; perspective; CLAHE/brightness/gamma; sharpen/blur; contrast/hue. Masks nearest theo Albumentations; binarymask >128; prediction >0.5. Pad trước resize ở validation như nguồn.

Lưu last training state mỗi25update và epoch; lưu bestweights theo valid_dataset_iou mỗi epoch. Dùng đúng một policy đã khai báo cho cả real/synthetic/mixed và cả ba tập đánh giá. Mỗi nhánh cùng seed42; không tìm seed bằngtest.

**Nếu policy best_kvasir_validation:** cột Kvasir_Test100 lịch sử là validation-selected. Không mô tả nó là test độc lập. Training vẫn có900ảnh, không thêm ảnh validation vào train. Hyper cần kiểm tra trùng nội dung; ETIS là đánh giá ngoài theo dữ liệu attach.


In [ ]:
run_phase("paper")
actual,_,coverage=study.tables(PAPER)
display(actual)
if not actual.empty:
    # Bảng đầy đủ: giữ mixed để thấy đúng giá trị augmentation.
    full=actual.pivot(index=["dataset","method"],columns="architecture",
        values=["dataset_iou","dataset_dice","mean_image_iou","mean_image_dice","boundary_f1"])
    display(full);full.to_csv(Path(OUTPUT)/"ALL_ARMS.csv")
    # TableII của paper chỉ gồm real và synthetic-only: giữ riêng cho đối chiếu.
    methods=["real","Polyp-DDPM","LDM","SinGAN-Seg"]
    table=actual[actual.method.isin(methods)].pivot(index=["dataset","method"],columns="architecture",
        values=["dataset_iou","dataset_dice","accuracy","precision"])
    display(table);table.to_csv(Path(OUTPUT)/"TABLE_II_available.csv")


### So sánh real/synthetic/mixed
So sánh cùng seed, preprocessing, augmentation và checkpoint policy. Kvasir có vai trò validation khi chọn best. Tăng mixed ở tập này không phải bằng chứng độc lập về tổng quát hóa. Không thay checkpoint theo từng testset.


In [ ]:
if not actual.empty:
    columns=["architecture","seed","dataset"]
    paired=actual[actual.method.isin(["real","Polyp-DDPM","mixed","TextureBackground","mixed_repaired"])].pivot(
        index=columns,columns="method",values=["dataset_iou","mean_image_iou","dataset_dice","boundary_f1"])
    display(paired);paired.to_csv(Path(OUTPUT)/"mixed_comparison.csv")
    deltas=[]
    for metric in ["dataset_iou","mean_image_iou","dataset_dice","boundary_f1"]:
        for arm,reference in [("mixed","real"),("mixed_repaired","real"),("mixed_repaired","mixed"),
                              ("TextureBackground","Polyp-DDPM")]:
            if arm in paired[metric] and reference in paired[metric]:
                delta=(paired[metric][arm]-paired[metric][reference]).rename("delta").reset_index()
                delta["metric"]=metric;delta["contrast"]=arm+" - "+reference
                deltas.append(delta)
    differences=pd.concat(deltas,ignore_index=True)
    display(differences);differences.to_csv(Path(OUTPUT)/"ARM_DELTAS.csv",index=False)


**Phân tích downstream — cell có thể sửa/xoá.**
[Ghi rõ mixed cải thiện trên tập nào, giảm ở tập nào, và giới hạn của một selected seed/architecture. Tăng ở ETIS có thể là tín hiệu generalization, cần các seed lặp lại trước khi khẳng định bền vững.]


## 6. FID/KID/IS và ablation ảnh
Dùng 900 ảnh bank + 100 ảnh metric bổ sung nếu tìm được checkpoint. 100 ảnh bổ sung không vào downstream. Nếu chỉ có 900 ảnh, giữ n=900 và ghi rõ chưa khớp số lượng paper; không lấy lại ảnh rồi đổi tên để giả đủ 1000.

FID dùng Inception2048; KID 50 subsets × tối đa 100 ảnh; IS 10 splits. Các setting evaluator này được lưu trong cache, chưa xác nhận giống setting paper. Cùng một tập ảnh sinh có cùng IS dù thay reference.

Ảnh màu/nền/boundary là ablation riêng: pilot 16 ảnh không đủ cho metric ổn định sẽ được ghi insufficient_sample_pilot, không điền 0. Baseline chưa attach được ghi MISSING_INPUT; không khiến real/synthetic/mixed training dừng.

In [ ]:
run_phase("quality")
actual,image_metrics,coverage=study.tables(PAPER)
columns=[name for name in ["method","reference","status","n_real","n_fake","FID","KID_mean","KID_std","IS_mean","IS_std"] if name in image_metrics]
display(image_metrics[columns])
if not image_metrics.empty and "FID" in image_metrics:
    table=image_metrics.pivot(index="method",columns="reference",values=["FID","KID_mean","IS_mean"])
    display(table);table.to_csv(Path(OUTPUT)/"TABLE_I_available.csv")
ablation=Path(OUTPUT)/"image_ablation.csv"
if ablation.exists():display(pd.read_csv(ablation))


## 7. Hiển thị dự đoán và loss
Các ảnh dưới là segmentation prediction trên cùng ảnh thật; không dùng ảnh từ model khác làm kết quả của model đang train. Vòng predict lưu probability map cho ba ảnh đầu mỗi test set.

In [ ]:
from PIL import Image
if not actual.empty:
    architecture=ARCHITECTURES[0]
    seed=selected[architecture]
    method="mixed"
    folder=Path(OUTPUT)/"predictions"/f"paper_{architecture}_s{seed}_{method}"/"Kvasir_Test100"
    records={r["id"]:r for r in parent["records"]}
    for path in sorted(folder.glob("*.npy"))[:3]:
        row=records[path.stem];prob=np.load(path)
        fig,axes=plt.subplots(1,4,figsize=(12,3))
        with Image.open(row["image"]) as im:axes[0].imshow(im)
        with Image.open(row["mask"]) as im:axes[1].imshow(im,cmap="gray")
        axes[2].imshow(prob,vmin=0,vmax=1,cmap="viridis")
        axes[3].imshow(prob>=0.5,cmap="gray")
        for ax,title in zip(axes,["Real","Ground truth","Probability","Prediction"]):
            ax.set_title(title);ax.axis("off")
        fig.tight_layout();display(fig);plt.close(fig)


In [ ]:
fig,ax=plt.subplots(figsize=(10,3))
for path in sorted((Path(OUTPUT)/"checkpoints").glob("paper_*_loss.csv")):
    frame=pd.read_csv(path)
    epoch_loss=frame.assign(weighted=frame.loss*frame.samples).groupby("epoch").agg(weighted=("weighted","sum"),n=("samples","sum"))
    ax.plot(epoch_loss.index,epoch_loss.weighted/epoch_loss.n,label=path.stem.removesuffix("_loss"))
ax.set(xlabel="Epoch",ylabel="Mean training Dice loss")
if ax.lines:ax.legend(fontsize=7,bbox_to_anchor=(1.02,1),loc="upper left")
fig.tight_layout();display(fig);plt.close(fig)


### Kiểm tra đủ/thiếu
Hai khái niệm tách riêng: **core** (real/Polyp-DDPM/mixed trên các dataset đã attach) và **toàn bộ bảng paper** (thêm đầy đủ LDM, SinGAN và các tập ngoài). Hoàn tất core không có nghĩa đã tái lập đủ paper.

In [ ]:
actual,image_metrics,coverage=study.tables(PAPER)
active_sets=["Kvasir_Test100"]+[name for name,rows in PAPER["external_records"].items() if rows]
core=coverage[(coverage.table.isin(["II","mixed"])) &
    coverage.method.isin(["real","Polyp-DDPM","mixed"]) & coverage.dataset.isin(active_sets)]
seed_complete=len(selected)==len(ARCHITECTURES)
core_complete=seed_complete and len(core)>0 and (core.status=="COMPLETE").all()
paper_complete=(coverage.status=="COMPLETE").all()
display(pd.DataFrame([{"core_downstream_complete":bool(core_complete),
    "all_paper_tables_complete":bool(paper_complete),"external_sets_available":active_sets}]))
display(coverage[coverage.status!="COMPLETE"])
study.write(Path(OUTPUT)/"summary_status.json",{
    "core_downstream_complete":bool(core_complete),"all_paper_tables_complete":bool(paper_complete),
    "missing":coverage[coverage.status!="COMPLETE"].to_dict("records"),
})
if not core_complete:raise RuntimeError("Core downstream chưa đủ. Xem status/log và resume; không báo hoàn tất giả.")

repair_enabled=bool(PAPER["baseline_pairs"].get("TextureBackground"))
if repair_enabled:
    required={(a,d,m) for a in ARCHITECTURES for d in active_sets for m in ["TextureBackground","mixed_repaired"]}
    observed=set(zip(actual.architecture,actual.dataset,actual.method)) if not actual.empty else set()
    missing_repair=sorted(required-observed)
    display(pd.DataFrame(missing_repair,columns=["architecture","dataset","missing_arm"]))
    if missing_repair:raise RuntimeError("Nhánh cải tiến đã cấu hình nhưng chưa chạy đủ. Xem log/resume.")
else:
    print("Nhánh texture/nền chưa cấu hình bank900; chưa có kết luận về checkpoint cải tiến.")


## Tiếp tục và báo cáo
Save Version toàn bộ outputv8 rồi attach ở phiên sau, giữ cùng recipe. RESUME_EVALUATION trỏ root outputv8. Không dùng outputv7 làm resumev8; chỉ attach nếu muốn đối chiếu lịch sử. Notebook không đọc checkpoint cũ để tạo số mới.

Báo rõ: nguồn bank tác giả hoặc texture/nền; split; seed; sốepoch; phiên bảnphầnmềm; preprocessing_mode; checkpoint_policy; selected_epoch. Với best_kvasir_validation, ghi Kvasir là validation-selected. Không báo trùng sốpaper, không giả đủ LDM/SinGAN nếu chưa cóbank.

File snapshot nguyên bản ở runtime/reference_snapshot. Augmentation dùng hàm được trích từ nguồn, không chạy mã Lightning cũ trong Kaggle. Cơ chế best lấy từ YAML[9]; việc tác giả Polyp-DDPM có chỉnh lại preprocessing/checkpoint riêng hay không chưa xác minh được từ repo công khai.


## Giấy phép nguồn tham chiếu

MIT License

Copyright (c) 2022 simulamet-host

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
